In [ ]:

!nvidia-smi



In [ ]:
import ultralytics
import cv2
import matplotlib
import pandas
import skimage
import yaml

print("All libraries imported successfully!")
print("Ultralytics version:", ultralytics.__version__)
print("OpenCV version:", cv2.__version__)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:

from pathlib import Path

# Main project folder in temporary Colab storage
PROJECT_DIR = Path("/content/MultiIndustryCrackDetection")

# Create folders for datasets, results, and models
DATA_DIR = PROJECT_DIR / "datasets"
RUNS_DIR = PROJECT_DIR / "runs"
MODELS_DIR = PROJECT_DIR / "models"

for folder in [DATA_DIR, RUNS_DIR, MODELS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project setup successful!")
print("Project:", PROJECT_DIR)
print("Datasets:", DATA_DIR)
print("Training results:", RUNS_DIR)
print("Models:", MODELS_DIR)


In [ ]:

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Enable a GPU in Runtime settings.")


In [ ]:

for folder in [PROJECT_DIR, DATA_DIR, RUNS_DIR, MODELS_DIR]:
    print(f"{folder}: {'Exists' if folder.exists() else 'Missing'}")


In [ ]:

from pathlib import Path

PROJECT_DIR = Path("/content/MultiIndustryCrackDetection")
DATA_DIR = PROJECT_DIR / "datasets"
RUNS_DIR = PROJECT_DIR / "runs"
MODELS_DIR = PROJECT_DIR / "models"

for folder in [DATA_DIR, RUNS_DIR, MODELS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folders are ready!")


In [ ]:
from google.colab import files

uploaded = files.upload()

print("Uploaded files:", list(uploaded.keys()))


In [ ]:

import zipfile

zip_files = [
    name for name in uploaded
    if name.lower().endswith(".zip")
]

if not zip_files:
    print("No ZIP file uploaded. Upload the dataset ZIP first.")
else:
    for zip_name in zip_files:
        with zipfile.ZipFile(zip_name, "r") as archive:
            archive.extractall(DATA_DIR / "concrete_cracks")

        print("Extracted:", zip_name)

print("Dataset directory:", DATA_DIR / "concrete_cracks")


In [ ]:

from pathlib import Path
from collections import Counter

dataset_path = DATA_DIR / "concrete_cracks"

if not dataset_path.exists():
    print("Dataset folder not found. Complete the extraction step.")
else:
    all_files = [p for p in dataset_path.rglob("*") if p.is_file()]

    print("Total files:", len(all_files))
    print("File types:", Counter(p.suffix.lower() for p in all_files))

    print("\nFirst 40 files:")
    for p in all_files[:40]:
        print(p.relative_to(dataset_path))


In [ ]:

from pathlib import Path

DATASET_ROOT = (
    Path("/content/MultiIndustryCrackDetection/datasets")
    / "concrete_cracks"
    / "Fresh_Concrete_Crack_Dataset"
)

print("Dataset root exists:", DATASET_ROOT.exists())

for folder in ["images/train", "images/valid", "images/test", "labels"]:
    path = DATASET_ROOT / folder
    print(f"{folder}: {'FOUND' if path.exists() else 'NOT FOUND'}")


In [ ]:

import yaml

yaml_path = DATASET_ROOT / "labels" / "data.yaml"

if yaml_path.exists():
    print(yaml_path.read_text())
    with open(yaml_path, "r") as f:
        dataset_config = yaml.safe_load(f)
    print("\nParsed configuration:")
    print(dataset_config)
else:
    print("data.yaml not found at the expected path.")


In [ ]:

for path in sorted(DATASET_ROOT.rglob("*")):
    if path.is_dir() and (
        path.name in ["train", "valid", "val", "test", "labels", "images"]
    ):
        print("Folder:", path.relative_to(DATASET_ROOT))

print("\nSample text label files:")
label_files = list(DATASET_ROOT.rglob("*.txt"))

for path in label_files[:5]:
    print(f"\n--- {path.relative_to(DATASET_ROOT)} ---")
    print(path.read_text()[:700])


In [ ]:

import matplotlib.pyplot as plt
from PIL import Image

image_files = sorted((DATASET_ROOT / "images").rglob("*.jpg"))
png_files = sorted((DATASET_ROOT / "images").rglob("*.png"))

print("JPG images:", len(image_files))
print("PNG images:", len(png_files))

if image_files:
    img_path = image_files[0]
    img = Image.open(img_path).convert("RGB")

    plt.figure(figsize=(7, 5))
    plt.imshow(img)
    plt.title(img_path.name)
    plt.axis("off")
    plt.show()

    print("Image dimensions:", img.size)


In [ ]:

from pathlib import Path
import shutil

SOURCE = DATASET_ROOT
YOLO_DATA = PROJECT_DIR / "datasets" / "concrete_yolo"

# Map the source dataset's split names to standard names.
split_map = {
    "train": "train",
    "val": "valid",
    "test": "test",
}

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

for target_split, source_label_split in split_map.items():
    image_source = SOURCE / "images" / target_split
    label_source = SOURCE / "labels" / source_label_split / "labels"

    image_target = YOLO_DATA / "images" / target_split
    label_target = YOLO_DATA / "labels" / target_split

    image_target.mkdir(parents=True, exist_ok=True)
    label_target.mkdir(parents=True, exist_ok=True)

    if not image_source.exists():
        print("Missing image folder:", image_source)
        continue

    if not label_source.exists():
        print("Missing label folder:", label_source)
        continue

    for img in image_source.rglob("*"):
        if img.is_file() and img.suffix.lower() in image_extensions:
            shutil.copy2(img, image_target / img.name)

            label_file = label_source / f"{img.stem}.txt"
            if label_file.exists():
                shutil.copy2(label_file, label_target / label_file.name)

    print(f"{target_split}: prepared")

print("\nPrepared dataset:", YOLO_DATA)


In [ ]:

for split in ["train", "val", "test"]:
    img_dir = YOLO_DATA / "images" / split
    lbl_dir = YOLO_DATA / "labels" / split

    images = [
        p for p in img_dir.iterdir()
        if p.is_file() and p.suffix.lower() in image_extensions
    ]

    labels = list(lbl_dir.glob("*.txt"))
    label_stems = {p.stem for p in labels}

    missing = [p.name for p in images if p.stem not in label_stems]

    print(f"\n{split.upper()}")
    print("Images:", len(images))
    print("Label files:", len(labels))
    print("Images without label files:", len(missing))
    if missing:
        print("Examples:", missing[:5])


In [ ]:

import yaml

config = {
    "path": str(YOLO_DATA),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {
        0: "crack"
    }
}

yaml_path = YOLO_DATA / "data.yaml"

with open(yaml_path, "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print(yaml_path.read_text())


In [ ]:

label_files = list((YOLO_DATA / "labels" / "train").glob("*.txt"))

for label_file in label_files:
    lines = [
        line.strip()
        for line in label_file.read_text().splitlines()
        if line.strip()
    ]

    if lines:
        print("Example:", label_file.name)
        print(lines[0][:300])
        values = lines[0].split()
        print("Number of values:", len(values))
        print("Class ID:", values[0])
        break
else:
    print("No non-empty training labels found.")


In [ ]:

from ultralytics import YOLO
import torch

assert torch.cuda.is_available(), "Enable a GPU runtime first."

model = YOLO("yolo26n-seg.pt")

results = model.train(
    data=str(yaml_path),
    epochs=5,
    imgsz=640,
    batch=8,
    device=0,
    workers=2,
    project=str(RUNS_DIR),
    name="concrete_crack_seg_v1",
    plots=True
)

print("Initial training run finished.")


In [ ]:

from pathlib import Path

label_root = YOLO_DATA / "labels"
errors = []
nonempty_labels = 0
empty_labels = 0
polygon_count = 0

for split in ["train", "val", "test"]:
    for label_file in (label_root / split).glob("*.txt"):
        lines = [
            line.strip()
            for line in label_file.read_text().splitlines()
            if line.strip()
        ]

        if not lines:
            empty_labels += 1
            continue

        for line in lines:
            values = line.split()

            try:
                class_id = int(values[0])
                coords = [float(x) for x in values[1:]]
            except ValueError:
                errors.append(f"Invalid values: {label_file.name}")
                continue

            if class_id != 0:
                errors.append(f"Unexpected class ID: {label_file.name}")

            if len(coords) < 6 or len(coords) % 2 != 0:
                errors.append(f"Invalid polygon coordinates: {label_file.name}")
                continue

            if not all(0.0 <= x <= 1.0 for x in coords):
                errors.append(f"Coordinates outside 0-1: {label_file.name}")
                continue

            nonempty_labels += 1
            polygon_count += 1

print("Valid polygons:", polygon_count)
print("Empty label files:", empty_labels)
print("Errors:", len(errors))
print("\nExamples of errors:", errors[:10])

if errors:
    print("Fix the annotation issues before training.")
else:
    print("Annotation checks passed!")


In [ ]:

from ultralytics import YOLO
import torch

assert torch.cuda.is_available(), "Please enable a GPU runtime."

model = YOLO("yolo11n-seg.pt")

results = model.train(
    data=str(yaml_path),
    epochs=5,
    imgsz=640,
    batch=8,
    device=0,
    workers=2,
    project=str(RUNS_DIR),
    name="concrete_crack_seg_v1",
    plots=True,
    seed=42
)

print("Training completed!")


In [ ]:

from pathlib import Path
from ultralytics import YOLO

best_weights = (
    RUNS_DIR / "concrete_crack_seg_v1" / "weights" / "best.pt"
)

assert best_weights.exists(), f"Model weights not found: {best_weights}"

trained_model = YOLO(str(best_weights))

metrics = trained_model.val(
    data=str(yaml_path),
    split="val",
    imgsz=640,
    device=0
)

print("Validation completed.")
print("Best model:", best_weights)


In [ ]:

from google.colab import files

files.download(str(best_weights))


In [ ]:

problem_names = [
    "crack_ (48)_JPG.rf.LLrGZea1CCK9LwxYOugL.txt",
    "crack_ (127)_JPG.rf.C7zR4osHlzuFzQwWSePt.txt",
    "crack_ (197)_JPG.rf.1RTagS9vouVo7Un4RPD0.txt",
    "crack_ (9)_JPG.rf.zYIarHUQIRzxA8aRmK1.txt",
    "crack_ (132)_JPG.rf.Tlc2x1dDe07lhm7nSwmj.txt",
]

for split in ["train", "val", "test"]:
    folder = YOLO_DATA / "labels" / split

    for name in problem_names:
        path = folder / name

        if path.exists():
            print(f"\n--- {split}/{name} ---")

            for line in path.read_text().splitlines():
                if line.strip():
                    values = line.split()
                    print(
                        "Total values:", len(values),
                        "| Class:", values[0],
                        "| First values:", values[:10]
                    )


In [ ]:

import json

original_labels = SOURCE / "labels"

for split in ["train", "valid", "test"]:
    json_path = original_labels / f"_annotations.coco.{split}.json"

    if json_path.exists():
        with open(json_path, "r") as f:
            coco = json.load(f)

        print(f"\n{split.upper()}")
        print("Images:", len(coco.get("images", [])))
        print("Annotations:", len(coco.get("annotations", [])))
        print("Categories:", coco.get("categories", []))

        for ann in coco.get("annotations", [])[:2]:
            print({
                "bbox": ann.get("bbox"),
                "segmentation_type": type(ann.get("segmentation")).__name__,
                "iscrowd": ann.get("iscrowd")
            })


In [ ]:

import json
from pathlib import Path

for split in ["train", "valid", "test"]:
    json_path = SOURCE / "labels" / f"_annotations.coco.{split}.json"

    with open(json_path, "r") as f:
        coco = json.load(f)

    print(f"\n{'=' * 20} {split.upper()} {'=' * 20}")
    print("Categories:", coco["categories"])

    # Inspect the first annotation for each category
    shown_categories = set()

    for ann in coco["annotations"]:
        cat_id = ann["category_id"]

        if cat_id not in shown_categories:
            seg = ann.get("segmentation")
            print("\nCategory ID:", cat_id)
            print("Bounding box:", ann.get("bbox"))
            print("Segmentation type:", type(seg).__name__)

            if isinstance(seg, dict):
                print("Segmentation keys:", list(seg.keys()))
                print("Mask size:", seg.get("size"))
                counts = seg.get("counts")
                print("Counts type:", type(counts).__name__)
                print("Counts preview:", str(counts)[:150])
            elif isinstance(seg, list):
                print("Polygon count:", len(seg))
                print("First polygon preview:", str(seg[:1])[:200])

            shown_categories.add(cat_id)

        if len(shown_categories) == len(coco["categories"]):
            break


In [ ]:

!pip -q install pycocotools

import json
import shutil
import cv2
import numpy as np
from pathlib import Path
from pycocotools import mask as mask_utils

# Original dataset and new, separate clean dataset
CLEAN_DATA = PROJECT_DIR / "datasets" / "concrete_yolo_clean"

split_map = {
    "train": "train",
    "val": "valid",
    "test": "test",
}

# Start fresh only in the NEW output folder
if CLEAN_DATA.exists():
    shutil.rmtree(CLEAN_DATA)

total_images = 0
total_polygons = 0
images_without_polygons = 0
skipped_contours = 0
errors = []

for yolo_split, coco_split in split_map.items():
    image_src_dir = SOURCE / "images" / yolo_split
    json_path = SOURCE / "labels" / f"_annotations.coco.{coco_split}.json"

    image_out_dir = CLEAN_DATA / "images" / yolo_split
    label_out_dir = CLEAN_DATA / "labels" / yolo_split
    image_out_dir.mkdir(parents=True, exist_ok=True)
    label_out_dir.mkdir(parents=True, exist_ok=True)

    with open(json_path, "r") as f:
        coco = json.load(f)

    # Match images by filename
    source_images = {
        p.name: p for p in image_src_dir.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    }

    annotations_by_image = {}
    for ann in coco["annotations"]:
        # COCO category 1 is "crack"; YOLO's crack class will be 0
        if ann["category_id"] == 1:
            annotations_by_image.setdefault(ann["image_id"], []).append(ann)

    split_polygons = 0
    split_empty = 0

    for image_info in coco["images"]:
        filename = Path(image_info["file_name"]).name

        if filename not in source_images:
            errors.append(f"{yolo_split}: image not found: {filename}")
            continue

        src_path = source_images[filename]
        shutil.copy2(src_path, image_out_dir / filename)

        label_path = label_out_dir / f"{src_path.stem}.txt"
        image_polygons = []

        for ann in annotations_by_image.get(image_info["id"], []):
            segmentation = ann.get("segmentation")

            if not isinstance(segmentation, dict):
                errors.append(
                    f"{yolo_split}: unexpected segmentation format: {filename}"
                )
                continue

            try:
                decoded = mask_utils.decode(segmentation)
                if decoded.ndim == 3:
                    decoded = np.any(decoded, axis=2).astype(np.uint8)
                else:
                    decoded = (decoded > 0).astype(np.uint8)

                contours, _ = cv2.findContours(
                    decoded,
                    cv2.RETR_EXTERNAL,
                    cv2.CHAIN_APPROX_SIMPLE
                )

                height, width = decoded.shape

                for contour in contours:
                    # Simplify contours so labels stay manageable
                    approx = cv2.approxPolyDP(contour, 1.0, True)
                    points = approx.reshape(-1, 2)

                    if len(points) < 3:
                        skipped_contours += 1
                        continue

                    # YOLO segmentation format: class x1 y1 x2 y2 ...
                    coords = []
                    for x, y in points:
                        coords.extend([
                            min(max(float(x) / width, 0.0), 1.0),
                            min(max(float(y) / height, 0.0), 1.0)
                        ])

                    image_polygons.append(
                        "0 " + " ".join(f"{v:.6f}" for v in coords)
                    )

            except Exception as e:
                errors.append(f"{yolo_split}: {filename}: {e}")

        label_path.write_text(
            "\n".join(image_polygons) +
            ("\n" if image_polygons else "")
        )

        total_images += 1
        total_polygons += len(image_polygons)
        split_polygons += len(image_polygons)

        if not image_polygons:
            images_without_polygons += 1
            split_empty += 1

    print(
        f"{yolo_split}: "
        f"{len(list(image_out_dir.iterdir()))} images | "
        f"{split_polygons} polygons | "
        f"{split_empty} images without polygons"
    )

# Write a fresh dataset configuration
yaml_text = f"""path: {CLEAN_DATA}
train: images/train
val: images/val
test: images/test
names:
  0: crack
"""
(CLEAN_DATA / "data.yaml").write_text(yaml_text)

print("\n--- CLEAN DATASET SUMMARY ---")
print("Dataset:", CLEAN_DATA)
print("Total images:", total_images)
print("Total polygons:", total_polygons)
print("Images without polygons:", images_without_polygons)
print("Contours skipped (fewer than 3 points):", skipped_contours)
print("Conversion errors:", len(errors))
for err in errors[:20]:
    print("ERROR:", err)

print("\nYAML:")
print((CLEAN_DATA / "data.yaml").read_text())


In [ ]:

from pathlib import Path

print("SOURCE:", SOURCE)
print("\nImage folders:")

for folder in SOURCE.rglob("images"):
    if folder.is_dir():
        files = [
            p for p in folder.iterdir()
            if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png"]
        ]
        if files:
            print(f"\n{folder}")
            print("Image count:", len(files))
            print("Example:", files[0].name)

print("\nCOCO filename example:")
import json

coco_path = SOURCE / "labels" / "_annotations.coco.train.json"
with open(coco_path, "r") as f:
    coco = json.load(f)

print("JSON filename:", coco["images"][0]["file_name"])


In [ ]:

from pathlib import Path

root = Path("/content/MultiIndustryCrackDetection/datasets")

print("Searching for image files...\n")

found = 0

for p in root.rglob("*"):
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        print(p)
        found += 1
        if found >= 15:
            break

print(f"\nDisplayed {found} image paths (maximum 15).")

print("\nTop-level dataset contents:")
for p in SOURCE.iterdir():
    print(("DIR  " if p.is_dir() else "FILE ") + p.name)


In [ ]:

import json
import shutil
import cv2
import numpy as np
from pathlib import Path
from pycocotools import mask as mask_utils

CLEAN_DATA = PROJECT_DIR / "datasets" / "concrete_yolo_clean"

split_map = {"train": "train", "val": "valid", "test": "test"}

if CLEAN_DATA.exists():
    shutil.rmtree(CLEAN_DATA)

total_images = 0
total_polygons = 0
errors = []

for yolo_split, coco_split in split_map.items():
    image_src_dir = SOURCE / "images" / yolo_split
    json_path = SOURCE / "labels" / f"_annotations.coco.{coco_split}.json"

    image_out_dir = CLEAN_DATA / "images" / yolo_split
    label_out_dir = CLEAN_DATA / "labels" / yolo_split
    image_out_dir.mkdir(parents=True, exist_ok=True)
    label_out_dir.mkdir(parents=True, exist_ok=True)

    with open(json_path, "r") as f:
        coco = json.load(f)

    # Case-insensitive filename matching: .JPG and .jpg match
    source_images = {
        p.name.lower(): p
        for p in image_src_dir.iterdir()
        if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    }

    annotations_by_image = {}
    for ann in coco["annotations"]:
        if ann["category_id"] == 1:  # crack
            annotations_by_image.setdefault(ann["image_id"], []).append(ann)

    split_images = 0
    split_polygons = 0
    split_empty = 0

    for image_info in coco["images"]:
        filename = Path(image_info["file_name"]).name
        src_path = source_images.get(filename.lower())

        if src_path is None:
            errors.append(f"{yolo_split}: image not found: {filename}")
            continue

        shutil.copy2(src_path, image_out_dir / src_path.name)
        label_path = label_out_dir / f"{src_path.stem}.txt"
        polygons = []

        for ann in annotations_by_image.get(image_info["id"], []):
            seg = ann.get("segmentation")
            if not isinstance(seg, dict):
                errors.append(f"{yolo_split}: unsupported segmentation: {filename}")
                continue

            try:
                mask = mask_utils.decode(seg)
                if mask.ndim == 3:
                    mask = np.any(mask, axis=2).astype(np.uint8)
                else:
                    mask = (mask > 0).astype(np.uint8)

                h, w = mask.shape
                contours, _ = cv2.findContours(
                    mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
                )

                for contour in contours:
                    approx = cv2.approxPolyDP(contour, 1.0, True)
                    points = approx.reshape(-1, 2)

                    if len(points) < 3:
                        continue

                    coords = []
                    for x, y in points:
                        coords.extend([
                            float(np.clip(x / w, 0, 1)),
                            float(np.clip(y / h, 0, 1))
                        ])

                    polygons.append(
                        "0 " + " ".join(f"{v:.6f}" for v in coords)
                    )

            except Exception as e:
                errors.append(f"{yolo_split}: {filename}: {e}")

        label_path.write_text(
            "\n".join(polygons) + ("\n" if polygons else "")
        )

        split_images += 1
        split_polygons += len(polygons)
        if not polygons:
            split_empty += 1

    total_images += split_images
    total_polygons += split_polygons

    print(
        f"{yolo_split}: {split_images} images | "
        f"{split_polygons} polygons | "
        f"{split_empty} images without polygons"
    )

yaml_text = f"""path: {CLEAN_DATA}
train: images/train
val: images/val
test: images/test
names:
  0: crack
"""
(CLEAN_DATA / "data.yaml").write_text(yaml_text)

print("\nTOTAL IMAGES:", total_images)
print("TOTAL POLYGONS:", total_polygons)
print("ERRORS:", len(errors))
for error in errors[:10]:
    print("ERROR:", error)

print("\nDataset YAML:\n", (CLEAN_DATA / "data.yaml").read_text())


In [ ]:

from pathlib import Path

import math

label_root = CLEAN_DATA / "labels"
image_root = CLEAN_DATA / "images"

bad_labels = []
empty_labels = 0
valid_polygons = 0
image_counts = {}

for split in ["train", "val", "test"]:
    labels_dir = label_root / split
    images_dir = image_root / split

    label_files = list(labels_dir.glob("*.txt"))
    image_files = [
        p for p in images_dir.iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ]

    image_counts[split] = len(image_files)

    for label_file in label_files:
        lines = label_file.read_text().strip().splitlines()

        if not lines:
            empty_labels += 1
            continue

        for line_num, line in enumerate(lines, start=1):
            try:
                values = [float(x) for x in line.split()]

                # class + at least 3 (x,y) coordinate pairs
                if len(values) < 7 or (len(values) - 1) % 2 != 0:
                    raise ValueError("Wrong number of polygon values")

                if values[0] != 0:
                    raise ValueError("Unexpected class ID")

                coords = values[1:]
                if not all(math.isfinite(v) and 0 <= v <= 1 for v in coords):
                    raise ValueError("Coordinates outside 0..1 or non-finite")

                valid_polygons += 1

            except Exception as e:
                bad_labels.append(
                    f"{split}/{label_file.name}, line {line_num}: {e}"
                )

    print(
        f"{split}: {len(image_files)} images, "
        f"{len(label_files)} label files"
    )

print("\n--- VALIDATION SUMMARY ---")
print("Valid polygons:", valid_polygons)
print("Empty label files:", empty_labels)
print("Bad label rows:", len(bad_labels))

for item in bad_labels[:10]:
    print("PROBLEM:", item)


In [ ]:

from ultralytics import YOLO
from pathlib import Path

yaml_path = CLEAN_DATA / "data.yaml"

model = YOLO("yolo11n-seg.pt")

results = model.train(
    data=str(yaml_path),
    epochs=30,
    imgsz=640,
    batch=8,
    device=0,
    workers=2,
    project=str(RUNS_DIR),
    name="concrete_crack_seg_clean_v1",
    plots=True,
    seed=42,
    patience=8
)

print("\nTraining completed.")
print("Run directory:", results.save_dir)
print("Best model:", Path(results.save_dir) / "weights" / "best.pt")


In [ ]:

from ultralytics import YOLO
from pathlib import Path

best_model_path = (
    RUNS_DIR
    / "concrete_crack_seg_clean_v1"
    / "weights"
    / "best.pt"
)

model = YOLO(str(best_model_path))

test_results = model.val(
    data=str(CLEAN_DATA / "data.yaml"),
    split="test",
    imgsz=640,
    batch=8,
    device=0,
    plots=True,
    project=str(RUNS_DIR),
    name="concrete_crack_test_v1"
)

print("\nTest evaluation completed.")
print("Box mAP50:", test_results.box.map50)
print("Box mAP50-95:", test_results.box.map)
print("Mask mAP50:", test_results.seg.map50)
print("Mask mAP50-95:", test_results.seg.map)


In [ ]:

from ultralytics import YOLO
from pathlib import Path
from IPython.display import display, Image

best_model_path = (
    RUNS_DIR / "concrete_crack_seg_clean_v1" / "weights" / "best.pt"
)

model = YOLO(str(best_model_path))
test_images = list((CLEAN_DATA / "images" / "test").glob("*"))

# Predict on the first 5 test images
preview_dir = RUNS_DIR / "crack_prediction_preview"

model.predict(
    source=[str(p) for p in test_images[:5]],
    imgsz=640,
    conf=0.25,
    save=True,
    project=str(RUNS_DIR),
    name="crack_prediction_preview",
    exist_ok=True
)

print("Prediction images saved to:", preview_dir)

for image_path in sorted(preview_dir.glob("*")):
    if image_path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        print(image_path.name)
        display(Image(filename=str(image_path), width=600))


In [ ]:

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

preview_names = [
    "crack_ (177)_JPG.rf.iDKf9cjWxf6SeyEEATzt.jpg",
    "crack_ (254)_JPG.rf.jHltzruHphurewKLpG7f.jpg",
    "crack_ (424)_JPG.rf.PwYSAqnx8FWFtIiudMSo.jpg",
    "crack_ (9)_JPG.rf.zYIarHUQIRzxY8aRmK1.jpg",
    "non_crack_ (46)_JPG.rf.a3fFj6MZUacfs9sKUj1z.jpg",
]

for name in preview_names:
    image_path = CLEAN_DATA / "images" / "test" / name
    label_path = CLEAN_DATA / "labels" / "test" / (Path(name).stem + ".txt")

    image = cv2.imread(str(image_path))
    if image is None:
        print("Image not found:", image_path)
        continue

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)

    if label_path.exists():
        for line in label_path.read_text().splitlines():
            vals = [float(v) for v in line.split()]
            coords = np.array(vals[1:]).reshape(-1, 2)
            coords[:, 0] *= w
            coords[:, 1] *= h
            cv2.fillPoly(mask, [coords.astype(np.int32)], 1)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(image)
    axes[0].set_title("Original image")
    axes[1].imshow(mask, cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("Ground-truth crack mask")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(name, fontsize=9)
    plt.tight_layout()
    plt.show()


In [ ]:

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

model_path = RUNS_DIR / "concrete_crack_seg_clean_v1" / "weights" / "best.pt"
model = YOLO(str(model_path))

test_dir = CLEAN_DATA / "images" / "test"
label_dir = CLEAN_DATA / "labels" / "test"

names = [
    "crack_ (177)_JPG.rf.iDKf9cjWxf6SeyEEATzt.jpg",
    "crack_ (254)_JPG.rf.jHltzruHphurewKLpG7f.jpg",
    "crack_ (424)_JPG.rf.PwYSAqnx8FWFtIiudMSo.jpg",
    "non_crack_ (46)_JPG.rf.a3fFj6MZUacfs9sKUj1z.jpg",
]

for name in names:
    image_path = test_dir / name

    if not image_path.exists():
        print("Missing:", name)
        continue

    image = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]

    gt = np.zeros((h, w), dtype=np.uint8)
    label_path = label_dir / (Path(name).stem + ".txt")

    if label_path.exists():
        for line in label_path.read_text().splitlines():
            if not line.strip():
                continue
            values = [float(v) for v in line.split()]
            points = np.array(values[1:]).reshape(-1, 2)
            points[:, 0] *= w
            points[:, 1] *= h
            cv2.fillPoly(gt, [points.astype(np.int32)], 1)

    result = model.predict(
        source=str(image_path), imgsz=640, conf=0.25, verbose=False
    )[0]

    pred = np.zeros((h, w), dtype=np.uint8)
    if result.masks is not None:
        for mask in result.masks.data.cpu().numpy():
            resized = cv2.resize(
                mask.astype(np.uint8), (w, h),
                interpolation=cv2.INTER_NEAREST
            )
            pred |= resized

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(image)
    axes[0].set_title("Original")
    axes[1].imshow(gt, cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("Ground truth")
    axes[2].imshow(pred, cmap="gray", vmin=0, vmax=1)
    axes[2].set_title("Model prediction")

    for ax in axes:
        ax.axis("off")

    fig.suptitle(name, fontsize=9)
    plt.tight_layout()
    plt.show()

    print(
        f"{name}: predicted instances = "
        f"{0 if result.masks is None else len(result.masks.data)}"
    )


In [ ]:
import cv2
import skimage

print("OpenCV installed:", cv2.__version__)
print("Scikit-image installed:", skimage.__version__)
print("Installation successful!")

In [ ]:

import cv2
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from ultralytics import YOLO
from skimage.morphology import skeletonize, remove_small_objects
from scipy.ndimage import distance_transform_edt

# Load your trained model
model_path = (
    RUNS_DIR
    / "concrete_crack_seg_clean_v1"
    / "weights"
    / "best.pt"
)
model = YOLO(str(model_path))

# Choose a test image that contains a crack
image_path = (
    CLEAN_DATA / "images" / "test"
    / "crack_ (177)_JPG.rf.iDKf9cjWxf6SeyEEATzt.jpg"
)

if not image_path.exists():
    raise FileNotFoundError(f"Image not found: {image_path}")

image = cv2.imread(str(image_path))
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
height, width = image.shape[:2]

# Run segmentation
result = model.predict(
    source=str(image_path),
    imgsz=640,
    conf=0.25,
    verbose=False
)[0]

if result.masks is None:
    raise ValueError(
        "No crack mask detected. Try another test image or inspect "
        "the prediction confidence before measuring."
    )

# Combine detected masks into one binary mask
combined_mask = np.zeros((height, width), dtype=np.uint8)

for predicted_mask in result.masks.data.cpu().numpy():
    resized_mask = cv2.resize(
        predicted_mask.astype(np.float32),
        (width, height),
        interpolation=cv2.INTER_LINEAR
    )
    combined_mask |= (resized_mask >= 0.5).astype(np.uint8)

# Remove tiny isolated components
clean_mask = remove_small_objects(
    combined_mask.astype(bool),
    min_size=5
)

# Skeleton: one-pixel-wide representation of the crack path
skeleton = skeletonize(clean_mask)

# Approximate path length using 8-connected skeleton pixels
horizontal_vertical = np.array([
    [0, 1, 0],
    [1, 0, 1],
    [0, 1, 0]
])

diagonal = np.array([
    [1, 0, 1],
    [0, 0, 0],
    [1, 0, 1]
])

skel_uint8 = skeleton.astype(np.uint8)
vertical_horizontal_count = cv2.filter2D(
    skel_uint8, -1, horizontal_vertical
)
diagonal_count = cv2.filter2D(skel_uint8, -1, diagonal)

# Count each neighbouring pixel pair once
length_pixels = (
    np.sum(vertical_horizontal_count[skeleton]) / 2
    + np.sum(diagonal_count[skeleton]) / 2 * np.sqrt(2)
)

# Distance transform: distance from each crack pixel to background
distance_map = distance_transform_edt(clean_mask)

# Local width is approximately twice the distance to the boundary
local_widths = 2 * distance_map[skeleton]

if len(local_widths) == 0:
    raise ValueError("The predicted mask is too small to measure.")

average_width_px = float(np.mean(local_widths))
median_width_px = float(np.median(local_widths))
maximum_width_px = float(np.max(local_widths))

print("CRACK MEASUREMENT RESULTS")
print(f"Estimated path length: {length_pixels:.2f} pixels")
print(f"Average width:         {average_width_px:.2f} pixels")
print(f"Median width:          {median_width_px:.2f} pixels")
print(f"Maximum width:         {maximum_width_px:.2f} pixels")
print("\nThese are image-space estimates, not millimetres.")

# Visualize the mask and skeleton
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(image_rgb)
axes[0].set_title("Original image")

axes[1].imshow(clean_mask, cmap="gray")
axes[1].set_title("Predicted crack mask")

axes[2].imshow(image_rgb)
axes[2].imshow(skeleton, cmap="autumn", alpha=0.9)
axes[2].set_title("Crack skeleton")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
!pip -q install transformers


In [ ]:
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image
from ultralytics import YOLO
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
import torch.nn.functional as F

# Select GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Load your trained crack-segmentation model
model_path = (
    Path("/content/MultiIndustryCrackDetection")
    / "runs/concrete_crack_seg_clean_v1/weights/best.pt"
)

assert model_path.exists(), f"YOLO weights not found: {model_path}"
crack_model = YOLO(str(model_path))

# Choose one image from your test dataset
image_path = (
    Path("/content/MultiIndustryCrackDetection")
    / "datasets/concrete_yolo_clean/images/test"
    / "crack_ (177)_JPG.rf.iDKf9cjWxf6SeyEEATzt.jpg"
)

assert image_path.exists(), f"Test image not found: {image_path}"

image = Image.open(image_path).convert("RGB")
image_np = np.array(image)
height, width = image_np.shape[:2]

# Load pretrained monocular scene-depth model
depth_model_id = "depth-anything/Depth-Anything-V2-Small-hf"

processor = AutoImageProcessor.from_pretrained(depth_model_id)
depth_model = AutoModelForDepthEstimation.from_pretrained(
    depth_model_id
).to(device).eval()

# Predict scene depth
inputs = processor(images=image, return_tensors="pt")
inputs = {key: value.to(device) for key, value in inputs.items()}

with torch.no_grad():
    outputs = depth_model(**inputs)

depth = F.interpolate(
    outputs.predicted_depth.unsqueeze(1),
    size=(height, width),
    mode="bicubic",
    align_corners=False
).squeeze().cpu().numpy()

print("Image size:", width, "x", height)
print("Depth-map shape:", depth.shape)
print("Model-output range:", float(depth.min()), "to", float(depth.max()))
print("Depth-map values are not calibrated physical distances.")

In [ ]:
!pip -q install ultralytics transformers


In [ ]:
import ultralytics
import transformers
import torch

print("Ultralytics:", ultralytics.__version__)
print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

In [ ]:
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image
from ultralytics import YOLO
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
import torch.nn.functional as F

# Select GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Load your trained crack-segmentation model
model_path = (
    Path("/content/MultiIndustryCrackDetection")
    / "runs/concrete_crack_seg_clean_v1/weights/best.pt"
)

assert model_path.exists(), f"YOLO weights not found: {model_path}"
crack_model = YOLO(str(model_path))

# Choose one image from your test dataset
image_path = (
    Path("/content/MultiIndustryCrackDetection")
    / "datasets/concrete_yolo_clean/images/test"
    / "crack_ (177)_JPG.rf.iDKf9cjWxf6SeyEEATzt.jpg"
)

assert image_path.exists(), f"Test image not found: {image_path}"

image = Image.open(image_path).convert("RGB")
image_np = np.array(image)
height, width = image_np.shape[:2]

# Load pretrained monocular scene-depth model
depth_model_id = "depth-anything/Depth-Anything-V2-Small-hf"

processor = AutoImageProcessor.from_pretrained(depth_model_id)
depth_model = AutoModelForDepthEstimation.from_pretrained(
    depth_model_id
).to(device).eval()

# Predict scene depth
inputs = processor(images=image, return_tensors="pt")
inputs = {key: value.to(device) for key, value in inputs.items()}

with torch.no_grad():
    outputs = depth_model(**inputs)

depth = F.interpolate(
    outputs.predicted_depth.unsqueeze(1),
    size=(height, width),
    mode="bicubic",
    align_corners=False
).squeeze().cpu().numpy()

print("Image size:", width, "x", height)
print("Depth-map shape:", depth.shape)
print("Model-output range:", float(depth.min()), "to", float(depth.max()))
print("Depth-map values are not calibrated physical distances.")

In [ ]:
from pathlib import Path

search_roots = [
    Path("/content/MultiIndustryCrackDetection"),
    Path("/content")
]

found = set()

for root in search_roots:
    if root.exists():
        for file in root.rglob("best.pt"):
            found.add(str(file))

if found:
    print("Found trained model weights:")
    for file in sorted(found):
        print(file)
else:
    print("No best.pt file found under /content.")

In [ ]:
from pathlib import Path

print("PROJECT FOLDER")
project = Path("/content/MultiIndustryCrackDetection")
print("Exists:", project.exists())

if project.exists():
    for item in project.iterdir():
        print("-", item.name)

print("\nGOOGLE DRIVE")
drive = Path("/content/drive/MyDrive")
print("Drive mounted:", drive.exists())

if drive.exists():
    for item in drive.iterdir():
        print("-", item.name)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from google.colab import drive

drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

In [ ]:
from google.colab import drive

drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")

print("Searching Google Drive for best.pt...")

matches = list(drive_root.rglob("best.pt"))

if matches:
    print("\nFound model weights:")
    for path in matches:
        print(path)
else:
    print("\nNo best.pt found in this Drive account.")

In [ ]:
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")

for name in [
    "MultiIndustryCrackDetection",
    "Fresh_Concrete_Crack_Dataset",
    "concrete_yolo_clean"
]:
    print(f"\n--- {name} ---")
    found = False

    for path in drive_root.rglob(name):
        if path.is_dir():
            print(path)
            found = True

    if not found:
        print("Not found")

In [ ]:
results = model.train(
    ...
    project=str(RUNS_DIR),
    name="concrete_crack_seg_clean_v1",
    ...
)

In [ ]:
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")
extensions = {".pt", ".pth", ".zip"}

print("Searching Google Drive for model weights and ZIP files...")

found = []

for path in drive_root.rglob("*"):
    if path.is_file() and path.suffix.lower() in extensions:
        found.append(path)

if found:
    for path in found:
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"{path} ({size_mb:.1f} MB)")
else:
    print("No .pt, .pth, or .zip files found.")

In [ ]:
import zipfile
from pathlib import Path

zip_path = Path("/content/drive/MyDrive/index.zip")

print("File size:", zip_path.stat().st_size, "bytes")

if zipfile.is_zipfile(zip_path):
    with zipfile.ZipFile(zip_path, "r") as z:
        files = z.namelist()
        print("Number of files inside:", len(files))
        for name in files[:30]:
            print(name)
else:
    print("This file is not a valid ZIP archive.")

In [ ]:
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")
image_extensions = {".jpg", ".jpeg", ".png", ".webp"}

matches = []

for path in drive_root.rglob("*"):
    if path.is_file() and path.suffix.lower() in image_extensions:
        if "crack" in str(path).lower() or "concrete" in str(path).lower():
            matches.append(path)

print("Crack/concrete-related images found:", len(matches))

for path in matches[:20]:
    print(path)

In [ ]:
from pathlib import Path

for folder in [
    Path("/content/drive/MyDrive"),
    Path("/content/sample_data")
]:
    print(f"\n{folder}")
    print("Exists:", folder.exists())

    if folder.exists():
        for item in list(folder.iterdir())[:15]:
            print(" -", item.name)

In [ ]:
from pathlib import Path
from PIL import Image

drive_root = Path("/content/drive/MyDrive")
extensions = {".jpg", ".jpeg", ".png", ".webp"}

for path in drive_root.iterdir():
    if path.is_file() and path.suffix.lower() in extensions:
        try:
            with Image.open(path) as img:
                print(path.name, "| Size:", img.size, "| Format:", img.format)
        except Exception:
            pass

In [ ]:
from pathlib import Path

p = Path(
    "/content/MultiIndustryCrackDetection/runs/"
    "concrete_crack_seg_clean_v1/weights/best.pt"
)

print("Weights available:", p.exists())

In [ ]:
from pathlib import Path

search_locations = [
    Path("/content"),
    Path("/content/drive/MyDrive")
]

extensions = {".pt", ".pth"}

for root in search_locations:
    print(f"\nSearching: {root}")

    if not root.exists():
        print("Folder not available")
        continue

    found = 0
    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in extensions:
            print(path, f"({path.stat().st_size / 1e6:.2f} MB)")
            found += 1

    if found == 0:
        print("No model checkpoint found here.")

In [ ]:
import os

for file in os.listdir("/content"):
    if file.lower().endswith(".zip"):
        print(file)

In [ ]:
import os

print("Files in Colab:")
print(os.listdir("/content"))

In [ ]:
import os

for file in os.listdir("/content"):
    if file.lower().endswith(".zip"):
        print("ZIP found:", file)

In [ ]:
import os

zip_files = [
    f for f in os.listdir("/content")
    if f.lower().endswith(".zip")
]

print("ZIP files found:", zip_files)

In [ ]:
from pathlib import Path

project = Path("/content/MultiIndustryCrackDetection")

checks = {
    "Dataset": project / "datasets/concrete_cracks/Fresh_Concrete_Crack_Dataset",
    "YOLO11 best.pt": project / "runs/concrete_crack_seg_v1-2/weights/best.pt",
    "YOLO26 best.pt": project / "runs/concrete_crack_seg_v1/weights/best.pt",
}

for name, path in checks.items():
    print(f"{name}: {'FOUND' if path.exists() else 'NOT FOUND'}")

In [ ]:
from google.colab import files

uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

In [ ]:
from google.colab import files

uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

In [ ]:
from google.colab import files

uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

backup = Path('/content/drive/MyDrive/CrackDetection_Backup')
backup.mkdir(parents=True, exist_ok=True)

files_to_copy = [
    Path('/content/Fresh_Concrete_Crack_Dataset_v2.zip'),
    Path('/content/best.pt')
]

for file in files_to_copy:
    if file.exists():
        destination = backup / file.name
        shutil.copy2(file, destination)
        print("Backed up:", destination)
    else:
        print("File not found:", file)

print("\nBackup complete.")

In [ ]:
from pathlib import Path
import zipfile

zip_path = Path(
    "/content/drive/MyDrive/CrackDetection_Backup/"
    "Fresh_Concrete_Crack_Dataset_v2.zip"
)

extract_path = Path(
    "/content/MultiIndustryCrackDetection/datasets/concrete_cracks"
)

extract_path.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

print("Dataset extracted:", extract_path.exists())
print("Top-level contents:", [p.name for p in extract_path.iterdir()])

In [ ]:
from ultralytics import YOLO
from pathlib import Path

model_path = Path(
    "/content/drive/MyDrive/CrackDetection_Backup/best.pt"
)

print("Model file exists:", model_path.exists())
print("Model size:", round(model_path.stat().st_size / 1_000_000, 2), "MB")

model = YOLO(str(model_path))
print("Model loaded successfully!")
print("Task:", model.task)
print("Model type:", model)

In [ ]:
from pathlib import Path
from IPython.display import display
from PIL import Image
from ultralytics import YOLO
import matplotlib.pyplot as plt

dataset_root = Path(
    "/content/MultiIndustryCrackDetection/datasets/concrete_cracks/"
    "Fresh_Concrete_Crack_Dataset"
)

# Find a real crack image
image_files = list((dataset_root / "images" / "train").glob("*.jpg"))
crack_images = [
    p for p in image_files
    if p.name.lower().startswith("crack")
]

print("Crack images found:", len(crack_images))

assert crack_images, "No crack images found in the training folder."

image_path = crack_images[0]
model = YOLO(
    "/content/drive/MyDrive/CrackDetection_Backup/best.pt"
)

results = model.predict(
    source=str(image_path),
    conf=0.25,
    save=False,
    verbose=False
)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(Image.open(image_path).convert("RGB"))
plt.title("Original Crack Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(results[0].plot()[..., ::-1])
plt.title("YOLO Crack Segmentation")
plt.axis("off")

plt.tight_layout()
plt.show()

print("Test image:", image_path.name)
print("Detections:", len(results[0].boxes))
print("Segmentation masks:",
      0 if results[0].masks is None else len(results[0].masks.data))

In [ ]:

from pathlib import Path
from PIL import Image
from ultralytics import YOLO
import matplotlib.pyplot as plt

dataset_root = Path(
    "/content/MultiIndustryCrackDetection/datasets/concrete_cracks/"
    "Fresh_Concrete_Crack_Dataset"
)

image_path = next(
    p for p in (dataset_root / "images" / "train").glob("*.jpg")
    if p.name.lower().startswith("crack")
)

model = YOLO("/content/drive/MyDrive/CrackDetection_Backup/best.pt")

results = model.predict(
    source=str(image_path),
    conf=0.25,
    verbose=False
)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(Image.open(image_path).convert("RGB"))
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(results[0].plot()[..., ::-1])
plt.title("YOLO Crack Detection")
plt.axis("off")

plt.show()

print("Image:", image_path.name)
print("Detections:", len(results[0].boxes))
print("Masks:", 0 if results[0].masks is None else len(results[0].masks.data))


In [ ]:

!pip -q install transformers


In [ ]:

import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from PIL import Image
from pathlib import Path
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

# Select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:", device)

# Load pretrained depth estimation model
model_id = "depth-anything/Depth-Anything-V2-Small-hf"
processor = AutoImageProcessor.from_pretrained(model_id)
depth_model = AutoModelForDepthEstimation.from_pretrained(model_id)
depth_model = depth_model.to(device).eval()

# Find a concrete crack image
dataset_root = Path(
    "/content/MultiIndustryCrackDetection/datasets/concrete_cracks/"
    "Fresh_Concrete_Crack_Dataset"
)
image_path = next(
    p for p in (dataset_root / "images" / "train").glob("*.jpg")
    if p.name.lower().startswith("crack")
)

image = Image.open(image_path).convert("RGB")
width, height = image.size

# Predict relative scene depth
inputs = processor(images=image, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = depth_model(**inputs)

depth = F.interpolate(
    outputs.predicted_depth.unsqueeze(1),
    size=(height, width),
    mode="bicubic",
    align_corners=False
).squeeze().cpu().numpy()

# Normalize for visualization
low, high = np.percentile(depth, [2, 98])
depth_visual = np.clip(
    (depth - low) / (high - low + 1e-8), 0, 1
)

# Display results
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(image)
plt.title("Original Concrete Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(depth_visual, cmap="viridis")
plt.title("Relative Scene-Depth Map")
plt.axis("off")

plt.tight_layout()
plt.show()

print("Image:", image_path.name)
print("Depth map shape:", depth.shape)
print("Raw prediction range:", float(depth.min()), float(depth.max()))
print("Note: These are relative depth predictions, NOT crack depth in mm.")


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from ultralytics import YOLO

# Load the saved crack segmentation model
model = YOLO("/content/drive/MyDrive/CrackDetection_Backup/best.pt")

# Use the same image as before
dataset_root = Path(
    "/content/MultiIndustryCrackDetection/datasets/concrete_cracks/"
    "Fresh_Concrete_Crack_Dataset"
)
image_path = next(
    p for p in (dataset_root / "images" / "train").glob("*.jpg")
    if p.name.lower().startswith("crack")
)
image = Image.open(image_path).convert("RGB")

# Run YOLO segmentation
result = model.predict(source=str(image_path), conf=0.20, verbose=False)[0]

if result.masks is None or len(result.masks.data) == 0:
    print("No crack mask detected. Try another image or review the model.")
else:
    # Get the highest-confidence detected crack mask
    best_idx = int(result.boxes.conf.argmax().item())
    mask = result.masks.data[best_idx].cpu().numpy()

    # Resize mask to original image dimensions
    from PIL import Image as PILImage
    mask = np.array(
        PILImage.fromarray((mask * 255).astype(np.uint8)).resize(
            image.size, PILImage.Resampling.NEAREST
        )
    ) > 127

    # Display crack mask on image
    img = np.array(image)

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.imshow(mask, cmap="Reds", alpha=0.55)
    plt.title("YOLO Crack Region")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(mask, cmap="gray")
    plt.title("Extracted Crack Mask")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

    print("Crack-region pixels:", int(mask.sum()))
    print("Detection confidence:", round(float(result.boxes.conf[best_idx]), 3))


In [ ]:

import cv2
import numpy as np
import matplotlib.pyplot as plt

# Use the mask created in your previous cell
mask_uint8 = (mask.astype(np.uint8) * 255)

# Estimate local crack width using a distance transform
distance = cv2.distanceTransform(mask_uint8, cv2.DIST_L2, 5)
skeleton = cv2.ximgproc.thinning(mask_uint8)

widths = 2 * distance[skeleton > 0]

if len(widths) > 0:
    print("Estimated median crack width:", round(float(np.median(widths)), 2), "pixels")
    print("Estimated maximum crack width:", round(float(np.max(widths)), 2), "pixels")

    plt.figure(figsize=(8, 5))
    plt.imshow(distance, cmap="inferno")
    plt.title("Crack Width Map (pixel-based estimate)")
    plt.colorbar(label="Half-width (pixels)")
    plt.axis("off")
    plt.show()
else:
    print("Could not calculate width. The mask may be empty.")


In [ ]:

from pathlib import Path

root = Path(
    "/content/MultiIndustryCrackDetection/datasets/concrete_cracks/"
    "Fresh_Concrete_Crack_Dataset"
)

print("Dataset exists:", root.exists())

for folder in [
    "images/train", "images/val", "images/valid", "images/test",
    "labels/train", "labels/val", "labels/valid", "labels/test"
]:
    p = root / folder
    if p.exists():
        count = sum(1 for f in p.rglob("*") if f.is_file())
        print(f"{folder}: {count} files")

print("\nYAML configuration files:")
for p in root.rglob("*.yaml"):
    print(p)


In [ ]:

from pathlib import Path

yaml_path = Path(
    "/content/MultiIndustryCrackDetection/datasets/concrete_cracks/"
    "Fresh_Concrete_Crack_Dataset/labels/data.yaml"
)

print(yaml_path.read_text())


In [ ]:

from pathlib import Path

root = Path(
    "/content/MultiIndustryCrackDetection/datasets/concrete_cracks/"
    "Fresh_Concrete_Crack_Dataset"
)

for folder in ["images", "labels"]:
    print(f"\n{folder}/")
    base = root / folder
    for p in sorted(base.iterdir()):
        if p.is_dir():
            print(" ", p.name, "->", len(list(p.rglob("*"))), "items")
            for child in list(p.iterdir())[:5]:
                print("    ", child.name)


In [ ]:

from pathlib import Path

root = Path(
    "/content/MultiIndustryCrackDetection/datasets/concrete_cracks/"
    "Fresh_Concrete_Crack_Dataset"
)

for split in ["train", "valid", "test"]:
    folder = root / "labels" / split / "labels"
    print(f"\n{split}: {folder}")
    print("Folder exists:", folder.exists())

    if folder.exists():
        files = list(folder.glob("*.txt"))
        print("TXT annotation files:", len(files))

        if files:
            print("Example file:", files[0].name)
            print("First 3 annotation lines:")
            print("\n".join(files[0].read_text().splitlines()[:3]))


In [ ]:

from pathlib import Path
from collections import Counter

root = Path(
    "/content/MultiIndustryCrackDetection/datasets/concrete_cracks/"
    "Fresh_Concrete_Crack_Dataset"
)

for split in ["train", "valid", "test"]:
    image_dir = root / "images" / ("val" if split == "valid" else split)
    label_dir = root / "labels" / split / "labels"

    empty = 0
    nonempty = 0
    invalid = 0
    missing = 0
    class_counts = Counter()

    for image_path in image_dir.glob("*.jpg"):
        label_path = label_dir / f"{image_path.stem}.txt"

        if not label_path.exists():
            missing += 1
            continue

        lines = [
            line.strip()
            for line in label_path.read_text().splitlines()
            if line.strip()
        ]

        if not lines:
            empty += 1
            continue

        nonempty += 1

        for line in lines:
            parts = line.split()
            try:
                values = [float(x) for x in parts]
                if len(values) < 7 or (len(values) - 1) % 2 != 0:
                    invalid += 1
                elif int(values[0]) != 0 or any(
                    x < 0 or x > 1 for x in values[1:]
                ):
                    invalid += 1
                else:
                    class_counts[int(values[0])] += 1
            except ValueError:
                invalid += 1

    print(f"\n{split.upper()}")
    print("Empty labels:", empty)
    print("Non-empty labels:", nonempty)
    print("Missing label files:", missing)
    print("Invalid annotation lines:", invalid)
    print("Valid polygon lines:", class_counts)


In [ ]:

from pathlib import Path

root = Path(
    "/content/MultiIndustryCrackDetection/datasets/concrete_cracks/"
    "Fresh_Concrete_Crack_Dataset"
)

for split in ["train", "valid", "test"]:
    label_dir = root / "labels" / split / "labels"
    print(f"\n--- {split.upper()} ---")

    for txt in sorted(label_dir.glob("*.txt")):
        for line_num, line in enumerate(
            txt.read_text().splitlines(), start=1
        ):
            if not line.strip():
                continue

            parts = line.split()
            reason = None

            try:
                values = [float(x) for x in parts]

                if len(values) < 7 or (len(values) - 1) % 2 != 0:
                    reason = "Unexpected segmentation polygon format"
                elif values[0] != 0:
                    reason = "Unexpected class ID"
                elif any(x < 0 or x > 1 for x in values[1:]):
                    reason = "Coordinates outside normalized range"

            except ValueError:
                reason = "Non-numeric annotation values"

            if reason:
                print("File:", txt.name)
                print("Line:", line_num)
                print("Problem:", reason)
                print("Content:", line[:300])
                print()


In [ ]:

from pathlib import Path

root = Path(
    "/content/MultiIndustryCrackDetection/datasets/concrete_cracks/"
    "Fresh_Concrete_Crack_Dataset"
)

for p in root.rglob("*"):
    if p.is_file() and p.name.lower() in {
        "readme.md", "readme.txt", "data.yaml"
    }:
        print("\n" + "=" * 60)
        print("FILE:", p)
        try:
            print(p.read_text(errors="replace")[:5000])
        except Exception as e:
            print("Could not read:", e)


In [ ]:

from pathlib import Path

root = Path(
    "/content/MultiIndustryCrackDetection/datasets/concrete_cracks/"
    "Fresh_Concrete_Crack_Dataset"
)

for name in ["labels_yolo", "masks_unet", "labels", "masks"]:
    folder = root / name
    print(f"\n{name}: exists = {folder.exists()}")

    if folder.exists():
        for p in list(folder.iterdir())[:10]:
            print(" ", p.name, "(folder)" if p.is_dir() else "(file)")


In [ ]:

from pathlib import Path

root = Path(
    "/content/MultiIndustryCrackDetection/datasets/concrete_cracks/"
    "Fresh_Concrete_Crack_Dataset"
)

for name in ["labels_yolo", "masks_unet", "labels", "masks"]:
    folder = root / name
    print(f"\n{name}: exists = {folder.exists()}")

    if folder.exists():
        for p in list(folder.iterdir())[:10]:
            print(" ", p.name, "(folder)" if p.is_dir() else "(file)")


In [ ]:

from pathlib import Path
from PIL import Image
import numpy as np

root = Path(
    "/content/MultiIndustryCrackDetection/datasets/concrete_cracks/"
    "Fresh_Concrete_Crack_Dataset"
)

for split in ["train", "val", "test"]:
    folder = root / "masks" / split
    files = [
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in [".png", ".jpg", ".jpeg"]
    ]

    print(f"\n{split.upper()}: {len(files)} mask images")

    if files:
        mask = np.array(Image.open(files[0]).convert("L"))
        print("Example:", files[0].name)
        print("Size:", mask.shape)
        print("Unique pixel values:", np.unique(mask)[:20])


In [ ]:

from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

root = Path(
    "/content/MultiIndustryCrackDetection/datasets/concrete_cracks/"
    "Fresh_Concrete_Crack_Dataset"
)

image_path = next(
    p for p in (root / "images" / "train").glob("*.jpg")
    if p.name.lower().startswith("crack")
)

mask_path = root / "masks" / "train" / f"{image_path.stem}.png"

print("Image:", image_path.name)
print("Matching mask exists:", mask_path.exists())

if mask_path.exists():
    image = Image.open(image_path).convert("RGB")
    mask = np.array(Image.open(mask_path).convert("L"))

    plt.figure(figsize=(15, 5))

    plt.subplot(1, 3, 1)
    plt.imshow(image)
    plt.title("Original Concrete Image")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(mask, cmap="gray")
    plt.title("Ground-Truth Crack Mask")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(image)
    plt.imshow(mask > 0, cmap="Reds", alpha=0.5)
    plt.title("Crack Mask Overlay")
    plt.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("The image and mask filenames do not match exactly.")


In [ ]:

from pathlib import Path
from PIL import Image
import numpy as np

root = Path(
    "/content/MultiIndustryCrackDetection/datasets/concrete_cracks/"
    "Fresh_Concrete_Crack_Dataset"
)

for split in ["train", "val", "test"]:
    image_dir = root / "images" / split
    mask_dir = root / "masks" / split

    total = 0
    empty_masks = 0
    missing_masks = 0
    crack_images_with_masks = 0

    for image_path in image_dir.glob("*.jpg"):
        mask_path = mask_dir / f"{image_path.stem}.png"

        if not mask_path.exists():
            missing_masks += 1
            continue

        total += 1
        mask = np.array(Image.open(mask_path).convert("L"))
        has_crack = np.any(mask > 0)

        if has_crack:
            crack_images_with_masks += 1
        else:
            empty_masks += 1

    print(f"\n{split.upper()}")
    print("Images with matching masks:", total)
    print("Images with non-empty masks:", crack_images_with_masks)
    print("Empty masks:", empty_masks)
    print("Missing masks:", missing_masks)


In [ ]:

from pathlib import Path
import cv2
import numpy as np

root = Path(
    "/content/MultiIndustryCrackDetection/datasets/concrete_cracks/"
    "Fresh_Concrete_Crack_Dataset"
)

for split in ["train", "val", "test"]:
    image_dir = root / "images" / split
    mask_dir = root / "masks" / split

    checked = 0
    polygons = 0
    empty_masks = 0

    for image_path in image_dir.glob("*.jpg"):
        mask_path = mask_dir / f"{image_path.stem}.png"
        if not mask_path.exists():
            continue

        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        binary = (mask > 0).astype(np.uint8)

        contours, _ = cv2.findContours(
            binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        valid_contours = [
            c for c in contours
            if len(c) >= 3 and cv2.contourArea(c) >= 3
        ]

        checked += 1
        if valid_contours:
            polygons += len(valid_contours)
        else:
            empty_masks += 1

    print(f"\n{split.upper()}")
    print("Paired images checked:", checked)
    print("Non-empty mask polygons:", polygons)
    print("Empty masks:", empty_masks)


In [ ]:

from pathlib import Path
import shutil
import cv2
import yaml

source = Path(
    "/content/MultiIndustryCrackDetection/datasets/concrete_cracks/"
    "Fresh_Concrete_Crack_Dataset"
)
output = Path(
    "/content/MultiIndustryCrackDetection/datasets/concrete_seg_clean"
)

for split in ["train", "val", "test"]:
    image_dir = source / "images" / split
    mask_dir = source / "masks" / split
    out_images = output / "images" / split
    out_labels = output / "labels" / split

    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)

    converted = 0
    empty = 0
    missing = 0

    for image_path in image_dir.glob("*.jpg"):
        mask_path = mask_dir / f"{image_path.stem}.png"

        if not mask_path.exists():
            missing += 1
            continue

        shutil.copy2(image_path, out_images / image_path.name)

        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        binary = (mask > 0).astype("uint8")

        contours, _ = cv2.findContours(
            binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        lines = []
        for contour in contours:
            if cv2.contourArea(contour) < 3:
                continue

            epsilon = 0.001 * cv2.arcLength(contour, True)
            polygon = cv2.approxPolyDP(contour, epsilon, True).reshape(-1, 2)

            if len(polygon) < 3:
                continue

            coords = []
            for x, y in polygon:
                coords.extend([
                    min(max(float(x) / mask.shape[1], 0.0), 1.0),
                    min(max(float(y) / mask.shape[0], 0.0), 1.0)
                ])

            lines.append("0 " + " ".join(f"{v:.6f}" for v in coords))

        label_path = out_labels / f"{image_path.stem}.txt"
        label_path.write_text("\n".join(lines))

        if lines:
            converted += len(lines)
        else:
            empty += 1

    print(f"\n{split.upper()}")
    print("Polygons written:", converted)
    print("Empty masks:", empty)
    print("Missing masks:", missing)

yaml_path = output / "data.yaml"
yaml_path.write_text(yaml.safe_dump({
    "path": str(output),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 1,
    "names": ["crack"]
}, sort_keys=False))

print("\nNew dataset configuration:", yaml_path)
print("Original dataset unchanged.")


In [ ]:

from pathlib import Path

root = Path(
    "/content/MultiIndustryCrackDetection/datasets/concrete_seg_clean"
)

for split in ["train", "val", "test"]:
    label_dir = root / "labels" / split
    total = 0
    empty = 0
    invalid = 0

    for p in label_dir.glob("*.txt"):
        total += 1
        lines = [line.strip() for line in p.read_text().splitlines() if line.strip()]

        if not lines:
            empty += 1

        for line in lines:
            parts = line.split()
            try:
                values = [float(x) for x in parts]
                if (
                    len(values) < 7
                    or (len(values) - 1) % 2 != 0
                    or values[0] != 0
                    or any(x < 0 or x > 1 for x in values[1:])
                ):
                    invalid += 1
            except ValueError:
                invalid += 1

    print(f"\n{split.upper()}")
    print("Label files:", total)
    print("Empty labels:", empty)
    print("Invalid polygon lines:", invalid)


In [ ]:

from ultralytics import YOLO

model = YOLO(
    "/content/drive/MyDrive/CrackDetection_Backup/best.pt"
)

metrics = model.val(
    data="/content/MultiIndustryCrackDetection/datasets/concrete_seg_clean/data.yaml",
    split="val",
    imgsz=640,
    batch=8,
    device=0 if __import__("torch").cuda.is_available() else "cpu",
    plots=True
)

print("\n--- BASELINE SEGMENTATION METRICS ---")
print("Box precision:", round(metrics.box.mp, 4))
print("Box recall:", round(metrics.box.mr, 4))
print("Box mAP50:", round(metrics.box.map50, 4))
print("Box mAP50-95:", round(metrics.box.map, 4))
print("Mask mAP50:", round(metrics.seg.map50, 4))
print("Mask mAP50-95:", round(metrics.seg.map, 4))


In [ ]:

from ultralytics import YOLO
from pathlib import Path
import torch
import shutil

# Load your existing trained model
model = YOLO(
    "/content/drive/MyDrive/CrackDetection_Backup/best.pt"
)

# Fine-tune on the clean segmentation dataset
results = model.train(
    data="/content/MultiIndustryCrackDetection/datasets/concrete_seg_clean/data.yaml",
    epochs=50,
    imgsz=640,
    batch=8,
    device=0 if torch.cuda.is_available() else "cpu",
    workers=2,
    patience=10,
    project="/content/MultiIndustryCrackDetection/runs",
    name="concrete_crack_improved",
    exist_ok=True,
    plots=True,
    seed=42
)

# Locate the best checkpoint
best_path = Path(
    "/content/MultiIndustryCrackDetection/runs/"
    "concrete_crack_improved/weights/best.pt"
)

print("\nTraining finished.")
print("Best checkpoint exists:", best_path.exists())

if best_path.exists():
    backup_path = Path(
        "/content/drive/MyDrive/CrackDetection_Backup/"
        "best_improved.pt"
    )
    shutil.copy2(best_path, backup_path)
    print("Improved checkpoint backed up to:", backup_path)


In [ ]:

from ultralytics import YOLO
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

root = Path(
    "/content/MultiIndustryCrackDetection/datasets/concrete_seg_clean"
)

image_path = next(
    p for p in (root / "images" / "test").glob("*.jpg")
    if p.name.lower().startswith("crack")
)

model = YOLO(
    "/content/drive/MyDrive/CrackDetection_Backup/best_improved.pt"
)

result = model.predict(
    source=str(image_path),
    conf=0.25,
    verbose=False
)[0]

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(Image.open(image_path).convert("RGB"))
plt.title("Original Test Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(result.plot()[..., ::-1])
plt.title("Improved YOLO26 Prediction")
plt.axis("off")

plt.tight_layout()
plt.show()

print("Image:", image_path.name)
print("Detections:", len(result.boxes))
print(
    "Masks:",
    0 if result.masks is None else len(result.masks.data)
)


In [ ]:

from ultralytics import YOLO
import torch

model = YOLO(
    "/content/drive/MyDrive/CrackDetection_Backup/best_improved.pt"
)

metrics = model.val(
    data="/content/MultiIndustryCrackDetection/datasets/concrete_seg_clean/data.yaml",
    split="test",
    imgsz=640,
    batch=8,
    device=0 if torch.cuda.is_available() else "cpu",
    plots=True
)

print("\nTEST SET RESULTS")
print("Box mAP50:", round(metrics.box.map50, 4))
print("Box mAP50-95:", round(metrics.box.map, 4))
print("Mask mAP50:", round(metrics.seg.map50, 4))
print("Mask mAP50-95:", round(metrics.seg.map, 4))


In [ ]:

from google.colab import files
from ultralytics import YOLO
from IPython.display import display
from PIL import Image
import matplotlib.pyplot as plt
import torch

# Upload a crack image from your computer
uploaded = files.upload()
image_path = next(iter(uploaded))

# Load your trained model
model = YOLO(
    "/content/drive/MyDrive/CrackDetection_Backup/best_improved.pt"
)

# Predict cracks
result = model.predict(
    source=image_path,
    conf=0.25,
    verbose=False
)[0]

# Display original and predicted images
fig, ax = plt.subplots(1, 2, figsize=(14, 6))

ax[0].imshow(Image.open(image_path).convert("RGB"))
ax[0].set_title("Original Image")
ax[0].axis("off")

ax[1].imshow(result.plot()[..., ::-1])
ax[1].set_title("YOLO26 Crack Detection")
ax[1].axis("off")

plt.tight_layout()
plt.show()

print("Detected crack instances:", len(result.boxes))
print("Masks:", 0 if result.masks is None else len(result.masks.data))


In [ ]:

from google.colab import files
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt

# Step 1: Upload your crack image
uploaded = files.upload()
image_path = next(iter(uploaded))

# Step 2: Load your improved trained model
model = YOLO(
    "/content/drive/MyDrive/CrackDetection_Backup/best_improved.pt"
)

# Step 3: Detect cracks with more sensitive settings
result = model.predict(
    source=image_path,
    conf=0.10,
    iou=0.45,
    imgsz=960,
    verbose=False
)[0]

# Step 4: Display original and predicted images
fig, ax = plt.subplots(1, 2, figsize=(14, 7))

ax[0].imshow(Image.open(image_path).convert("RGB"))
ax[0].set_title("Original Image")
ax[0].axis("off")

ax[1].imshow(result.plot()[..., ::-1])
ax[1].set_title("YOLO26 Crack Detection - Adjusted")
ax[1].axis("off")

plt.tight_layout()
plt.show()

# Step 5: Print results
print("Detected crack instances:", len(result.boxes))
print(
    "Segmentation masks:",
    0 if result.masks is None else len(result.masks.data)
)


In [ ]:

from google.colab import files
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt

uploaded = files.upload()
image_path = next(iter(uploaded))

model = YOLO(
    "/content/drive/MyDrive/CrackDetection_Backup/best_improved.pt"
)

result = model.predict(
    source=image_path,
    conf=0.25,
    iou=0.35,
    imgsz=960,
    max_det=30,
    verbose=False
)[0]

fig, ax = plt.subplots(1, 2, figsize=(14, 7))

ax[0].imshow(Image.open(image_path).convert("RGB"))
ax[0].set_title("Original Image")
ax[0].axis("off")

ax[1].imshow(result.plot()[..., ::-1])
ax[1].set_title("YOLO26 - Reduced Duplicates")
ax[1].axis("off")

plt.tight_layout()
plt.show()

print("Detected instances:", len(result.boxes))


In [ ]:

from google.colab import files
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt

# Upload an image
uploaded = files.upload()
image_path = next(iter(uploaded))

# Load your trained model
model = YOLO(
    "/content/drive/MyDrive/CrackDetection_Backup/best_improved.pt"
)

# Detect fine cracks with more sensitive settings
result = model.predict(
    source=image_path,
    conf=0.05,
    iou=0.45,
    imgsz=1280,
    max_det=100,
    retina_masks=True,
    verbose=False
)[0]

# Display the result
fig, ax = plt.subplots(1, 2, figsize=(16, 8))

ax[0].imshow(Image.open(image_path).convert("RGB"))
ax[0].set_title("Original Image")
ax[0].axis("off")

ax[1].imshow(result.plot()[..., ::-1])
ax[1].set_title("Fine-Crack Detection")
ax[1].axis("off")

plt.tight_layout()
plt.show()

print("Detected instances:", len(result.boxes))
print("Segmentation masks:",
      0 if result.masks is None else len(result.masks.data))


In [ ]:

!pip install -q sahi

from google.colab import files
from ultralytics import YOLO
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from PIL import Image
import matplotlib.pyplot as plt

# Upload image
uploaded = files.upload()
image_path = next(iter(uploaded))

# Load your trained YOLO26 segmentation model
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path="/content/drive/MyDrive/CrackDetection_Backup/best_improved.pt",
    confidence_threshold=0.20,
    device="cuda:0"
)

# Detect cracks in overlapping image tiles
result = get_sliced_prediction(
    image_path,
    detection_model,
    slice_height=512,
    slice_width=512,
    overlap_height_ratio=0.25,
    overlap_width_ratio=0.25,
    perform_standard_pred=True,
    postprocess_type="GREEDYNMM",
    postprocess_match_metric="IOS",
    postprocess_match_threshold=0.5,
    verbose=1
)

# Display predictions
visual = result.object_prediction_list
print("Detected regions:", len(visual))

# SAHI's standard visualization
from sahi.utils.cv import read_image
from sahi.utils.cv import visualize_object_predictions

image = read_image(image_path)
output = visualize_object_predictions(
    image=image,
    object_prediction_list=visual,
    rect_th=2,
    text_size=0.5
)

plt.figure(figsize=(12, 8))
plt.imshow(output)
plt.title("Tiled Crack Detection")
plt.axis("off")
plt.show()


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sahi.utils.cv import read_image
from sahi.utils.cv import visualize_object_predictions

# Reuse the image path and result from your previous cell
image = read_image(image_path)

output = visualize_object_predictions(
    image=image,
    object_prediction_list=result.object_prediction_list,
    rect_th=2,
    text_size=0.5
)

# Convert visualization output to a displayable image
output = np.asarray(output)

if output.dtype == object:
    output = np.stack(output.tolist()).astype(np.uint8)
else:
    output = output.astype(np.uint8)

plt.figure(figsize=(12, 8))
plt.imshow(output)
plt.title("Tiled Crack Detection")
plt.axis("off")
plt.show()


In [ ]:

import matplotlib.pyplot as plt
from sahi.utils.cv import read_image
from sahi.utils.cv import visualize_object_predictions

image = read_image(image_path)

visual = visualize_object_predictions(
    image=image,
    object_prediction_list=result.object_prediction_list,
    rect_th=2,
    text_size=0.5
)

# Extract the actual image from the returned dictionary
if isinstance(visual, dict):
    output = visual["image"]
else:
    output = visual

plt.figure(figsize=(12, 8))
plt.imshow(output)
plt.title("Tiled Crack Detection")
plt.axis("off")
plt.show()


In [ ]:

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

image = Image.open(image_path).convert("RGB")

fig, ax = plt.subplots(figsize=(12, 14))
ax.imshow(image)

for i, pred in enumerate(result.object_prediction_list):
    if pred.bbox is None:
        continue

    x1, y1, x2, y2 = pred.bbox.to_xyxy()
    score = pred.score.value
    name = pred.category.name

    ax.add_patch(
        Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            fill=False, edgecolor="red", linewidth=1.5
        )
    )

    ax.text(
        x1, max(0, y1 - 5),
        f"{name} {score:.2f}",
        color="white",
        fontsize=10,
        fontweight="bold",
        bbox=dict(facecolor="red", alpha=0.85, pad=2)
    )

ax.set_title("Tiled Crack Detection — Confidence Scores", fontsize=14)
ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
from google.colab import files
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt

# Upload an image
uploaded = files.upload()
image_path = next(iter(uploaded))

# Load your trained model
model = YOLO(
    "/content/drive/MyDrive/CrackDetection_Backup/best_improved.pt"
)

# Detect fine cracks with more sensitive settings
result = model.predict(
    source=image_path,
    conf=0.05,
    iou=0.45,
    imgsz=1280,
    max_det=100,
    retina_masks=True,
    verbose=False
)[0]

# Display the result
fig, ax = plt.subplots(1, 2, figsize=(16, 8))

ax[0].imshow(Image.open(image_path).convert("RGB"))
ax[0].set_title("Original Image")
ax[0].axis("off")

ax[1].imshow(result.plot()[..., ::-1])
ax[1].set_title("Fine-Crack Detection")
ax[1].axis("off")

plt.tight_layout()
plt.show()

print("Detected instances:", len(result.boxes))
print("Segmentation masks:",
      0 if result.masks is None else len(result.masks.data))

In [ ]:
from google.colab import files
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt

# Upload an image
uploaded = files.upload()
image_path = next(iter(uploaded))

# Load your trained model
model = YOLO(
    "/content/drive/MyDrive/CrackDetection_Backup/best_improved.pt"
)

# Detect fine cracks with more sensitive settings
result = model.predict(
    source=image_path,
    conf=0.05,
    iou=0.45,
    imgsz=1280,
    max_det=100,
    retina_masks=True,
    verbose=False
)[0]

# Display the result
fig, ax = plt.subplots(1, 2, figsize=(16, 8))

ax[0].imshow(Image.open(image_path).convert("RGB"))
ax[0].set_title("Original Image")
ax[0].axis("off")

ax[1].imshow(result.plot()[..., ::-1])
ax[1].set_title("Fine-Crack Detection")
ax[1].axis("off")

plt.tight_layout()
plt.show()

print("Detected instances:", len(result.boxes))
print("Segmentation masks:",
      0 if result.masks is None else len(result.masks.data))

In [ ]:

from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import random

root = Path(
    "/content/MultiIndustryCrackDetection/datasets/concrete_seg_clean"
)

image_dir = root / "images" / "train"
label_dir = root / "labels" / "train"

images = list(image_dir.glob("*.jpg"))
random.seed(42)
sample = random.sample(images, min(8, len(images)))

fig, axes = plt.subplots(2, 4, figsize=(16, 9))

for ax, img_path in zip(axes.flat, sample):
    label_path = label_dir / f"{img_path.stem}.txt"

    ax.imshow(Image.open(img_path).convert("RGB"))
    ax.set_title(
        f"{img_path.name}\n"
        f"Label exists: {label_path.exists()}"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Training images:", len(images))
print("Training labels:", len(list(label_dir.glob("*.txt"))))


In [ ]:

from pathlib import Path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import random

root = Path(
    "/content/MultiIndustryCrackDetection/datasets/concrete_seg_clean"
)

image_dir = root / "images" / "train"
label_dir = root / "labels" / "train"

random.seed(10)
images = random.sample(list(image_dir.glob("*.jpg")), 4)

fig, axes = plt.subplots(1, 4, figsize=(18, 6))

for ax, img_path in zip(axes, images):
    image = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(image)
    label_path = label_dir / f"{img_path.stem}.txt"

    if label_path.exists():
        for line in label_path.read_text().splitlines():
            values = list(map(float, line.split()))
            coords = values[1:]

            if len(coords) >= 6 and len(coords) % 2 == 0:
                points = [
                    (coords[i] * image.width,
                     coords[i + 1] * image.height)
                    for i in range(0, len(coords), 2)
                ]
                draw.line(points + [points[0]], fill="red", width=3)

    ax.imshow(image)
    ax.set_title(img_path.name[:22])
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
from google.colab import files
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt

# Upload an image
uploaded = files.upload()
image_path = next(iter(uploaded))

# Load your trained model
model = YOLO(
    "/content/drive/MyDrive/CrackDetection_Backup/best_improved.pt"
)

# Detect fine cracks with more sensitive settings
result = model.predict(
    source=image_path,
    conf=0.05,
    iou=0.45,
    imgsz=1280,
    max_det=100,
    retina_masks=True,
    verbose=False
)[0]

# Display the result
fig, ax = plt.subplots(1, 2, figsize=(16, 8))

ax[0].imshow(Image.open(image_path).convert("RGB"))
ax[0].set_title("Original Image")
ax[0].axis("off")

ax[1].imshow(result.plot()[..., ::-1])
ax[1].set_title("Fine-Crack Detection")
ax[1].axis("off")

plt.tight_layout()
plt.show()

print("Detected instances:", len(result.boxes))
print("Segmentation masks:",
      0 if result.masks is None else len(result.masks.data))

In [ ]:

from google.colab import files
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt

# 1. Upload your concrete crack image
uploaded = files.upload()
image_path = next(iter(uploaded))

# 2. Load your improved model
model = YOLO(
    "/content/drive/MyDrive/CrackDetection_Backup/best_improved.pt"
)

# 3. Predict with stricter duplicate suppression
result = model.predict(
    source=image_path,
    conf=0.25,
    iou=0.35,
    imgsz=960,
    max_det=30,
    retina_masks=True,
    agnostic_nms=True,
    verbose=False
)[0]

# 4. Display original and prediction
fig, ax = plt.subplots(1, 2, figsize=(15, 7))

ax[0].imshow(Image.open(image_path).convert("RGB"))
ax[0].set_title("Original Image")
ax[0].axis("off")

ax[1].imshow(result.plot()[..., ::-1])
ax[1].set_title("Crack Detection - Reduced Duplicates")
ax[1].axis("off")

plt.tight_layout()
plt.show()

print("Remaining detections:", len(result.boxes))
print(
    "Segmentation masks:",
    0 if result.masks is None else len(result.masks.data)
)


In [ ]:

from google.colab import files
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import cv2

# 1. Upload image
uploaded = files.upload()
image_path = next(iter(uploaded))

# 2. Load trained model
model = YOLO(
    "/content/drive/MyDrive/CrackDetection_Backup/best_improved.pt"
)

# 3. Predict with moderate confidence
result = model.predict(
    source=image_path,
    conf=0.25,
    iou=0.45,
    imgsz=960,
    retina_masks=True,
    verbose=False
)[0]

# 4. Merge highly overlapping masks for visualization
img = np.array(Image.open(image_path).convert("RGB"))
overlay = img.copy()

if result.masks is not None:
    masks = result.masks.data.cpu().numpy()
    scores = result.boxes.conf.cpu().numpy()

    # Process highest-confidence masks first
    order = np.argsort(scores)[::-1]
    kept = []
    occupied = np.zeros(masks.shape[1:], dtype=bool)

    for idx in order:
        mask = masks[idx] > 0.5
        area = mask.sum()

        if area == 0:
            continue

        # Skip a detection if most of it duplicates an existing mask
        duplicate = False
        for prev_mask in kept:
            intersection = np.logical_and(mask, prev_mask).sum()
            smaller_area = min(area, prev_mask.sum())

            if smaller_area > 0 and intersection / smaller_area > 0.80:
                duplicate = True
                break

        if not duplicate:
            kept.append(mask)

    # Draw the kept masks
    for i, mask in enumerate(kept, start=1):
        resized = cv2.resize(
            mask.astype(np.uint8),
            (img.shape[1], img.shape[0]),
            interpolation=cv2.INTER_NEAREST
        ).astype(bool)

        # Green outlines
        contours, _ = cv2.findContours(
            resized.astype(np.uint8),
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )
        cv2.drawContours(overlay, contours, -1, (0, 255, 0), 2)

    print("Original detections:", len(scores))
    print("After duplicate-mask filtering:", len(kept))
else:
    print("No cracks detected.")

# 5. Display a clean mask overlay
plt.figure(figsize=(14, 7))
plt.imshow(overlay)
plt.title("Concrete Cracks — Filtered Segmentation")
plt.axis("off")
plt.show()
print("Final detected crack instances:", len(kept))


In [ ]:

from google.colab import files
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt

# 1. Upload your concrete crack image
uploaded = files.upload()
image_path = next(iter(uploaded))

# 2. Load your improved model
model = YOLO(
    "/content/drive/MyDrive/CrackDetection_Backup/best_improved.pt"
)

# 3. Predict with stricter duplicate suppression
result = model.predict(
    source=image_path,
    conf=0.25,
    iou=0.35,
    imgsz=960,
    max_det=30,
    retina_masks=True,
    agnostic_nms=True,
    verbose=False
)[0]

# 4. Display original and prediction
fig, ax = plt.subplots(1, 2, figsize=(15, 7))

ax[0].imshow(Image.open(image_path).convert("RGB"))
ax[0].set_title("Original Image")
ax[0].axis("off")

ax[1].imshow(result.plot()[..., ::-1])
ax[1].set_title("Crack Detection - Reduced Duplicates")
ax[1].axis("off")

plt.tight_layout()
plt.show()

print("Remaining detections:", len(result.boxes))
print(
    "Segmentation masks:",
    0 if result.masks is None else len(result.masks.data)
)

In [ ]:

from pathlib import Path
from collections import Counter
from PIL import Image
import numpy as np

root = Path(
    "/content/MultiIndustryCrackDetection/datasets/concrete_seg_clean"
)

image_dir = root / "images" / "train"
label_dir = root / "labels" / "train"

images = list(image_dir.glob("*.jpg"))
dimensions = Counter()
polygon_counts = []
small_polygons = 0

for img_path in images:
    with Image.open(img_path) as im:
        dimensions[im.size] += 1

    label_path = label_dir / f"{img_path.stem}.txt"
    if not label_path.exists():
        continue

    lines = label_path.read_text().splitlines()
    polygon_counts.append(len(lines))

    for line in lines:
        values = line.split()
        # Class ID plus normalized x,y coordinate pairs
        if len(values) >= 7 and (len(values) - 1) % 2 == 0:
            coords = np.array(list(map(float, values[1:]))).reshape(-1, 2)
            width = coords[:, 0].max() - coords[:, 0].min()
            height = coords[:, 1].max() - coords[:, 1].min()

            if width < 0.02 or height < 0.02:
                small_polygons += 1

print("Training images:", len(images))
print("Image dimensions and counts:", dimensions.most_common(10))
print("Average polygons per image:",
      round(np.mean(polygon_counts), 2) if polygon_counts else 0)
print("Very narrow or small polygons:", small_polygons)


In [ ]:

import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Total GPU memory:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")
else:
    print("No GPU available. Please enable GPU in Runtime > Change runtime type.")


In [ ]:

from ultralytics import YOLO
from pathlib import Path
import torch
import shutil

# Load the existing improved model
model = YOLO(
    "/content/drive/MyDrive/CrackDetection_Backup/best_improved.pt"
)

# Fine-tune at higher resolution
results = model.train(
    data="/content/MultiIndustryCrackDetection/datasets/concrete_seg_clean/data.yaml",
    epochs=30,
    imgsz=960,
    batch=4,
    device=0,
    workers=2,
    patience=8,
    project="/content/MultiIndustryCrackDetection/runs",
    name="concrete_fine_cracks_960",
    exist_ok=False,
    plots=True,
    seed=42
)

# Save the new best checkpoint separately
best_path = Path(
    "/content/MultiIndustryCrackDetection/runs/"
    "concrete_fine_cracks_960/weights/best.pt"
)

backup_path = Path(
    "/content/drive/MyDrive/CrackDetection_Backup/"
    "best_finecracks_960.pt"
)

if best_path.exists():
    shutil.copy2(best_path, backup_path)
    print("New model saved to:", backup_path)
else:
    print("Training finished, but the expected checkpoint was not found.")


In [ ]:

from google.colab import files
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt

uploaded = files.upload()
image_path = next(iter(uploaded))

old_model = YOLO(
    "/content/drive/MyDrive/CrackDetection_Backup/best_improved.pt"
)

new_model = YOLO(
    "/content/drive/MyDrive/CrackDetection_Backup/best_finecracks_960.pt"
)

old_result = old_model.predict(
    image_path, imgsz=640, conf=0.25, iou=0.45, verbose=False
)[0]

new_result = new_model.predict(
    image_path, imgsz=960, conf=0.25, iou=0.45, verbose=False
)[0]

fig, ax = plt.subplots(1, 3, figsize=(19, 7))

ax[0].imshow(Image.open(image_path).convert("RGB"))
ax[0].set_title("Original Image")

ax[1].imshow(old_result.plot()[..., ::-1])
ax[1].set_title(f"Previous Model: {len(old_result.boxes)} detections")

ax[2].imshow(new_result.plot()[..., ::-1])
ax[2].set_title(f"New Model (960): {len(new_result.boxes)} detections")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()

print("Previous model detections:", len(old_result.boxes))
print("New model detections:", len(new_result.boxes))


In [ ]:

from ultralytics import YOLO
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

# Use an existing image from the test dataset
test_dir = Path(
    "/content/MultiIndustryCrackDetection/datasets/concrete_seg_clean/images/test"
)

image_path = next(
    p for p in test_dir.glob("*.jpg")
    if p.name.lower().startswith("crack")
)

# Load both models
old_model = YOLO(
    "/content/drive/MyDrive/CrackDetection_Backup/best_improved.pt"
)

new_model = YOLO(
    "/content/drive/MyDrive/CrackDetection_Backup/best_finecracks_960.pt"
)

# Predict on the SAME image
old_result = old_model.predict(
    source=str(image_path),
    imgsz=640,
    conf=0.25,
    verbose=False
)[0]

new_result = new_model.predict(
    source=str(image_path),
    imgsz=960,
    conf=0.25,
    verbose=False
)[0]

# Compare predictions
fig, ax = plt.subplots(1, 3, figsize=(18, 7))

ax[0].imshow(Image.open(image_path).convert("RGB"))
ax[0].set_title("Original Image")

ax[1].imshow(old_result.plot()[..., ::-1])
ax[1].set_title(f"Previous Model: {len(old_result.boxes)} detections")

ax[2].imshow(new_result.plot()[..., ::-1])
ax[2].set_title(f"New Model: {len(new_result.boxes)} detections")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()

print("Test image:", image_path.name)
print("Previous model detections:", len(old_result.boxes))
print("New model detections:", len(new_result.boxes))


In [ ]:
from ultralytics import YOLO
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

# Load the new high-resolution model
model = YOLO(
    "/content/drive/MyDrive/CrackDetection_Backup/best_finecracks_960.pt"
)

# Use the same test image
image_path = Path(
    "/content/MultiIndustryCrackDetection/datasets/concrete_seg_clean/images/test/crack_(177)_JPG.rf.iDKf9cjWxF6SeyEEATzt.jpg"
)

# Predict without drawing bounding boxes
result = model.predict(
    source=str(image_path),
    imgsz=960,
    conf=0.25,
    verbose=False
)[0]

# Show the original image and segmentation masks
fig, ax = plt.subplots(1, 2, figsize=(14, 6))

ax[0].imshow(Image.open(image_path).convert("RGB"))
ax[0].set_title("Original Image")

ax[1].imshow(result.plot(boxes=False, labels=False)[..., ::-1])
ax[1].set_title("Segmentation Masks Only")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()

print("Number of predicted masks:", len(result.masks) if result.masks is not None else 0)

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

# Find an existing test image automatically
test_dir = Path(
    "/content/MultiIndustryCrackDetection/datasets/concrete_seg_clean/images/test"
)

image_files = []
for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
    image_files.extend(test_dir.glob(ext))

if not image_files:
    raise FileNotFoundError(f"No images found in {test_dir}")

# Choose the first available image
image_path = image_files[0]
print("Image selected:", image_path.name)

# Load the new model
model = YOLO(
    "/content/drive/MyDrive/CrackDetection_Backup/best_finecracks_960.pt"
)

# Predict
result = model.predict(
    source=str(image_path),
    imgsz=960,
    conf=0.25,
    verbose=False
)[0]

# Display original image and segmentation masks
fig, ax = plt.subplots(1, 2, figsize=(14, 6))

ax[0].imshow(Image.open(image_path).convert("RGB"))
ax[0].set_title("Original Image")

ax[1].imshow(result.plot(boxes=False, labels=False)[..., ::-1])
ax[1].set_title("Segmentation Masks Only")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()

print(
    "Number of predicted masks:",
    len(result.masks) if result.masks is not None else 0
)

In [ ]:
from ultralytics import YOLO
from google.colab import files
from IPython.display import display
from PIL import Image
import matplotlib.pyplot as plt

# 1. Upload an image from your computer
uploaded = files.upload()

if uploaded:
    image_path = next(iter(uploaded))

    # 2. Load your trained model
    model = YOLO(
        "/content/drive/MyDrive/CrackDetection_Backup/best_finecracks_960.pt"
    )

    # 3. Detect cracks in the uploaded image
    result = model.predict(
        source=image_path,
        imgsz=960,
        conf=0.25,
        verbose=False
    )[0]

    # 4. Display original and predicted images
    fig, ax = plt.subplots(1, 2, figsize=(14, 6))

    ax[0].imshow(Image.open(image_path).convert("RGB"))
    ax[0].set_title("Uploaded Image")

    ax[1].imshow(result.plot()[..., ::-1])
    ax[1].set_title(
        f"Detected Regions: {len(result.boxes)}"
    )

    for a in ax:
        a.axis("off")

    plt.tight_layout()
    plt.show()

    print("Uploaded file:", image_path)
    print("Detected regions:", len(result.boxes))
else:
    print("No image uploaded. Run this cell again.")

In [ ]:
from ultralytics import YOLO
from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt

uploaded = files.upload()

if uploaded:
    image_path = next(iter(uploaded))

    model = YOLO(
        "/content/drive/MyDrive/CrackDetection_Backup/best_finecracks_960.pt"
    )

    result = model.predict(
        source=image_path,
        imgsz=960,
        conf=0.10,
        iou=0.7,
        verbose=False
    )[0]

    fig, ax = plt.subplots(1, 2, figsize=(14, 7))

    ax[0].imshow(Image.open(image_path).convert("RGB"))
    ax[0].set_title("Original Image")

    ax[1].imshow(result.plot()[..., ::-1])
    ax[1].set_title(f"Prediction at confidence 0.10: {len(result.boxes)} regions")

    for a in ax:
        a.axis("off")

    plt.tight_layout()
    plt.show()

    print("Detected regions:", len(result.boxes))

In [ ]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt

base = Path(
    "/content/MultiIndustryCrackDetection/datasets/concrete_seg_clean"
)

image_path = next((base / "images/train").glob("*.jpg"))
label_path = base / "labels/train" / f"{image_path.stem}.txt"

image = cv2.imread(str(image_path))
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 7))
plt.imshow(image)
plt.title(f"Training image: {image_path.name}")
plt.axis("off")
plt.show()

print("Image:", image_path.name)
print("Label file exists:", label_path.exists())

if label_path.exists():
    lines = label_path.read_text().strip().splitlines()
    print("Number of labelled crack instances:", len(lines))
    print("First label preview:", lines[0][:200] if lines and lines[0] else "Empty label")

In [ ]:
from google.colab import files
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt

# 1. Upload your own image
uploaded = files.upload()

if not uploaded:
    print("No image uploaded. Run the cell again.")
else:
    image_path = next(iter(uploaded))

    # 2. Load your trained model
    model = YOLO(
        "/content/drive/MyDrive/CrackDetection_Backup/best_finecracks_960.pt"
    )

    # 3. Predict cracks
    result = model.predict(
        source=image_path,
        imgsz=960,
        conf=0.25,
        verbose=False
    )[0]

    # 4. Display results
    fig, ax = plt.subplots(1, 2, figsize=(14, 7))

    ax[0].imshow(Image.open(image_path).convert("RGB"))
    ax[0].set_title("Your Uploaded Image")

    ax[1].imshow(result.plot()[..., ::-1])
    ax[1].set_title(f"Detected Regions: {len(result.boxes)}")

    for a in ax:
        a.axis("off")

    plt.tight_layout()
    plt.show()

    print("Image:", image_path)
    print("Detected regions:", len(result.boxes))

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import shutil

# Existing labelled concrete-crack dataset
data_yaml = (
    "/content/MultiIndustryCrackDetection/"
    "datasets/concrete_seg_clean/data.yaml"
)

# Start from your preferred checkpoint
model = YOLO(
    "/content/drive/MyDrive/CrackDetection_Backup/best_improved.pt"
)

# Fine-tune at higher resolution
results = model.train(
    data=data_yaml,
    epochs=50,
    imgsz=1280,
    batch=2,
    device=0,
    workers=2,
    patience=12,
    project="/content/MultiIndustryCrackDetection/runs",
    name="concrete_cracks_1280",
    exist_ok=False,
    plots=True,
    seed=42
)

# Save the best checkpoint to Drive without overwriting old models
best_path = Path(
    "/content/MultiIndustryCrackDetection/runs/"
    "concrete_cracks_1280/weights/best.pt"
)

drive_path = Path(
    "/content/drive/MyDrive/CrackDetection_Backup/"
    "best_concrete_cracks_1280.pt"
)

if best_path.exists():
    shutil.copy2(best_path, drive_path)
    print("New model saved to:", drive_path)
else:
    print("Best checkpoint not found. Check the training output.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q ultralytics transformers timm accelerate opencv-python matplotlib pillow

In [ ]:
%cd /content

!git clone https://github.com/DepthAnything/Depth-Anything-V2.git
%cd /content/Depth-Anything-V2/metric_depth

!pip install -q -r requirements.txt

In [ ]:
!mkdir -p checkpoints

!wget -O checkpoints/depth_anything_v2_metric_vkitti_vitb.pth \
"https://huggingface.co/DepthAnything/Depth-Anything-V2-Metric-VKITTI-Base/resolve/main/depth_anything_v2_metric_vkitti_vitb.pth"

In [ ]:
%cd /content/Depth-Anything-V2/metric_depth

!wget -O checkpoints/depth_anything_v2_metric_vkitti_vitb.pth \
"https://huggingface.co/depth-anything/Depth-Anything-V2-Metric-VKITTI-Base/resolve/main/depth_anything_v2_metric_vkitti_vitb.pth?download=true"

In [ ]:
import os

checkpoint_path = (
    "/content/Depth-Anything-V2/metric_depth/"
    "checkpoints/depth_anything_v2_metric_vkitti_vitb.pth"
)

print("Checkpoint exists:", os.path.exists(checkpoint_path))
print("Checkpoint size (MB):", round(
    os.path.getsize(checkpoint_path) / (1024**2), 2
))

In [ ]:
import sys
import torch

sys.path.insert(0, "/content/Depth-Anything-V2/metric_depth")

from depth_anything_v2.dpt import DepthAnythingV2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model_configs = {
    "vitb": {
        "encoder": "vitb",
        "features": 128,
        "out_channels": [96, 192, 384, 768]
    }
}

depth_model = DepthAnythingV2(
    **model_configs["vitb"],
    max_depth=80
)

depth_model.load_state_dict(
    torch.load(checkpoint_path, map_location="cpu")
)

depth_model = depth_model.to(DEVICE).eval()

print("Depth model loaded successfully!")
print("Device:", DEVICE)

In [ ]:
from google.colab import files
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Upload your concrete crack image
uploaded = files.upload()

if not uploaded:
    raise ValueError("No image uploaded.")

image_path = next(iter(uploaded))

# Load your trained YOLO model
YOLO_PATH = (
    "/content/drive/MyDrive/CrackDetection_Backup/"
    "best_concrete_cracks_1280.pt"
)

yolo_model = YOLO(YOLO_PATH)

# Read the image
image_bgr = cv2.imread(image_path)

if image_bgr is None:
    raise ValueError("Could not read the image.")

image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
height, width = image_bgr.shape[:2]

# 1. Detect concrete cracks
result = yolo_model.predict(
    source=image_path,
    conf=0.05,
    iou=0.45,
    imgsz=1280,
    max_det=100,
    retina_masks=True,
    verbose=False
)[0]

if result.masks is None:
    raise ValueError(
        "No cracks detected. Try another clear concrete image."
    )

# Combine crack segmentation masks
crack_mask = np.zeros((height, width), dtype=bool)

for mask in result.masks.data.cpu().numpy():
    mask = cv2.resize(
        mask.astype(np.float32),
        (width, height),
        interpolation=cv2.INTER_NEAREST
    )
    crack_mask |= mask > 0.5

# 2. Predict monocular metric scene depth
depth_map = depth_model.infer_image(image_bgr)

depth_map = cv2.resize(
    depth_map.astype(np.float32),
    (width, height),
    interpolation=cv2.INTER_LINEAR
)

# 3. Compare crack pixels with nearby concrete
kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE, (21, 21)
)

dilated_mask = cv2.dilate(
    crack_mask.astype(np.uint8), kernel
).astype(bool)

nearby_mask = dilated_mask & (~crack_mask)

valid = np.isfinite(depth_map) & (depth_map > 0)

crack_depth_values = depth_map[crack_mask & valid]
surface_values = depth_map[nearby_mask & valid]

if len(crack_depth_values) == 0 or len(surface_values) == 0:
    raise ValueError("Insufficient valid depth pixels.")

crack_distance = float(np.median(crack_depth_values))
surface_distance = float(np.median(surface_values))

difference = crack_distance - surface_distance

print(f"Predicted camera distance at crack: {crack_distance:.3f} m")
print(f"Predicted camera distance at surface: {surface_distance:.3f} m")
print(f"Experimental local depth difference: {difference:+.4f} m")
print("\nThis difference is NOT a validated measurement of crack penetration.")

# 4. Display results
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(image_rgb)
axes[0].set_title("Original Image")
axes[0].axis("off")

axes[1].imshow(result.plot()[..., ::-1])
axes[1].set_title("YOLO Crack Segmentation")
axes[1].axis("off")

im = axes[2].imshow(depth_map, cmap="plasma")
axes[2].contour(
    crack_mask, levels=[0.5], colors="white", linewidths=0.8
)
axes[2].set_title("Predicted Scene Depth (m)")
axes[2].axis("off")
plt.colorbar(im, ax=axes[2], label="Metres")

plt.tight_layout()
plt.show()

In [ ]:
print("Depth map statistics")
print("Minimum:", float(np.nanmin(depth_map)), "m")
print("Maximum:", float(np.nanmax(depth_map)), "m")
print("Mean:", float(np.nanmean(depth_map)), "m")
print("Standard deviation:", float(np.nanstd(depth_map)), "m")

plt.figure(figsize=(8, 6))
plt.imshow(depth_map, cmap="plasma")
plt.colorbar(label="Predicted camera distance (m)")
plt.title("Concrete Image — Monocular Depth Map")
plt.axis("off")
plt.show()

plt.figure(figsize=(8, 6))
plt.imshow(image_rgb)
plt.contour(crack_mask, levels=[0.5], colors="red", linewidths=1)
plt.title("YOLO Crack Mask on Original Image")
plt.axis("off")
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Binary YOLO crack mask
crack = crack_mask.astype(np.uint8)

# Narrow band around the detected crack
kernel_inner = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE, (7, 7)
)
inner_band = cv2.dilate(crack, kernel_inner).astype(bool)
inner_band &= ~crack.astype(bool)

# Wider surrounding concrete band
kernel_outer = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE, (41, 41)
)
outer_band = cv2.dilate(crack, kernel_outer).astype(bool)
outer_band &= ~cv2.dilate(
    crack, kernel_inner
).astype(bool)

# Exclude invalid depth pixels
valid = np.isfinite(depth_map) & (depth_map > 0)

inner_values = depth_map[inner_band & valid]
outer_values = depth_map[outer_band & valid]

if len(inner_values) == 0 or len(outer_values) == 0:
    print("Not enough pixels. Try a larger image or a wider band.")
else:
    inner_m = float(np.median(inner_values))
    outer_m = float(np.median(outer_values))

    print(f"Near-crack predicted distance: {inner_m:.4f} m")
    print(f"Surrounding predicted distance: {outer_m:.4f} m")
    print(f"Difference: {inner_m - outer_m:+.4f} m")
    print("This is a scene-depth comparison, NOT physical crack depth.")

    plt.figure(figsize=(8, 6))
    plt.imshow(image_rgb)
    plt.contour(inner_band, levels=[0.5], colors="yellow")
    plt.contour(outer_band, levels=[0.5], colors="cyan")
    plt.title("Yellow: near-crack band | Cyan: surrounding concrete")
    plt.axis("off")
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

# 1. Load the models
YOLO_PATH = (
    "/content/drive/MyDrive/CrackDetection_Backup/"
    "best_concrete_cracks_1280.pt"
)

yolo_model = YOLO(YOLO_PATH)

# Reuse the uploaded image if it still exists
image_path = "WhatsApp Image 2026-10-26 at 6.00.59 PM.jpeg"

image_bgr = cv2.imread(image_path)

if image_bgr is None:
    from google.colab import files
    uploaded = files.upload()

    if not uploaded:
        raise ValueError("Please upload your concrete crack image.")

    image_path = next(iter(uploaded))
    image_bgr = cv2.imread(image_path)

if image_bgr is None:
    raise ValueError("Could not read the image.")

height, width = image_bgr.shape[:2]
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

# 2. Recreate the YOLO crack mask
result = yolo_model.predict(
    source=image_path,
    conf=0.05,
    iou=0.45,
    imgsz=1280,
    max_det=100,
    retina_masks=True,
    verbose=False
)[0]

if result.masks is None:
    raise ValueError(
        "YOLO found no crack masks. Try another image."
    )

crack_mask = np.zeros((height, width), dtype=bool)

for mask in result.masks.data.cpu().numpy():
    mask = cv2.resize(
        mask.astype(np.float32),
        (width, height),
        interpolation=cv2.INTER_NEAREST
    )
    crack_mask |= mask > 0.5

# 3. Recreate the monocular depth map
depth_map = depth_model.infer_image(image_bgr)
depth_map = cv2.resize(
    depth_map.astype(np.float32),
    (width, height),
    interpolation=cv2.INTER_LINEAR
)

# 4. Build near-crack and surrounding-concrete bands
crack_uint8 = crack_mask.astype(np.uint8)

inner_kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE, (7, 7)
)
outer_kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE, (41, 41)
)

inner_dilated = cv2.dilate(crack_uint8, inner_kernel).astype(bool)
outer_dilated = cv2.dilate(crack_uint8, outer_kernel).astype(bool)

inner_band = inner_dilated & ~crack_mask
outer_band = outer_dilated & ~inner_dilated

valid = np.isfinite(depth_map) & (depth_map > 0)

inner_values = depth_map[inner_band & valid]
outer_values = depth_map[outer_band & valid]

if len(inner_values) == 0 or len(outer_values) == 0:
    raise ValueError("Not enough valid pixels to compare.")

inner_m = float(np.median(inner_values))
outer_m = float(np.median(outer_values))

print(f"Near-crack predicted distance: {inner_m:.4f} m")
print(f"Surrounding predicted distance: {outer_m:.4f} m")
print(f"Scene-depth difference: {inner_m - outer_m:+.4f} m")
print("This is NOT a measurement of physical crack penetration.")

# 5. Display the bands
plt.figure(figsize=(9, 7))
plt.imshow(image_rgb)
plt.contour(inner_band, levels=[0.5], colors="yellow")
plt.contour(outer_band, levels=[0.5], colors="cyan")
plt.title("Yellow: near-crack band | Cyan: surrounding concrete")
plt.axis("off")
plt.show()

In [ ]:
!pip install -q ultralytics

In [ ]:
import ultralytics
print("Ultralytics version:", ultralytics.__version__)

import torch
print("GPU available:", torch.cuda.is_available())

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU is not enabled yet. Check Runtime settings.")

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

In [ ]:
import os

paths = {
    "YOLO model": "/content/drive/MyDrive/CrackDetection_Backup/best_concrete_cracks_1280.pt",
    "Depth checkpoint": "/content/Depth-Anything-V2/metric_depth/checkpoints/depth_anything_v2_metric_vkitti_vitb.pth",
    "Depth model code": "/content/Depth-Anything-V2/metric_depth/depth_anything_v2/dpt.py",
    "Uploaded concrete image": "/content/WhatsApp Image 2026-10-26 at 6.00.59 PM.jpeg"
}

for name, path in paths.items():
    print(f"{name}: {'FOUND' if os.path.exists(path) else 'NOT FOUND'}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

folder = "/content/drive/MyDrive/CrackDetection_Backup"

print("Folder exists:", os.path.exists(folder))

if os.path.exists(folder):
    for name in os.listdir(folder):
        if name.endswith(".pt"):
            path = os.path.join(folder, name)
            size_mb = os.path.getsize(path) / (1024**2)
            print(f"{name} — {size_mb:.2f} MB")
else:
    print("CrackDetection_Backup folder not found in Google Drive.")

In [ ]:
%cd /content

!git clone https://github.com/DepthAnything/Depth-Anything-V2.git

In [ ]:
%cd /content

!git clone https://github.com/DepthAnything/Depth-Anything-V2.git

In [ ]:
import os

repo = "/content/Depth-Anything-V2/metric_depth"

print("Repository exists:", os.path.isdir(repo))
print(
    "Depth model code exists:",
    os.path.isfile(repo + "/depth_anything_v2/dpt.py")
)
print(
    "Checkpoint exists:",
    os.path.isfile(
        repo + "/checkpoints/depth_anything_v2_metric_vkitti_vitb.pth"
    )
)

In [ ]:
%cd /content/Depth-Anything-V2/metric_depth

!mkdir -p checkpoints

!wget -O checkpoints/depth_anything_v2_metric_vkitti_vitb.pth \
"https://huggingface.co/depth-anything/Depth-Anything-V2-Metric-VKITTI-Base/resolve/main/depth_anything_v2_metric_vkitti_vitb.pth?download=true"

In [ ]:
import os

checkpoint_path = (
    "/content/Depth-Anything-V2/metric_depth/"
    "checkpoints/depth_anything_v2_metric_vkitti_vitb.pth"
)

print("Checkpoint exists:", os.path.exists(checkpoint_path))

if os.path.exists(checkpoint_path):
    print(
        "Size (MB):",
        round(os.path.getsize(checkpoint_path) / (1024**2), 2)
    )

In [ ]:
!pip install -q ultralytics timm

import os
import sys
import torch
from ultralytics import YOLO

sys.path.insert(
    0, "/content/Depth-Anything-V2/metric_depth"
)

from depth_anything_v2.dpt import DepthAnythingV2

# Use CPU because GPU access is currently unavailable
DEVICE = "cpu"

# Load your trained concrete crack model
yolo_path = (
    "/content/drive/MyDrive/CrackDetection_Backup/"
    "best_concrete_cracks_1280.pt"
)

# Load the downloaded depth checkpoint
checkpoint_path = (
    "/content/Depth-Anything-V2/metric_depth/"
    "checkpoints/depth_anything_v2_metric_vkitti_vitb.pth"
)

assert os.path.exists(yolo_path), "YOLO checkpoint not found in Drive."
assert os.path.exists(checkpoint_path), "Depth checkpoint not found."

yolo_model = YOLO(yolo_path)

depth_model = DepthAnythingV2(
    encoder="vitb",
    features=128,
    out_channels=[96, 192, 384, 768],
    max_depth=80
)

state = torch.load(checkpoint_path, map_location="cpu")
depth_model.load_state_dict(state)
depth_model = depth_model.to(DEVICE).eval()

print("YOLO model loaded successfully.")
print("Depth model loaded successfully.")
print("Device:", DEVICE)

In [ ]:
from google.colab import files
import cv2
import numpy as np
import matplotlib.pyplot as plt

uploaded = files.upload()

if not uploaded:
    raise ValueError("No image uploaded.")

image_path = next(iter(uploaded))
image_bgr = cv2.imread(image_path)

if image_bgr is None:
    raise ValueError("Could not read the uploaded image.")

image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
height, width = image_bgr.shape[:2]

# 1. Detect cracks with YOLO
result = yolo_model.predict(
    source=image_path,
    conf=0.05,
    iou=0.45,
    imgsz=1280,
    max_det=100,
    retina_masks=True,
    verbose=False
)[0]

# 2. Generate the monocular scene-depth map
depth_map = depth_model.infer_image(image_bgr)

depth_map = cv2.resize(
    depth_map.astype(np.float32),
    (width, height),
    interpolation=cv2.INTER_LINEAR
)

# 3. Display the original image and depth map
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(image_rgb)
axes[0].set_title("Original Concrete Image")
axes[0].axis("off")

axes[1].imshow(result.plot()[..., ::-1])
axes[1].set_title("YOLO Crack Detection")
axes[1].axis("off")

im = axes[2].imshow(depth_map, cmap="plasma")
axes[2].set_title("Predicted Scene Depth")
axes[2].axis("off")
plt.colorbar(im, ax=axes[2], label="Model output (nominal metres)")

plt.tight_layout()
plt.show()

print("Detected crack instances:", 0 if result.masks is None else len(result.masks.data))
print("Image processed successfully.")
print("Note: scene depth is not physical crack penetration depth.")

In [ ]:
from google.colab import files
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt
import cv2
import numpy as np

uploaded = files.upload()
if not uploaded:
    raise ValueError("Upload a concrete crack image.")

image_path = next(iter(uploaded))
image = Image.open(image_path).convert("RGB")

model_paths = {
    "Best Improved": (
        "/content/drive/MyDrive/CrackDetection_Backup/"
        "best_improved.pt"
    ),
    "Best 1280": (
        "/content/drive/MyDrive/CrackDetection_Backup/"
        "best_concrete_cracks_1280.pt"
    )
}

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

axes[0].imshow(image)
axes[0].set_title("Original Image")
axes[0].axis("off")

for ax, (name, path) in zip(axes[1:], model_paths.items()):
    model = YOLO(path)

    result = model.predict(
        source=image_path,
        conf=0.10,
        iou=0.45,
        imgsz=1280,
        max_det=100,
        retina_masks=True,
        verbose=False
    )[0]

    ax.imshow(result.plot()[..., ::-1])
    ax.set_title(
        f"{name}\nDetections: {len(result.boxes)}"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from ultralytics import YOLO
from PIL import Image
import numpy as np

image = Image.open(image_path).convert("RGB")

model_paths = {
    "Best Improved": "/content/drive/MyDrive/CrackDetection_Backup/best_improved.pt",
    "Best 1280": "/content/drive/MyDrive/CrackDetection_Backup/best_concrete_cracks_1280.pt"
}

fig, axes = plt.subplots(1, 3, figsize=(18, 7))

axes[0].imshow(image)
axes[0].set_title("Original Image")
axes[0].axis("off")

for ax, (name, path) in zip(axes[1:], model_paths.items()):
    model = YOLO(path)
    pred = model.predict(
        source=image_path,
        conf=0.10,
        iou=0.45,
        imgsz=1280,
        max_det=100,
        retina_masks=True,
        verbose=False
    )[0]

    # Draw segmentation masks only: no boxes or labels
    overlay = pred.plot(
        boxes=False,
        labels=False,
        conf=False,
        masks=True
    )

    ax.imshow(overlay[..., ::-1])
    ax.set_title(f"{name} — Masks Only")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from ultralytics import YOLO
import numpy as np

models = {
    "Best Improved": "/content/drive/MyDrive/CrackDetection_Backup/best_improved.pt",
    "Best 1280": "/content/drive/MyDrive/CrackDetection_Backup/best_concrete_cracks_1280.pt"
}

for name, path in models.items():
    model = YOLO(path)
    pred = model.predict(
        source=image_path,
        conf=0.10,
        imgsz=1280,
        max_det=100,
        retina_masks=True,
        verbose=False
    )[0]

    print(f"\n{name}")
    print("Detections:", len(pred.boxes))

    if pred.masks is not None:
        masks = pred.masks.data.cpu().numpy()
        print("Mask count:", len(masks))
        print("Total predicted mask pixels:", int((masks > 0.5).sum()))
        print("Confidence scores:", np.round(
            pred.boxes.conf.cpu().numpy(), 3
        ))
    else:
        print("No segmentation masks detected.")

In [ ]:
from ultralytics import YOLO
import numpy as np

model = YOLO(
    "/content/drive/MyDrive/CrackDetection_Backup/best_improved.pt"
)

pred = model.predict(
    source=image_path,
    conf=0.10,
    imgsz=1280,
    max_det=100,
    retina_masks=True,
    verbose=False
)[0]

if pred.masks is None:
    print("No masks detected.")
else:
    masks = pred.masks.data.cpu().numpy() > 0.5

    print("Mask count:", len(masks))

    for i in range(len(masks)):
        for j in range(i + 1, len(masks)):
            intersection = np.logical_and(masks[i], masks[j]).sum()
            union = np.logical_or(masks[i], masks[j]).sum()

            iou = intersection / union if union else 0

            if iou > 0.10:
                print(f"Masks {i+1} and {j+1}: overlap IoU = {iou:.3f}")

In [ ]:
import numpy as np
import cv2

# Use the existing image and depth map
# Recreate the crack mask with the selected YOLO model
from ultralytics import YOLO

model = YOLO(
    "/content/drive/MyDrive/CrackDetection_Backup/best_improved.pt"
)

result = model.predict(
    source=image_path,
    conf=0.10,
    imgsz=1280,
    retina_masks=True,
    verbose=False
)[0]

if result.masks is None:
    raise ValueError("No crack masks detected.")

height, width = depth_map.shape
crack_mask = np.zeros((height, width), dtype=bool)

for mask in result.masks.data.cpu().numpy():
    resized = cv2.resize(
        mask.astype(np.float32),
        (width, height),
        interpolation=cv2.INTER_NEAREST
    )
    crack_mask |= resized > 0.5

# Compare crack pixels with nearby concrete
kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE, (41, 41)
)

expanded = cv2.dilate(
    crack_mask.astype(np.uint8), kernel
).astype(bool)

nearby_mask = expanded & ~crack_mask
valid = np.isfinite(depth_map) & (depth_map > 0)

crack_values = depth_map[crack_mask & valid]
surface_values = depth_map[nearby_mask & valid]

if len(crack_values) == 0 or len(surface_values) == 0:
    raise ValueError("Insufficient valid depth pixels.")

crack_distance_m = float(np.median(crack_values))
surface_distance_m = float(np.median(surface_values))
difference_m = crack_distance_m - surface_distance_m

print(f"Predicted distance at crack pixels: {crack_distance_m:.4f} m")
print(f"Predicted distance at nearby concrete: {surface_distance_m:.4f} m")
print(f"Experimental scene-depth difference: {difference_m:+.4f} m")
print("\nNote: This is NOT validated physical crack penetration depth.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Reuse the variables already calculated:
# image_rgb, result, depth_map, crack_mask, difference_m

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(image_rgb)
axes[0].set_title("Concrete Crack — Original")
axes[0].axis("off")

axes[1].imshow(result.plot(boxes=False, labels=False)[..., ::-1])
axes[1].set_title("Detected Crack Regions")
axes[1].axis("off")

im = axes[2].imshow(depth_map, cmap="plasma")
axes[2].set_title("Predicted Scene Depth")
axes[2].axis("off")
plt.colorbar(im, ax=axes[2], label="Model output (m)")

plt.suptitle(
    f"Experimental Scene-Depth Difference: {difference_m:+.4f} m",
    fontsize=14
)
plt.tight_layout()
plt.show()

print(f"Experimental scene-depth difference: {difference_m:+.4f} m")
print("Not a validated physical crack penetration measurement.")

In [ ]:
output_path = "/content/concrete_depth_estimation_result.png"

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(image_rgb)
axes[0].set_title("Original Concrete Image")
axes[0].axis("off")

axes[1].imshow(result.plot(boxes=False, labels=False)[..., ::-1])
axes[1].set_title("YOLO Crack Segmentation")
axes[1].axis("off")

im = axes[2].imshow(depth_map, cmap="plasma")
axes[2].set_title("Predicted Scene Depth")
axes[2].axis("off")
plt.colorbar(im, ax=axes[2], label="Model output (m)")

plt.suptitle("Experimental Concrete Scene-Depth Analysis")
plt.tight_layout()
plt.savefig(output_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", output_path)

In [ ]:
from google.colab import files
files.download("/content/concrete_depth_estimation_result.png")

In [ ]:
from google.colab import files
files.download("/content/concrete_depth_estimation_result.png")

In [ ]:
from google.colab import files
import cv2
import numpy as np
import matplotlib.pyplot as plt

uploaded = files.upload()

for image_path in uploaded:
    image_bgr = cv2.imread(image_path)

    if image_bgr is None:
        print("Could not read:", image_path)
        continue

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    height, width = image_bgr.shape[:2]

    # Crack detection
    result = yolo_model.predict(
        source=image_path,
        conf=0.10,
        imgsz=1280,
        max_det=100,
        retina_masks=True,
        verbose=False
    )[0]

    # Scene-depth prediction
    depth_map = depth_model.infer_image(image_bgr)
    depth_map = cv2.resize(
        depth_map.astype(np.float32),
        (width, height),
        interpolation=cv2.INTER_LINEAR
    )

    # Display results for the SAME image
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(image_rgb)
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    axes[1].imshow(
        result.plot(boxes=False, labels=False)[..., ::-1]
    )
    axes[1].set_title("Crack Segmentation")
    axes[1].axis("off")

    depth_display = axes[2].imshow(depth_map, cmap="plasma")
    axes[2].set_title("Estimated Scene Depth")
    axes[2].axis("off")
    plt.colorbar(depth_display, ax=axes[2], label="Model output (m)")

    plt.suptitle(image_path)
    plt.tight_layout()
    plt.show()

    print("Image:", image_path)
    print("Detected crack instances:",
          0 if result.masks is None else len(result.masks.data))
    print("Depth-map range:",
          round(float(np.nanmin(depth_map)), 3), "to",
          round(float(np.nanmax(depth_map)), 3), "m")

In [ ]:
import cv2
import numpy as np

# Combine the crack masks from the latest image
if result.masks is None:
    raise ValueError("No crack masks detected.")

crack_mask = np.zeros((height, width), dtype=bool)

for mask in result.masks.data.cpu().numpy():
    resized = cv2.resize(
        mask.astype(np.float32),
        (width, height),
        interpolation=cv2.INTER_NEAREST
    )
    crack_mask |= resized > 0.5

# Create a surrounding-concrete region
kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE, (41, 41)
)

expanded = cv2.dilate(
    crack_mask.astype(np.uint8), kernel
).astype(bool)

nearby_mask = expanded & ~crack_mask

valid = np.isfinite(depth_map) & (depth_map > 0)

crack_values = depth_map[crack_mask & valid]
surface_values = depth_map[nearby_mask & valid]

if len(crack_values) == 0 or len(surface_values) == 0:
    print("Not enough valid pixels to compare.")
else:
    crack_distance = float(np.median(crack_values))
    surface_distance = float(np.median(surface_values))
    difference = crack_distance - surface_distance

    print(f"Predicted distance at crack pixels: {crack_distance:.4f} m")
    print(f"Predicted distance at nearby concrete: {surface_distance:.4f} m")
    print(f"Experimental scene-depth difference: {difference:+.4f} m")
    print("Not a validated physical crack-depth measurement.")

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

# ---------- 1. Load the current concrete image ----------
if "image_path" not in globals() or not os.path.exists(image_path):
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("Please upload a concrete crack image.")
    image_path = next(iter(uploaded))

image_bgr = cv2.imread(image_path)
if image_bgr is None:
    raise ValueError("Could not read image.")

image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
h, w = image_bgr.shape[:2]

# ---------- 2. Load YOLO if needed ----------
if "yolo_model" not in globals():
    yolo_model = YOLO(
        "/content/drive/MyDrive/CrackDetection_Backup/best_improved.pt"
    )

result = yolo_model.predict(
    source=image_path,
    conf=0.10,
    imgsz=1280,
    retina_masks=True,
    verbose=False
)[0]

if result.masks is None:
    raise ValueError("No crack masks detected. Try another image.")

# Combine masks into one crack region
crack_mask = np.zeros((h, w), dtype=bool)
for mask in result.masks.data.cpu().numpy():
    mask = cv2.resize(
        mask.astype(np.float32), (w, h),
        interpolation=cv2.INTER_NEAREST
    )
    crack_mask |= mask > 0.5

# ---------- 3. Load or predict scene depth ----------
if "depth_model" not in globals():
    import sys, torch
    sys.path.insert(0, "/content/Depth-Anything-V2/metric_depth")
    from depth_anything_v2.dpt import DepthAnythingV2

    checkpoint_path = (
        "/content/Depth-Anything-V2/metric_depth/checkpoints/"
        "depth_anything_v2_metric_vkitti_vitb.pth"
    )
    assert os.path.exists(checkpoint_path), "Depth checkpoint missing."

    depth_model = DepthAnythingV2(
        encoder="vitb", features=128,
        out_channels=[96, 192, 384, 768], max_depth=80
    )
    depth_model.load_state_dict(
        torch.load(checkpoint_path, map_location="cpu")
    )
    depth_model.eval()

depth_map = depth_model.infer_image(image_bgr)
depth_map = cv2.resize(
    depth_map.astype(np.float32), (w, h),
    interpolation=cv2.INTER_LINEAR
)

# ---------- 4. Extract experimental crack features ----------
crack_pixels = int(crack_mask.sum())
crack_area_percent = 100 * crack_pixels / (h * w)

gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
edges = cv2.Canny(gray, 60, 150)
crack_edge_pixels = int(np.logical_and(edges > 0, crack_mask).sum())

valid_depth = np.isfinite(depth_map) & (depth_map > 0)
crack_depth = depth_map[crack_mask & valid_depth]

if len(crack_depth) == 0:
    raise ValueError("No valid depth values inside the crack mask.")

median_scene_depth = float(np.median(crack_depth))
scene_depth_spread = float(
    np.percentile(crack_depth, 90) -
    np.percentile(crack_depth, 10)
)

# ---------- 5. Display honest prototype output ----------
print("\n===== CONCRETE CRACK ANALYSIS =====")
print("Detected crack instances:", len(result.boxes))
print(f"Predicted crack-mask area: {crack_area_percent:.3f}% of image")
print("Crack-mask pixels:", crack_pixels)
print("Crack-edge pixels:", crack_edge_pixels)
print(f"Median predicted scene depth: {median_scene_depth:.3f} m")
print(f"Scene-depth spread (P90-P10): {scene_depth_spread:.3f} m")

print("\nDEPTH ESTIMATION STATUS")
print("Physical crack penetration depth: NOT ESTIMATED")
print("Reason: no validated crack-depth training labels or calibration.")
print("The scene-depth measurements above are not crack penetration depth.")

# ---------- 6. Visualize all outputs ----------
fig, ax = plt.subplots(1, 3, figsize=(17, 6))

ax[0].imshow(image_rgb)
ax[0].set_title("Original Concrete Image")
ax[0].axis("off")

ax[1].imshow(result.plot(boxes=False, labels=False)[..., ::-1])
ax[1].set_title("YOLO Crack Segmentation")
ax[1].axis("off")

im = ax[2].imshow(depth_map, cmap="plasma")
ax[2].set_title("Estimated Scene Depth")
ax[2].axis("off")
plt.colorbar(im, ax=ax[2], label="Predicted camera distance (m)")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Uses variables from your latest successful analysis:
# image_rgb, result, depth_map, crack_mask

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(image_rgb)
axes[0].set_title("Original Concrete Image")
axes[0].axis("off")

axes[1].imshow(
    result.plot(boxes=False, labels=False)[..., ::-1]
)
axes[1].set_title("YOLO Crack Segmentation")
axes[1].axis("off")

im = axes[2].imshow(depth_map, cmap="plasma")
axes[2].set_title("Estimated Scene Depth")
axes[2].axis("off")
fig.colorbar(im, ax=axes[2], label="Predicted scene depth (m)")

plt.tight_layout()
plt.savefig(
    "/content/concrete_inspection_final.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()

with open("/content/concrete_inspection_summary.txt", "w") as f:
    f.write("CONCRETE CRACK INSPECTION — EXPERIMENTAL PROTOTYPE\n")
    f.write(f"Detected instances: {len(result.boxes)}\n")
    f.write(f"Crack-mask area: {100 * crack_mask.mean():.3f}%\n")
    f.write(
        f"Median predicted scene depth: "
        f"{np.median(depth_map):.3f} m\n"
    )
    f.write(
        "Physical crack penetration depth: NOT ESTIMATED\n"
    )
    f.write(
        "Reason: no validated crack-depth estimator or calibration.\n"
    )

print("Saved visualization and summary.")

In [ ]:
from google.colab import files

files.download("/content/concrete_inspection_final.png")
files.download("/content/concrete_inspection_summary.txt")

In [ ]:
import os

for path in [
    "/content/concrete_inspection_final.png",
    "/content/concrete_inspection_summary.txt",
    "/content/drive/MyDrive/CrackDetection_Backup/best_improved.pt"
]:
    print(
        os.path.basename(path),
        "FOUND" if os.path.exists(path) else "NOT FOUND"
    )

In [ ]:
import os

paths = [
    "/content/drive/MyDrive/CrackDetection_Backup/best_improved.pt",
    "/content/drive/MyDrive/CrackDetection_Backup/concrete_inspection_final.png",
    "/content/drive/MyDrive/CrackDetection_Backup/concrete_inspection_summary.txt",
    "/content/Depth-Anything-V2/metric_depth/checkpoints/depth_anything_v2_metric_vkitti_vitb.pth"
]

for path in paths:
    print(
        "SAVED/FOUND" if os.path.exists(path) else "MISSING",
        os.path.basename(path)
    )

In [ ]:
import os
import shutil

backup = "/content/drive/MyDrive/CrackDetection_Backup"
os.makedirs(backup, exist_ok=True)

wanted = [
    "concrete_inspection_final.png",
    "concrete_inspection_summary.txt"
]

for filename in wanted:
    matches = []

    # Search Colab storage and Google Drive
    for root, dirs, files in os.walk("/content"):
        # Avoid searching large, unnecessary cache folders
        dirs[:] = [
            d for d in dirs
            if d not in ["node_modules", ".cache"]
        ]

        if filename in files:
            path = os.path.join(root, filename)
            if path != os.path.join(backup, filename):
                matches.append(path)

    if matches:
        source = matches[0]
        destination = os.path.join(backup, filename)

        if os.path.abspath(source) != os.path.abspath(destination):
            shutil.copy2(source, destination)

        print("SAVED TO DRIVE:", destination)
    else:
        print("NOT FOUND — NEEDS TO BE RECREATED:", filename)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# 1. Get the crack mask from your YOLO segmentation result
if result.masks is None:
    raise ValueError("No crack masks detected. Try another image.")

masks = result.masks.data.cpu().numpy()
crack_mask = np.any(masks > 0.5, axis=0)

# Resize mask to match the depth map
from PIL import Image
crack_mask = np.array(
    Image.fromarray(crack_mask.astype("uint8") * 255).resize(
        (depth_map.shape[1], depth_map.shape[0]),
        Image.Resampling.NEAREST
    )
) > 0

# 2. Copy the original scene-depth map
scene_depth = np.asarray(depth_map, dtype=np.float32).squeeze()

if scene_depth.shape != crack_mask.shape:
    raise ValueError("Depth map and crack mask dimensions do not match.")

# 3. Create a SIMULATED 3 m contrast for visualization only
# Do not interpret this as actual crack penetration depth.
valid = np.isfinite(scene_depth)
if not np.any(crack_mask & valid):
    raise ValueError("No valid depth pixels overlap the crack mask.")

reference_depth = float(np.median(scene_depth[valid & ~crack_mask]))
if not np.isfinite(reference_depth):
    reference_depth = float(np.median(scene_depth[valid]))

display_depth = scene_depth.copy()
display_depth[crack_mask] = reference_depth + 3.0

# 4. Plot and save the results
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(image_rgb)
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(crack_mask, cmap="gray")
plt.title("YOLO Crack Mask")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(display_depth, cmap="turbo")
plt.colorbar(label="Display depth (m)")
plt.title("Experimental Visualization\nSIMULATED +3 m contrast")
plt.axis("off")

plt.tight_layout()

out = "/content/drive/MyDrive/CrackDetection_Backup/experimental_3m_visualization.png"
plt.savefig(out, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", out)
print("Simulated display contrast: 3.0 m")
print("WARNING: This is not a measured physical crack depth.")

In [ ]:
from ultralytics import YOLO
from google.colab import drive, files
from PIL import Image
import numpy as np
import os

# Mount Google Drive if needed
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")

# Load your trained model
model_path = "/content/drive/MyDrive/CrackDetection_Backup/best_improved.pt"
yolo_model = YOLO(model_path)
print("YOLO model loaded successfully!")

# Upload the concrete image
uploaded = files.upload()
image_path = next(iter(uploaded))

# Load image in RGB
image_rgb = np.array(Image.open(image_path).convert("RGB"))

# Run crack segmentation
results = yolo_model.predict(
    source=image_rgb,
    conf=0.10,
    imgsz=1280,
    retina_masks=True,
    verbose=False
)

result = results[0]
print("Detection completed.")
print("Crack masks detected:", 0 if result.masks is None else len(result.masks.data))

Image.fromarray(result.plot()[..., ::-1]).show()

In [ ]:
%pip install -q ultralytics

In [ ]:
from ultralytics import YOLO
from google.colab import drive, files
from PIL import Image
import numpy as np
import os

drive.mount('/content/drive')

model_path = "/content/drive/MyDrive/CrackDetection_Backup/best_improved.pt"
assert os.path.exists(model_path), f"Model not found: {model_path}"

yolo_model = YOLO(model_path)
print("YOLO model loaded successfully!")

uploaded = files.upload()
image_path = next(iter(uploaded))

image_rgb = np.array(Image.open(image_path).convert("RGB"))

results = yolo_model.predict(
    source=image_rgb,
    conf=0.10,
    imgsz=1280,
    retina_masks=True,
    verbose=False
)

result = results[0]

print("Detection completed.")
print("Crack masks detected:", 0 if result.masks is None else len(result.masks.data))

display(Image.fromarray(result.plot()[..., ::-1]))

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

# Run prediction with duplicate suppression
results = yolo_model.predict(
    source=image_rgb,
    conf=0.35,
    iou=0.45,
    imgsz=1280,
    retina_masks=True,
    max_det=30,
    verbose=False
)

result = results[0]

# Start with the original image
clean_image = image_rgb.copy()

if result.masks is not None:
    masks = result.masks.data.cpu().numpy()

    # Combine all predicted masks into one
    combined_mask = np.any(masks > 0.5, axis=0).astype(np.uint8)

    # Resize mask to original image dimensions
    combined_mask = cv2.resize(
        combined_mask,
        (clean_image.shape[1], clean_image.shape[0]),
        interpolation=cv2.INTER_NEAREST
    )

    # Draw only the crack overlay, without boxes or text
    overlay = clean_image.copy()
    overlay[combined_mask > 0] = [255, 50, 50]

    clean_image = cv2.addWeighted(clean_image, 0.65, overlay, 0.35, 0)

plt.figure(figsize=(10, 7))
plt.imshow(clean_image)
plt.title(f"Concrete Crack Segmentation | {len(result.masks.data) if result.masks is not None else 0} detections")
plt.axis("off")
plt.show()

In [ ]:

# COMPLETE CONCRETE CRACK + DEPTH INSPECTION
# Runs on CPU or GPU. No retraining required.

%pip -q install ultralytics timm

import os
import sys
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from google.colab import drive, files
from ultralytics import YOLO
from IPython.display import display

# 1. Connect Google Drive
drive.mount("/content/drive")

backup = "/content/drive/MyDrive/CrackDetection_Backup"
yolo_path = os.path.join(backup, "best_improved.pt")

assert os.path.exists(yolo_path), f"YOLO checkpoint missing: {yolo_path}"

# 2. Load crack segmentation model
yolo_model = YOLO(yolo_path)
print("YOLO crack model loaded.")

# 3. Load Depth Anything V2 if not already loaded
checkpoint = (
    "/content/Depth-Anything-V2/metric_depth/checkpoints/"
    "depth_anything_v2_metric_vkitti_vitb.pth"
)
repo_path = "/content/Depth-Anything-V2/metric_depth"

if not os.path.exists(checkpoint):
    raise FileNotFoundError(
        "Depth checkpoint not found. Restore/download it at:\n" + checkpoint
    )

if "depth_model" not in globals():
    sys.path.insert(0, repo_path)
    from depth_anything_v2.dpt import DepthAnythingV2

    depth_model = DepthAnythingV2(
        encoder="vitb",
        features=128,
        out_channels=[96, 192, 384, 768],
        max_depth=80
    )
    state = torch.load(checkpoint, map_location="cpu")
    depth_model.load_state_dict(state)
    depth_model = depth_model.to(
        "cuda" if torch.cuda.is_available() else "cpu"
    ).eval()

print("Depth model ready on", next(depth_model.parameters()).device)

# 4. Upload the concrete image
uploaded = files.upload()
if not uploaded:
    raise ValueError("No image uploaded.")

image_path = next(iter(uploaded))
image_rgb = np.array(Image.open(image_path).convert("RGB"))
image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
H, W = image_rgb.shape[:2]

# 5. Detect and segment cracks
predictions = yolo_model.predict(
    source=image_bgr,
    conf=0.10,
    iou=0.45,
    imgsz=1280,
    retina_masks=True,
    max_det=100,
    verbose=False
)
result = predictions[0]

crack_mask = np.zeros((H, W), dtype=bool)

if result.masks is not None:
    for mask in result.masks.data.cpu().numpy():
        resized = cv2.resize(
            mask.astype(np.float32),
            (W, H),
            interpolation=cv2.INTER_LINEAR
        )
        crack_mask |= resized > 0.5

# Draw a clean overlay without boxes or text
overlay = image_rgb.copy()
overlay[crack_mask] = (
    0.45 * image_rgb[crack_mask] + 0.55 * np.array([255, 40, 40])
).astype(np.uint8)

# 6. Estimate scene depth using the SAME image
depth_map = depth_model.infer_image(image_bgr)
depth_map = np.asarray(depth_map, dtype=np.float32).squeeze()

if depth_map.shape != (H, W):
    depth_map = cv2.resize(depth_map, (W, H))

valid = np.isfinite(depth_map) & (depth_map > 0)
crack_pixels = crack_mask & valid

# 7. Calculate scene-depth statistics, not physical crack penetration
if crack_pixels.any():
    crack_scene_depth = float(np.median(depth_map[crack_pixels]))

    # Compare with pixels near the crack, excluding the mask itself
    kernel = np.ones((21, 21), np.uint8)
    expanded = cv2.dilate(crack_mask.astype(np.uint8), kernel) > 0
    nearby_surface = expanded & ~crack_mask & valid

    if nearby_surface.any():
        surface_scene_depth = float(np.median(depth_map[nearby_surface]))
        scene_difference = crack_scene_depth - surface_scene_depth
    else:
        surface_scene_depth = None
        scene_difference = None
else:
    crack_scene_depth = None
    surface_scene_depth = None
    scene_difference = None

# 8. Display the complete inspection results
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(image_rgb)
axes[0].set_title("1. Original Image")

axes[1].imshow(overlay)
axes[1].set_title(
    f"2. Crack Segmentation ({int(result.masks.data.shape[0]) if result.masks is not None else 0} masks)"
)

im = axes[2].imshow(
    np.where(valid, depth_map, np.nan),
    cmap="turbo"
)
axes[2].set_title("3. Estimated Scene-Depth Map")
plt.colorbar(im, ax=axes[2], label="Model output (m; approximate)")

for ax in axes:
    ax.axis("off")

plt.tight_layout()

output_png = os.path.join(backup, "concrete_inspection_final.png")
plt.savefig(output_png, dpi=180, bbox_inches="tight")
plt.show()

# 9. Save numerical report
report = [
    "CONCRETE CRACK INSPECTION",
    f"Image: {image_path}",
    f"Crack masks detected: {0 if result.masks is None else len(result.masks.data)}",
    f"Crack mask area: {100 * crack_mask.mean():.3f}%",
    f"Depth map median: {np.median(depth_map[valid]):.3f} m (model estimate)" if valid.any() else "Depth map: no valid values",
    "",
    "Crack-region median scene depth: "
    + (f"{crack_scene_depth:.3f} m" if crack_scene_depth is not None else "Unavailable"),
    "Nearby-surface median scene depth: "
    + (f"{surface_scene_depth:.3f} m" if surface_scene_depth is not None else "Unavailable"),
    "Difference in estimated scene distance: "
    + (f"{scene_difference:+.3f} m" if scene_difference is not None else "Unavailable"),
    "",
    "IMPORTANT: Scene-depth difference is NOT physical crack penetration depth.",
    "This model is not validated for measuring crack depth in concrete."
]

report_path = os.path.join(backup, "concrete_inspection_summary.txt")
with open(report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(report))

print("\n".join(report))
print("\nSaved image:", output_png)
print("Saved report:", report_path)


In [ ]:

import os
import subprocess

# Restore official model code
repo = "/content/Depth-Anything-V2"

if not os.path.isdir(repo):
    subprocess.run([
        "git", "clone",
        "https://github.com/DepthAnything/Depth-Anything-V2.git",
        repo
    ], check=True)

# Install model dependencies
%pip -q install timm

# Download checkpoint into the expected location
checkpoint_dir = f"{repo}/metric_depth/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

checkpoint = f"{checkpoint_dir}/depth_anything_v2_metric_vkitti_vitb.pth"

if not os.path.exists(checkpoint):
    subprocess.run([
        "wget", "-O", checkpoint,
        "https://huggingface.co/depth-anything/Depth-Anything-V2-Metric-VKITTI-Base/resolve/main/depth_anything_v2_metric_vkitti_vitb.pth?download=true"
    ], check=True)

print("Checkpoint exists:", os.path.exists(checkpoint))
print("Checkpoint size (GB):", round(os.path.getsize(checkpoint) / 1e9, 3))

# Back it up to Drive
import shutil
drive_checkpoint = "/content/drive/MyDrive/CrackDetection_Backup/depth_anything_v2_metric_vkitti_vitb.pth"
shutil.copy2(checkpoint, drive_checkpoint)
print("Depth checkpoint backed up to:", drive_checkpoint)


In [ ]:

import os, sys, cv2, torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files, drive
from ultralytics import YOLO

# Drive is already mounted; reconnect only if necessary
drive.mount("/content/drive")

backup = "/content/drive/MyDrive/CrackDetection_Backup"
repo = "/content/Depth-Anything-V2/metric_depth"
checkpoint = os.path.join(
    backup, "depth_anything_v2_metric_vkitti_vitb.pth"
)

# Load YOLO if not already loaded
if "yolo_model" not in globals():
    yolo_model = YOLO(os.path.join(backup, "best_improved.pt"))

# Load Depth Anything V2
sys.path.insert(0, repo)
from depth_anything_v2.dpt import DepthAnythingV2

device = "cuda" if torch.cuda.is_available() else "cpu"

depth_model = DepthAnythingV2(
    encoder="vitb",
    features=128,
    out_channels=[96, 192, 384, 768],
    max_depth=80
)
depth_model.load_state_dict(
    torch.load(checkpoint, map_location="cpu")
)
depth_model = depth_model.to(device).eval()
print("Both models loaded on:", device)

# Upload the concrete image
uploaded = files.upload()
if not uploaded:
    raise ValueError("Please upload a concrete crack image.")

image_path = next(iter(uploaded))
image_rgb = np.array(Image.open(image_path).convert("RGB"))
image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
H, W = image_rgb.shape[:2]

# Crack segmentation
result = yolo_model.predict(
    source=image_bgr,
    conf=0.10,
    iou=0.45,
    imgsz=1280,
    retina_masks=True,
    verbose=False
)[0]

crack_mask = np.zeros((H, W), dtype=bool)
if result.masks is not None:
    for m in result.masks.data.cpu().numpy():
        m = cv2.resize(m, (W, H), interpolation=cv2.INTER_LINEAR)
        crack_mask |= m > 0.5

overlay = image_rgb.copy()
overlay[crack_mask] = (
    0.45 * overlay[crack_mask] +
    0.55 * np.array([255, 40, 40])
).astype(np.uint8)

# Scene-depth estimation on the same image
depth_map = np.asarray(
    depth_model.infer_image(image_bgr), dtype=np.float32
).squeeze()

if depth_map.shape != (H, W):
    depth_map = cv2.resize(depth_map, (W, H))

valid = np.isfinite(depth_map) & (depth_map > 0)
crack_pixels = crack_mask & valid
surface_pixels = valid & ~crack_mask

# Display all three results
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(image_rgb)
ax[0].set_title("Original Image")

ax[1].imshow(overlay)
ax[1].set_title("Crack Segmentation")

im = ax[2].imshow(np.where(valid, depth_map, np.nan), cmap="turbo")
ax[2].set_title("Estimated Scene Depth")
plt.colorbar(im, ax=ax[2], label="Estimated distance (m)")

for a in ax:
    a.axis("off")
plt.tight_layout()

out = os.path.join(backup, "concrete_inspection_final.png")
plt.savefig(out, dpi=180, bbox_inches="tight")
plt.show()

print("\n--- INSPECTION SUMMARY ---")
print("Detected crack masks:",
      0 if result.masks is None else len(result.masks.data))
print(f"Crack mask area: {100 * crack_mask.mean():.2f}%")

if crack_pixels.any():
    crack_depth = float(np.median(depth_map[crack_pixels]))
    print(f"Median scene distance at crack pixels: {crack_depth:.3f} m")
else:
    crack_depth = None
    print("No valid scene-depth pixels overlap the crack mask.")

if surface_pixels.any():
    print(f"Median scene distance elsewhere: "
          f"{np.median(depth_map[surface_pixels]):.3f} m")

print("\nSaved visualization:", out)
print("IMPORTANT: These are scene-distance estimates, NOT physical")
print("crack penetration depth. A 3 m crack-depth claim is not validated.")


In [ ]:

import torch
import os

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(
        torch.cuda.get_device_properties(0).total_memory / 1e9, 2
    ))
else:
    print("Running on CPU — large VLM inference may be very slow.")

print("Google Drive mounted:",
      os.path.isdir("/content/drive/MyDrive"))

print("YOLO checkpoint exists:",
      os.path.isfile(
          "/content/drive/MyDrive/CrackDetection_Backup/best_improved.pt"
      ))

print("Depth checkpoint exists:",
      os.path.isfile(
          "/content/drive/MyDrive/CrackDetection_Backup/"
          "depth_anything_v2_metric_vkitti_vitb.pth"
      ))


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import os

backup = "/content/drive/MyDrive/CrackDetection_Backup"

print("\nBackup folder exists:", os.path.isdir(backup))

if os.path.isdir(backup):
    print("\nFiles in your backup folder:")
    for filename in sorted(os.listdir(backup)):
        print(" -", filename)
else:
    print("\nFolder not found. We will locate the saved files next.")


In [ ]:

import psutil
import torch

ram = psutil.virtual_memory()

print(f"Total RAM: {ram.total / (1024**3):.2f} GB")
print(f"Available RAM: {ram.available / (1024**3):.2f} GB")
print(f"CPU cores: {psutil.cpu_count(logical=True)}")
print(f"GPU available: {torch.cuda.is_available()}")


In [ ]:

%pip -q install llama-cpp-python huggingface_hub
print("Installation step finished.")


In [ ]:

from huggingface_hub import hf_hub_download
import os

model_dir = "/content/local_vlm"
os.makedirs(model_dir, exist_ok=True)

repo_id = "ggml-org/SmolVLM-500M-Instruct-GGUF"

print("Downloading quantized vision-language model...")
model_path = hf_hub_download(
    repo_id=repo_id,
    filename="SmolVLM-500M-Instruct-Q4_K_M.gguf",
    local_dir=model_dir
)

print("\nModel downloaded:")
print(model_path)
print("Size (GB):", round(os.path.getsize(model_path) / 1e9, 2))


In [ ]:

from huggingface_hub import hf_hub_download
import os

model_dir = "/content/local_vlm"
os.makedirs(model_dir, exist_ok=True)

repo_id = "ggml-org/SmolVLM-500M-Instruct-GGUF"

files_to_download = [
    "SmolVLM-500M-Instruct-Q8_0.gguf",
    "mmproj-SmolVLM-500M-Instruct-Q8_0.gguf"
]

for filename in files_to_download:
    print(f"\nDownloading {filename}...")
    path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        local_dir=model_dir
    )
    print("Saved:", path)
    print("Size (MB):", round(os.path.getsize(path) / 1e6, 1))

print("\nBoth VLM files downloaded.")


In [ ]:

import llama_cpp
import inspect
from llama_cpp import Llama

print("llama-cpp-python version:", llama_cpp.__version__)
print("Llama class available:", Llama is not None)

try:
    print("Llama constructor:", inspect.signature(Llama))
except Exception as e:
    print("Could not inspect constructor:", e)

print("\nModel file exists:", __import__("os").path.isfile(
    "/content/local_vlm/SmolVLM-500M-Instruct-Q8_0.gguf"
))
print("Vision projector exists:", __import__("os").path.isfile(
    "/content/local_vlm/mmproj-SmolVLM-500M-Instruct-Q8_0.gguf"
))


In [ ]:

%pip -q install llama-cpp-python[server]

from llama_cpp import Llama
import os

model_path = "/content/local_vlm/SmolVLM-500M-Instruct-Q8_0.gguf"
mmproj_path = "/content/local_vlm/mmproj-SmolVLM-500M-Instruct-Q8_0.gguf"

assert os.path.exists(model_path), "VLM model missing"
assert os.path.exists(mmproj_path), "Vision projector missing"

print("Model and projector found.")
print("Next we will launch the multimodal handler for image testing.")


In [ ]:

import subprocess
import shutil

print("llama-server path:", shutil.which("llama-server"))

result = subprocess.run(
    ["llama-server", "--help"],
    capture_output=True,
    text=True
)

print((result.stdout or result.stderr)[:5000])


In [ ]:

import subprocess
import os
import shutil

install = subprocess.run(
    ["bash", "-lc",
     "curl -LsSf https://llama.app/install.sh | sh"],
    capture_output=True,
    text=True,
    timeout=300
)

print(install.stdout[-3000:])
if install.stderr:
    print(install.stderr[-1500:])

# Refresh PATH and look for the executable
for directory in [
    os.path.expanduser("~/.local/bin"),
    "/usr/local/bin",
    "/usr/bin"
]:
    if os.path.isdir(directory) and directory not in os.environ["PATH"]:
        os.environ["PATH"] += os.pathsep + directory

server = shutil.which("llama-server")
print("\nllama-server found:", server)

if server:
    check = subprocess.run(
        [server, "--help"],
        capture_output=True,
        text=True,
        timeout=30
    )
    print((check.stdout or check.stderr)[:2000])


In [ ]:

import os
import subprocess

llama = os.path.expanduser("~/.llama-app/llama")

print("Executable exists:", os.path.isfile(llama))

if os.path.isfile(llama):
    result = subprocess.run(
        [llama, "--help"],
        capture_output=True,
        text=True,
        timeout=30
    )
    print((result.stdout or result.stderr)[:3000])
else:
    print("Checking installer directory...")
    for root, dirs, files in os.walk(os.path.expanduser("~/.llama-app")):
        for name in files:
            if "llama" in name.lower() or "server" in name.lower():
                print(os.path.join(root, name))


In [ ]:

import subprocess

llama = "/root/.llama-app/llama"

for command in ["serve --help", "help all"]:
    print(f"\n===== llama {command} =====")
    result = subprocess.run(
        [llama] + command.split(),
        capture_output=True,
        text=True,
        timeout=30
    )
    print((result.stdout or result.stderr)[:5000])


In [ ]:
import torch
import os

print("GPU available:", torch.cuda.is_available())
print("VLM model exists:",
      os.path.exists("/content/local_vlm/SmolVLM-500M-Instruct-Q8_0.gguf"))
print("VLM projector exists:",
      os.path.exists("/content/local_vlm/mmproj-SmolVLM-500M-Instruct-Q8_0.gguf"))

In [ ]:

import subprocess

llama = "/root/.llama-app/llama"

result = subprocess.run(
    [llama, "serve", "--help"],
    capture_output=True,
    text=True,
    timeout=30
)

print((result.stdout or result.stderr)[:6000])


In [ ]:

import subprocess

llama = "/root/.llama-app/llama"

result = subprocess.run(
    [llama, "serve", "--help"],
    capture_output=True,
    text=True,
    timeout=30
)

help_text = result.stdout + "\n" + result.stderr

keywords = [
    "model", "mmproj", "vision", "image", "port",
    "host", "alias", "threads", "ctx-size"
]

for line in help_text.splitlines():
    if any(word in line.lower() for word in keywords):
        print(line)


In [ ]:

import subprocess, os, time

llama = "/root/.llama-app/llama"
model = "/content/local_vlm/SmolVLM-500M-Instruct-Q8_0.gguf"
projector = "/content/local_vlm/mmproj-SmolVLM-500M-Instruct-Q8_0.gguf"
log_path = "/content/smolvlm_server.log"

cmd = [
    llama, "serve",
    "--model", model,
    "--mmproj", projector,
    "--host", "127.0.0.1",
    "--port", "9931",
    "--threads", "2",
    "--ctx-size", "2048"
]

log = open(log_path, "w")
server_process = subprocess.Popen(
    cmd,
    stdout=log,
    stderr=subprocess.STDOUT
)

time.sleep(15)

print("Server process running:", server_process.poll() is None)
print("\nRecent server log:")
if os.path.exists(log_path):
    with open(log_path, "r", errors="replace") as f:
        print(f.read()[-5000:])


In [ ]:

import requests
import base64
import io
from PIL import Image
from google.colab import files

SERVER = "http://127.0.0.1:9931"

# 1. Check server health
health = requests.get(f"{SERVER}/health", timeout=10)
print("Server health:", health.status_code, health.text[:200])

# 2. Upload a concrete crack image
uploaded = files.upload()
if not uploaded:
    raise ValueError("Please upload a concrete crack image.")

filename = next(iter(uploaded))
image = Image.open(io.BytesIO(uploaded[filename])).convert("RGB")

# Resize for faster CPU inference
image.thumbnail((640, 640))
buffer = io.BytesIO()
image.save(buffer, format="JPEG", quality=85)

image_b64 = base64.b64encode(buffer.getvalue()).decode("utf-8")

# 3. Ask the VLM to analyze the image
payload = {
    "model": "SmolVLM",
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "Analyze this concrete surface image for visible cracks. "
                        "Describe their visible shape, location, and pattern. "
                        "Distinguish direct visual observations from uncertainty. "
                        "Do not claim to measure crack depth or structural safety."
                    )
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "data:image/jpeg;base64," + image_b64
                    }
                }
            ]
        }
    ],
    "temperature": 0.1,
    "max_tokens": 180,
    "stream": False
}

response = requests.post(
    f"{SERVER}/v1/chat/completions",
    json=payload,
    timeout=300
)

print("\nHTTP status:", response.status_code)
response.raise_for_status()

data = response.json()
print("\n===== SMOLVLM IMAGE ANALYSIS =====\n")
print(data["choices"][0]["message"]["content"])


In [ ]:

%pip -q install sentence-transformers faiss-cpu

print("RAG libraries installed.")


In [ ]:

# STEP 16: Create a starter concrete crack inspection knowledge base

knowledge_base = [
    {
        "id": "KB001",
        "topic": "Visible crack patterns",
        "text": (
            "Concrete cracks can appear as straight, curved, branching, "
            "or irregular lines. Image inspection can describe visible "
            "shape, orientation, distribution, and apparent continuity. "
            "Appearance alone does not establish the cause of a crack."
        )
    },
    {
        "id": "KB002",
        "topic": "Crack detection and segmentation",
        "text": (
            "Object detection identifies regions containing suspected "
            "cracks. Segmentation identifies crack-shaped pixels. Model "
            "predictions may include false positives, missed cracks, "
            "and incomplete boundaries, so results should be reviewed."
        )
    },
    {
        "id": "KB003",
        "topic": "Crack width measurement",
        "text": (
            "Measurements in pixels describe image-space dimensions. "
            "Converting pixel width into millimetres requires suitable "
            "scale calibration and image geometry. Camera angle, "
            "perspective, resolution, and segmentation errors can affect "
            "measurement accuracy."
        )
    },
    {
        "id": "KB004",
        "topic": "Depth estimation limitations",
        "text": (
            "Monocular depth estimation predicts scene depth from a "
            "single image. It does not directly measure how far a crack "
            "penetrates into concrete. Physical crack depth requires "
            "appropriate ground-truth measurements or a validated "
            "measurement method."
        )
    },
    {
        "id": "KB005",
        "topic": "Visual language model limitations",
        "text": (
            "A vision-language model can summarize visible image features, "
            "but may generate unsupported details. Separate observable "
            "features from hypotheses. Do not infer moisture, crack causes, "
            "structural safety, or physical depth unless supported by "
            "reliable evidence."
        )
    },
    {
        "id": "KB006",
        "topic": "Inspection documentation",
        "text": (
            "An inspection record should include the image identifier, "
            "inspection date if available, detected crack regions, "
            "model confidence when available, image-space measurements, "
            "measurement units, calibration details, and limitations. "
            "Keep estimated values distinct from verified measurements."
        )
    },
    {
        "id": "KB007",
        "topic": "Human verification and escalation",
        "text": (
            "AI inspection results are decision-support information, "
            "not a certified structural assessment. Uncertain results "
            "or concerns about structural integrity should be reviewed "
            "by a qualified civil or structural engineering professional."
        )
    }
]

print("Knowledge base created:", len(knowledge_base), "documents")
for doc in knowledge_base:
    print(f'{doc["id"]}: {doc["topic"]}')


In [ ]:

# STEP 17: Create embeddings and a FAISS search index

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load a lightweight text embedding model
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Prepare searchable text
knowledge_texts = [
    f'{doc["topic"]}: {doc["text"]}'
    for doc in knowledge_base
]

# Convert documents into embeddings
embeddings = embedding_model.encode(
    knowledge_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

# Create a FAISS cosine-similarity index
embedding_dimension = embeddings.shape[1]
rag_index = faiss.IndexFlatIP(embedding_dimension)
rag_index.add(embeddings)

print("Embedding shape:", embeddings.shape)
print("Documents indexed:", rag_index.ntotal)
print("RAG search index created successfully.")


In [ ]:

# STEP 18: Test RAG retrieval

def rag_search(question, top_k=3):
    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = rag_index.search(query_embedding, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0:
            doc = knowledge_base[int(idx)]
            results.append({
                "id": doc["id"],
                "topic": doc["topic"],
                "score": round(float(score), 3),
                "text": doc["text"]
            })

    return results


question = "Can monocular depth estimation measure the physical depth of a concrete crack?"

results = rag_search(question)

print("Question:", question)
print("\nRetrieved knowledge:")
for item in results:
    print(f'\n[{item["id"]}] {item["topic"]} | Score: {item["score"]}')
    print(item["text"])


In [ ]:

# STEP 19: Connect SmolVLM + RAG

import requests
import base64
import io
from PIL import Image
from google.colab import files

SERVER = "http://127.0.0.1:9931"

# Check that the local VLM server is available
health = requests.get(f"{SERVER}/health", timeout=10)
print("VLM server status:", health.status_code)

# Upload a concrete crack image
uploaded = files.upload()
filename = next(iter(uploaded))
image = Image.open(io.BytesIO(uploaded[filename])).convert("RGB")
image.thumbnail((640, 640))

buffer = io.BytesIO()
image.save(buffer, format="JPEG", quality=85)
image_b64 = base64.b64encode(buffer.getvalue()).decode("utf-8")

# Ask VLM for observable features only
payload = {
    "model": "SmolVLM",
    "messages": [{
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": (
                    "Inspect this image for visible concrete cracks. "
                    "Describe only observable features such as location, "
                    "shape, branching and apparent continuity. "
                    "Do not infer crack depth, moisture, cause, or safety. "
                    "If uncertain, say so."
                )
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": "data:image/jpeg;base64," + image_b64
                }
            }
        ]
    }],
    "temperature": 0.1,
    "max_tokens": 180,
    "stream": False
}

response = requests.post(
    f"{SERVER}/v1/chat/completions",
    json=payload,
    timeout=300
)
response.raise_for_status()

vlm_observations = response.json()["choices"][0]["message"]["content"]

# Use the VLM observations to retrieve relevant knowledge
retrieved = rag_search(vlm_observations, top_k=3)

print("\n========== VLM OBSERVATIONS ==========")
print(vlm_observations)

print("\n========== RAG KNOWLEDGE ==========")
for item in retrieved:
    print(f'\n[{item["id"]}] {item["topic"]}')
    print(item["text"])

print("\n========== COMBINED INSPECTION SUMMARY ==========")
print("Image:", filename)
print("\nVisual observations:")
print(vlm_observations)

print("\nRelevant inspection guidance:")
for item in retrieved:
    print(f'- {item["topic"]} ({item["id"]})')

print(
    "\nLimitation: This summary does not establish physical crack depth, "
    "root cause, or structural safety."
)


In [ ]:

# STEP 20: Generate a RAG-grounded inspection summary

# Reuse the VLM observations and retrieved documents from Step 19
context = "\n\n".join(
    f'[{item["id"]}] {item["topic"]}: {item["text"]}'
    for item in retrieved
)

final_prompt = f"""
You are assisting with concrete surface inspection.

VLM observations from the image:
{vlm_observations}

Retrieved knowledge-base guidance:
{context}

Write a concise inspection summary with these sections:

1. Visible observations
2. Relevant inspection guidance
3. Limitations and next steps

Rules:
- Separate image observations from general guidance.
- Do not claim that a visible line is definitely a crack unless verified.
- Do not invent measurements, causes, moisture, or physical crack depth.
- Monocular scene depth is not physical crack penetration depth.
- Do not claim structural safety or danger from this image alone.
- Use only the supplied observations and retrieved guidance.
"""

final_payload = {
    "model": "SmolVLM",
    "messages": [
        {
            "role": "user",
            "content": final_prompt
        }
    ],
    "temperature": 0.1,
    "max_tokens": 300,
    "stream": False
}

final_response = requests.post(
    f"{SERVER}/v1/chat/completions",
    json=final_payload,
    timeout=300
)
final_response.raise_for_status()

inspection_summary = (
    final_response.json()["choices"][0]["message"]["content"]
)

print("========== FINAL INSPECTION SUMMARY ==========\n")
print(inspection_summary)


In [ ]:

# STEP 21: Create a reliable structured inspection report

# Remove duplicate retrieved documents
unique_guidance = []
seen_ids = set()

for item in retrieved:
    if item["id"] not in seen_ids:
        unique_guidance.append(item)
        seen_ids.add(item["id"])

# Build a structured report using actual pipeline outputs
report = {
    "image": filename,
    "visual_observations": vlm_observations.strip(),
    "retrieved_knowledge": [
        {
            "source_id": item["id"],
            "topic": item["topic"],
            "guidance": item["text"]
        }
        for item in unique_guidance
    ],
    "limitations": [
        "Visual observations alone do not confirm that a line is a crack.",
        "Scene-depth estimates do not directly measure physical crack penetration.",
        "Pixel measurements require calibration to estimate physical dimensions.",
        "Structural safety and crack causes cannot be established from this summary alone."
    ]
}

import json

print("========== STRUCTURED INSPECTION REPORT ==========")
print(json.dumps(report, indent=2, ensure_ascii=False))

# Save the report for later use
report_path = "/content/concrete_inspection_rag_report.json"

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("\nReport saved to:", report_path)
print("Unique knowledge documents:", len(unique_guidance))


In [ ]:

# STEP 22: Check the existing project components

import os
import requests

checks = {
    "YOLO checkpoint": (
        "/content/drive/MyDrive/CrackDetection_Backup/best_improved.pt"
    ),
    "Depth checkpoint": (
        "/content/drive/MyDrive/CrackDetection_Backup/"
        "depth_anything_v2_metric_vkitti_vitb.pth"
    ),
    "RAG report": "/content/concrete_inspection_rag_report.json",
}

for name, path in checks.items():
    print(f"{'OK' if os.path.exists(path) else 'MISSING'} - {name}")

try:
    r = requests.get("http://127.0.0.1:9931/health", timeout=5)
    print(f"{'OK' if r.ok else 'CHECK'} - VLM server (HTTP {r.status_code})")
except Exception as e:
    print("CHECK - VLM server:", str(e)[:150])

for name, variable in [
    ("RAG index", "rag_index"),
    ("Embedding model", "embedding_model"),
    ("Knowledge base", "knowledge_base"),
    ("VLM observations", "vlm_observations"),
    ("Uploaded image filename", "filename"),
]:
    exists = variable in globals()
    print(f"{'OK' if exists else 'MISSING'} - {name}")

print("\nCheck complete.")


In [ ]:

# STEP 23: Run YOLO segmentation and scene-depth estimation together

import os
import sys
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

# 1. Paths
backup = "/content/drive/MyDrive/CrackDetection_Backup"
yolo_path = os.path.join(backup, "best_improved.pt")
depth_path = os.path.join(
    backup, "depth_anything_v2_metric_vkitti_vitb.pth"
)
output_dir = "/content/concrete_unified_outputs"
os.makedirs(output_dir, exist_ok=True)

# Use the previously uploaded image if it still exists
image_path = filename if "filename" in globals() else ""
if not image_path or not os.path.isfile(image_path):
    image_path = os.path.join(backup, "concrete_inspection_final.png")

if not os.path.isfile(image_path):
    raise FileNotFoundError(
        "Input image not found. Please upload a concrete crack image "
        "and set image_path to its full path."
    )

print("Input image:", image_path)

# 2. Load YOLO and detect/segment cracks
yolo_model = YOLO(yolo_path)
yolo_results = yolo_model.predict(
    source=image_path,
    conf=0.25,
    verbose=False
)
result = yolo_results[0]

# Save YOLO's annotated image
annotated = result.plot()
yolo_output_path = os.path.join(output_dir, "yolo_segmentation.jpg")
cv2.imwrite(yolo_output_path, annotated)

num_detections = len(result.boxes) if result.boxes is not None else 0
num_masks = (
    len(result.masks.data)
    if result.masks is not None
    else 0
)

print("YOLO detections:", num_detections)
print("YOLO segmentation masks:", num_masks)
print("YOLO image saved:", yolo_output_path)

# 3. Load Depth Anything V2 if it is not already in memory
if "depth_model" not in globals():
    sys.path.insert(0, "/content/Depth-Anything-V2/metric_depth")
    from depth_anything_v2.dpt import DepthAnythingV2

    depth_model = DepthAnythingV2(
        encoder="vitb",
        features=128,
        out_channels=[96, 192, 384, 768],
        max_depth=80
    )
    depth_model.load_state_dict(
        torch.load(depth_path, map_location="cpu")
    )
    depth_model = depth_model.to("cpu").eval()

# Run scene-depth estimation on the same image
raw_image = cv2.imread(image_path)
if raw_image is None:
    raise ValueError("OpenCV could not read the input image.")

with torch.inference_mode():
    depth_map = depth_model.infer_image(raw_image)

depth_map = np.asarray(depth_map, dtype=np.float32)
depth_output_path = os.path.join(output_dir, "scene_depth.npy")
np.save(depth_output_path, depth_map)

print("Depth map shape:", depth_map.shape)
print("Scene-depth map saved:", depth_output_path)

# 4. Visualize the image, YOLO result and depth map
original = cv2.cvtColor(raw_image, cv2.COLOR_BGR2RGB)
depth_resized = cv2.resize(
    depth_map,
    (original.shape[1], original.shape[0])
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(original)
axes[0].set_title("Original image")
axes[1].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
axes[1].set_title("YOLO crack predictions")
im = axes[2].imshow(depth_resized, cmap="viridis")
axes[2].set_title("Estimated scene depth")
plt.colorbar(im, ax=axes[2], fraction=0.046, label="Model output")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

print("\nCode 2 complete.")
print("Note: scene depth is not physical crack penetration depth.")


In [ ]:

%pip install -q ultralytics

import ultralytics
print("Ultralytics version:", ultralytics.__version__)

from ultralytics import YOLO
print("YOLO import successful!")


In [ ]:

# STEP 23: Run YOLO crack segmentation + scene-depth estimation

import os
import sys
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from ultralytics import YOLO

# 1. Set paths
backup = "/content/drive/MyDrive/CrackDetection_Backup"
yolo_path = os.path.join(backup, "best_improved.pt")
depth_path = os.path.join(
    backup, "depth_anything_v2_metric_vkitti_vitb.pth"
)
output_dir = "/content/concrete_unified_outputs"
os.makedirs(output_dir, exist_ok=True)

# 2. Find the input image
image_path = globals().get("filename", "")

if not image_path or not os.path.isfile(image_path):
    image_path = os.path.join(backup, "concrete_inspection_final.png")

if not os.path.isfile(image_path):
    from google.colab import files
    uploaded = files.upload()
    image_path = next(iter(uploaded))

print("Input image:", image_path)

# 3. Load the trained YOLO model
yolo_model = YOLO(yolo_path)

# Predict crack regions
yolo_results = yolo_model.predict(
    source=image_path,
    conf=0.25,
    verbose=False
)
result = yolo_results[0]

num_detections = len(result.boxes) if result.boxes is not None else 0
num_masks = len(result.masks.data) if result.masks is not None else 0

print("YOLO detections:", num_detections)
print("YOLO segmentation masks:", num_masks)

# Save annotated YOLO output
annotated = result.plot()
yolo_output_path = os.path.join(output_dir, "yolo_segmentation.jpg")
cv2.imwrite(yolo_output_path, annotated)
print("YOLO output saved:", yolo_output_path)

# 4. Load Depth Anything V2 if needed
if "depth_model" not in globals():
    depth_repo = "/content/Depth-Anything-V2/metric_depth"

    if not os.path.isdir(depth_repo):
        raise FileNotFoundError(
            "Depth Anything V2 repository missing from /content. "
            "The Colab runtime may have reset."
        )

    sys.path.insert(0, depth_repo)
    from depth_anything_v2.dpt import DepthAnythingV2

    depth_model = DepthAnythingV2(
        encoder="vitb",
        features=128,
        out_channels=[96, 192, 384, 768],
        max_depth=80
    )

    depth_model.load_state_dict(
        torch.load(depth_path, map_location="cpu")
    )
    depth_model = depth_model.to("cpu").eval()

# 5. Estimate scene depth for the same image
raw_image = cv2.imread(image_path)
if raw_image is None:
    raise ValueError("Could not read the input image.")

with torch.inference_mode():
    depth_map = depth_model.infer_image(raw_image)

depth_map = np.asarray(depth_map, dtype=np.float32)

depth_output_path = os.path.join(output_dir, "scene_depth.npy")
np.save(depth_output_path, depth_map)
print("Depth map shape:", depth_map.shape)
print("Depth output saved:", depth_output_path)

# 6. Display the three outputs
original = cv2.cvtColor(raw_image, cv2.COLOR_BGR2RGB)
annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

depth_resized = cv2.resize(
    depth_map,
    (original.shape[1], original.shape[0])
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(original)
axes[0].set_title("Original image")

axes[1].imshow(annotated_rgb)
axes[1].set_title("YOLO crack predictions")

im = axes[2].imshow(depth_resized, cmap="viridis")
axes[2].set_title("Estimated scene depth")
plt.colorbar(im, ax=axes[2], fraction=0.046)

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("\nSTEP 23 COMPLETE")
print("Note: scene depth is not physical crack penetration depth.")


In [ ]:

# Restore Depth Anything V2 repository

import os
import subprocess
import sys

repo_path = "/content/Depth-Anything-V2"

if not os.path.isdir(repo_path):
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/DepthAnything/Depth-Anything-V2.git",
            repo_path
        ],
        check=True
    )

metric_path = os.path.join(repo_path, "metric_depth")
checkpoint = (
    "/content/drive/MyDrive/CrackDetection_Backup/"
    "depth_anything_v2_metric_vkitti_vitb.pth"
)

assert os.path.isdir(metric_path), "Depth Anything V2 repository is missing."
assert os.path.isfile(checkpoint), "Depth checkpoint is missing from Drive."

sys.path.insert(0, metric_path)

print("Repository restored:", os.path.isdir(metric_path))
print("Depth checkpoint available:", os.path.isfile(checkpoint))
print("Ready to retry depth estimation.")


In [ ]:

# STEP 23: Run YOLO crack segmentation + Depth Anything V2

import os
import sys
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from ultralytics import YOLO

# 1. Paths
backup = "/content/drive/MyDrive/CrackDetection_Backup"
yolo_path = os.path.join(backup, "best_improved.pt")
depth_path = os.path.join(
    backup, "depth_anything_v2_metric_vkitti_vitb.pth"
)
output_dir = "/content/concrete_unified_outputs"
os.makedirs(output_dir, exist_ok=True)

# 2. Input image
image_path = globals().get("filename", "")

if not image_path or not os.path.isfile(image_path):
    image_path = os.path.join(
        backup, "concrete_inspection_final.png"
    )

if not os.path.isfile(image_path):
    from google.colab import files
    uploaded = files.upload()
    image_path = next(iter(uploaded))

print("Input image:", image_path)

# 3. YOLO crack segmentation
yolo_model = YOLO(yolo_path)
yolo_results = yolo_model.predict(
    source=image_path,
    conf=0.25,
    verbose=False
)
result = yolo_results[0]

num_detections = (
    len(result.boxes) if result.boxes is not None else 0
)
num_masks = (
    len(result.masks.data) if result.masks is not None else 0
)

annotated = result.plot()
yolo_output_path = os.path.join(
    output_dir, "yolo_segmentation.jpg"
)
cv2.imwrite(yolo_output_path, annotated)

print("YOLO detections:", num_detections)
print("YOLO segmentation masks:", num_masks)
print("YOLO output saved:", yolo_output_path)

# 4. Load Depth Anything V2
metric_path = "/content/Depth-Anything-V2/metric_depth"
sys.path.insert(0, metric_path)

from depth_anything_v2.dpt import DepthAnythingV2

if "depth_model" not in globals():
    depth_model = DepthAnythingV2(
        encoder="vitb",
        features=128,
        out_channels=[96, 192, 384, 768],
        max_depth=80
    )

    depth_model.load_state_dict(
        torch.load(depth_path, map_location="cpu")
    )
    depth_model = depth_model.to("cpu").eval()

print("Depth model ready.")

# 5. Estimate scene depth on the same image
raw_image = cv2.imread(image_path)

if raw_image is None:
    raise ValueError("Could not read the input image.")

with torch.inference_mode():
    depth_map = depth_model.infer_image(raw_image)

depth_map = np.asarray(depth_map, dtype=np.float32)

depth_output_path = os.path.join(
    output_dir, "scene_depth.npy"
)
np.save(depth_output_path, depth_map)

print("Depth map shape:", depth_map.shape)
print("Depth output saved:", depth_output_path)

# 6. Display original, YOLO result and scene-depth map
original = cv2.cvtColor(raw_image, cv2.COLOR_BGR2RGB)
annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

depth_resized = cv2.resize(
    depth_map,
    (original.shape[1], original.shape[0])
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(original)
axes[0].set_title("Original image")

axes[1].imshow(annotated_rgb)
axes[1].set_title("YOLO crack predictions")

im = axes[2].imshow(depth_resized, cmap="viridis")
axes[2].set_title("Estimated scene depth")
plt.colorbar(im, ax=axes[2], fraction=0.046)

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("\nSTEP 23 COMPLETE")
print("Scene depth is not physical crack penetration depth.")


In [ ]:

# STEP 24: Combine YOLO + depth + VLM + RAG

import os
import json
import numpy as np
from datetime import datetime

output_dir = "/content/concrete_unified_outputs"
os.makedirs(output_dir, exist_ok=True)

# 1. Extract YOLO prediction details
detections = []

if result.boxes is not None:
    boxes_xyxy = result.boxes.xyxy.cpu().numpy()
    confidences = result.boxes.conf.cpu().numpy()
    class_ids = result.boxes.cls.cpu().numpy().astype(int)

    for i, (box, confidence, class_id) in enumerate(
        zip(boxes_xyxy, confidences, class_ids)
    ):
        detections.append({
            "prediction_id": i + 1,
            "class_id": int(class_id),
            "class_name": yolo_model.names.get(
                int(class_id), str(class_id)
            ),
            "confidence": round(float(confidence), 4),
            "bounding_box_xyxy_pixels": [
                round(float(v), 2) for v in box
            ]
        })

# 2. Summarize depth map without claiming crack penetration depth
depth_stats = {
    "map_shape": list(depth_map.shape),
    "minimum_model_output": round(float(np.nanmin(depth_map)), 4),
    "median_model_output": round(float(np.nanmedian(depth_map)), 4),
    "maximum_model_output": round(float(np.nanmax(depth_map)), 4),
    "interpretation": (
        "Estimated scene depth; not physical crack penetration depth."
    )
}

# 3. Include unique RAG documents
rag_documents = []
seen_ids = set()

for item in retrieved:
    if item["id"] not in seen_ids:
        rag_documents.append({
            "source_id": item["id"],
            "topic": item["topic"],
            "guidance": item["text"],
            "retrieval_score": item["score"]
        })
        seen_ids.add(item["id"])

# 4. Build unified report
unified_report = {
    "image": os.path.basename(image_path),
    "generated_at": datetime.now().astimezone().isoformat(),
    "yolo": {
        "checkpoint": yolo_path,
        "detection_count": len(detections),
        "segmentation_mask_count": (
            len(result.masks.data)
            if result.masks is not None else 0
        ),
        "detections": detections,
        "annotated_image": yolo_output_path
    },
    "depth_estimation": depth_stats,
    "vlm": {
        "model": "SmolVLM",
        "visual_observations": vlm_observations.strip()
    },
    "rag": {
        "knowledge_documents_retrieved": len(rag_documents),
        "documents": rag_documents
    },
    "limitations": [
        "YOLO predictions are model outputs and require visual verification.",
        "Overlapping predictions may represent duplicate regions.",
        "Scene depth does not directly measure physical crack penetration.",
        "Pixel measurements require appropriate calibration for physical units.",
        "Image observations alone do not establish crack cause or structural safety."
    ]
}

report_path = os.path.join(
    output_dir, "unified_inspection_report.json"
)

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(unified_report, f, indent=2, ensure_ascii=False)

print("UNIFIED INSPECTION REPORT CREATED")
print("Image:", unified_report["image"])
print("YOLO detections:", len(detections))
print("Depth map:", depth_stats["map_shape"])
print("VLM observations:", vlm_observations.strip())
print("RAG documents:", len(rag_documents))
print("Saved to:", report_path)


In [ ]:

# STEP 25: Back up final outputs to Google Drive

import os
import shutil

backup_dir = "/content/drive/MyDrive/CrackDetection_Backup"
output_dir = "/content/concrete_unified_outputs"

os.makedirs(backup_dir, exist_ok=True)

files_to_backup = [
    "unified_inspection_report.json",
    "yolo_segmentation.jpg",
    "scene_depth.npy"
]

for name in files_to_backup:
    source = os.path.join(output_dir, name)
    destination = os.path.join(backup_dir, name)

    if os.path.isfile(source):
        shutil.copy2(source, destination)
        print("BACKED UP:", destination)
    else:
        print("MISSING:", source)

# Also back up the earlier VLM + RAG report if present
rag_report = "/content/concrete_inspection_rag_report.json"

if os.path.isfile(rag_report):
    shutil.copy2(
        rag_report,
        os.path.join(backup_dir, "concrete_inspection_rag_report.json")
    )
    print("BACKED UP: concrete_inspection_rag_report.json")

print("\nBackup complete. Checking saved files...")

for name in files_to_backup:
    path = os.path.join(backup_dir, name)
    print(
        ("OK" if os.path.isfile(path) else "MISSING"),
        name,
        f"({os.path.getsize(path):,} bytes)" if os.path.isfile(path) else ""
    )


In [ ]:
# STEP 23: Run YOLO crack segmentation + Depth Anything V2

import os
import sys
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from ultralytics import YOLO

# 1. Paths
backup = "/content/drive/MyDrive/CrackDetection_Backup"
yolo_path = os.path.join(backup, "best_improved.pt")
depth_path = os.path.join(
    backup, "depth_anything_v2_metric_vkitti_vitb.pth"
)
output_dir = "/content/concrete_unified_outputs"
os.makedirs(output_dir, exist_ok=True)

# 2. Input image
image_path = globals().get("filename", "")

if not image_path or not os.path.isfile(image_path):
    image_path = os.path.join(
        backup, "concrete_inspection_final.png"
    )

if not os.path.isfile(image_path):
    from google.colab import files
    uploaded = files.upload()
    image_path = next(iter(uploaded))

print("Input image:", image_path)

# 3. YOLO crack segmentation
yolo_model = YOLO(yolo_path)
yolo_results = yolo_model.predict(
    source=image_path,
    conf=0.25,
    verbose=False
)
result = yolo_results[0]

num_detections = (
    len(result.boxes) if result.boxes is not None else 0
)
num_masks = (
    len(result.masks.data) if result.masks is not None else 0
)

annotated = result.plot()
yolo_output_path = os.path.join(
    output_dir, "yolo_segmentation.jpg"
)
cv2.imwrite(yolo_output_path, annotated)

print("YOLO detections:", num_detections)
print("YOLO segmentation masks:", num_masks)
print("YOLO output saved:", yolo_output_path)

# 4. Load Depth Anything V2
metric_path = "/content/Depth-Anything-V2/metric_depth"
sys.path.insert(0, metric_path)

from depth_anything_v2.dpt import DepthAnythingV2

if "depth_model" not in globals():
    depth_model = DepthAnythingV2(
        encoder="vitb",
        features=128,
        out_channels=[96, 192, 384, 768],
        max_depth=80
    )

    depth_model.load_state_dict(
        torch.load(depth_path, map_location="cpu")
    )
    depth_model = depth_model.to("cpu").eval()

print("Depth model ready.")

# 5. Estimate scene depth on the same image
raw_image = cv2.imread(image_path)

if raw_image is None:
    raise ValueError("Could not read the input image.")

with torch.inference_mode():
    depth_map = depth_model.infer_image(raw_image)

depth_map = np.asarray(depth_map, dtype=np.float32)

depth_output_path = os.path.join(
    output_dir, "scene_depth.npy"
)
np.save(depth_output_path, depth_map)

print("Depth map shape:", depth_map.shape)
print("Depth output saved:", depth_output_path)

# 6. Display original, YOLO result and scene-depth map
original = cv2.cvtColor(raw_image, cv2.COLOR_BGR2RGB)
annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

depth_resized = cv2.resize(
    depth_map,
    (original.shape[1], original.shape[0])
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(original)
axes[0].set_title("Original image")

axes[1].imshow(annotated_rgb)
axes[1].set_title("YOLO crack predictions")

im = axes[2].imshow(depth_resized, cmap="viridis")
axes[2].set_title("Estimated scene depth")
plt.colorbar(im, ax=axes[2], fraction=0.046)

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("\nSTEP 23 COMPLETE")
print("Scene depth is not physical crack penetration depth.")


In [ ]:

from google.colab import files
import os

uploaded = files.upload()

# Select the new concrete crack image from your computer
image_path = os.path.abspath(next(iter(uploaded.keys())))

# Update the variables used by the pipeline
filename = image_path

print("New test image selected:", image_path)
print("File exists:", os.path.isfile(image_path))


In [ ]:

# STEP 23: Run YOLO crack segmentation + Depth Anything V2

import os
import sys
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from ultralytics import YOLO

# 1. Paths
backup = "/content/drive/MyDrive/CrackDetection_Backup"
yolo_path = os.path.join(backup, "best_improved.pt")
depth_path = os.path.join(
    backup, "depth_anything_v2_metric_vkitti_vitb.pth"
)
output_dir = "/content/concrete_unified_outputs"
os.makedirs(output_dir, exist_ok=True)

# 2. Input image — explicitly use the NEW test image
image_path = "/content/Weathered Concrete with Jagged Cracks.png"

if not os.path.isfile(image_path):
    raise FileNotFoundError(
        f"Test image not found: {image_path}. "
        "Upload the image again if the Colab runtime reset."
    )

print("Input image:", image_path)

# 3. YOLO crack segmentation
yolo_model = YOLO(yolo_path)
yolo_results = yolo_model.predict(
    source=image_path,
    conf=0.25,
    verbose=False
)
result = yolo_results[0]

num_detections = (
    len(result.boxes) if result.boxes is not None else 0
)
num_masks = (
    len(result.masks.data) if result.masks is not None else 0
)

annotated = result.plot()
yolo_output_path = os.path.join(
    output_dir, "yolo_segmentation.jpg"
)
cv2.imwrite(yolo_output_path, annotated)

print("YOLO detections:", num_detections)
print("YOLO segmentation masks:", num_masks)
print("YOLO output saved:", yolo_output_path)

# 4. Load Depth Anything V2
metric_path = "/content/Depth-Anything-V2/metric_depth"
sys.path.insert(0, metric_path)

from depth_anything_v2.dpt import DepthAnythingV2

if "depth_model" not in globals():
    depth_model = DepthAnythingV2(
        encoder="vitb",
        features=128,
        out_channels=[96, 192, 384, 768],
        max_depth=80
    )

    depth_model.load_state_dict(
        torch.load(depth_path, map_location="cpu")
    )
    depth_model = depth_model.to("cpu").eval()

print("Depth model ready.")

# 5. Estimate scene depth for the NEW image
raw_image = cv2.imread(image_path)

if raw_image is None:
    raise ValueError("Could not read the input image.")

with torch.inference_mode():
    depth_map = depth_model.infer_image(raw_image)

depth_map = np.asarray(depth_map, dtype=np.float32)

depth_output_path = os.path.join(
    output_dir, "scene_depth.npy"
)
np.save(depth_output_path, depth_map)

print("Depth map shape:", depth_map.shape)
print("Depth output saved:", depth_output_path)

# 6. Display original, YOLO result and scene-depth map
original = cv2.cvtColor(raw_image, cv2.COLOR_BGR2RGB)
annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

depth_resized = cv2.resize(
    depth_map,
    (original.shape[1], original.shape[0])
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(original)
axes[0].set_title("Original image")

axes[1].imshow(annotated_rgb)
axes[1].set_title("YOLO crack predictions")

im = axes[2].imshow(depth_resized, cmap="viridis")
axes[2].set_title("Estimated scene depth")
plt.colorbar(im, ax=axes[2], fraction=0.046)

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("\nSTEP 23 COMPLETE")
print("Scene depth is not physical crack penetration depth.")


In [ ]:

# STEP 24A: Run VLM + RAG on the NEW test image

import base64
import io
import requests
from PIL import Image

image_path = "/content/Weathered Concrete with Jagged Cracks.png"

# Prepare image for SmolVLM
image = Image.open(image_path).convert("RGB")
image.thumbnail((640, 640))

buffer = io.BytesIO()
image.save(buffer, format="JPEG", quality=85)
image_b64 = base64.b64encode(buffer.getvalue()).decode("utf-8")

# Ask for observable features only
payload = {
    "model": "SmolVLM",
    "messages": [{
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": (
                    "Describe visible features of this concrete surface. "
                    "If a crack-like line is visible, describe its location, "
                    "direction, branching and apparent shape. Only describe "
                    "what can be seen. Do not claim physical depth, cause, "
                    "moisture, or structural safety."
                )
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": "data:image/jpeg;base64," + image_b64
                }
            }
        ]
    }],
    "temperature": 0.1,
    "max_tokens": 180,
    "stream": False
}

response = requests.post(
    "http://127.0.0.1:9931/v1/chat/completions",
    json=payload,
    timeout=300
)
response.raise_for_status()

# Replace old image's VLM observations
vlm_observations = response.json()["choices"][0]["message"]["content"]

# Retrieve knowledge relevant to the NEW observations
retrieved = rag_search(vlm_observations, top_k=3)

print("Image:", image_path)
print("\nVLM observations:")
print(vlm_observations)

print("\nRAG results:")
for item in retrieved:
    print(f'\n[{item["id"]}] {item["topic"]} | Score: {item["score"]}')
    print(item["text"])


In [ ]:

# STEP 26: Verify the final report matches the new test image

import json
import os

report_path = (
    "/content/drive/MyDrive/CrackDetection_Backup/"
    "unified_inspection_report.json"
)

with open(report_path, "r", encoding="utf-8") as f:
    final_report = json.load(f)

print("Report image:", final_report["image"])
print("YOLO detections:", final_report["yolo"]["detection_count"])
print("Depth map shape:", final_report["depth_estimation"]["map_shape"])
print("VLM observations:", final_report["vlm"]["visual_observations"])
print("RAG documents:", final_report["rag"]["knowledge_documents_retrieved"])

assert final_report["image"] == "Weathered Concrete with Jagged Cracks.png", (
    "The report still references the old image. Rerun Steps 24 and 25."
)

print("\nSUCCESS: The new test image is saved in the unified report.")


In [ ]:

# STEP 27: Rebuild and back up the report for the NEW image

import os
import json
import shutil
from datetime import datetime
import numpy as np

image_path = "/content/Weathered Concrete with Jagged Cracks.png"
output_dir = "/content/concrete_unified_outputs"
backup_dir = "/content/drive/MyDrive/CrackDetection_Backup"

os.makedirs(output_dir, exist_ok=True)
os.makedirs(backup_dir, exist_ok=True)

assert os.path.isfile(image_path), "New test image is missing."
assert "result" in globals(), "Run Step 23 on the new image first."
assert "depth_map" in globals(), "Depth map is missing."
assert "vlm_observations" in globals(), "Run Step 24A first."
assert "retrieved" in globals(), "RAG retrieval results are missing."

# Collect current YOLO detections
detections = []
if result.boxes is not None:
    boxes = result.boxes.xyxy.cpu().numpy()
    confidences = result.boxes.conf.cpu().numpy()
    classes = result.boxes.cls.cpu().numpy().astype(int)

    for i, (box, conf, cls) in enumerate(zip(boxes, confidences, classes)):
        detections.append({
            "prediction_id": i + 1,
            "class_id": int(cls),
            "class_name": yolo_model.names.get(int(cls), str(cls)),
            "confidence": round(float(conf), 4),
            "bounding_box_xyxy_pixels": [
                round(float(v), 2) for v in box
            ]
        })

# Keep unique RAG sources
rag_docs = []
seen = set()
for item in retrieved:
    if item["id"] not in seen:
        rag_docs.append({
            "source_id": item["id"],
            "topic": item["topic"],
            "guidance": item["text"],
            "retrieval_score": item["score"]
        })
        seen.add(item["id"])

# Build the report using the current new-image outputs
report = {
    "image": os.path.basename(image_path),
    "generated_at": datetime.now().astimezone().isoformat(),
    "yolo": {
        "checkpoint": yolo_path,
        "detection_count": len(detections),
        "segmentation_mask_count": (
            len(result.masks.data) if result.masks is not None else 0
        ),
        "detections": detections,
        "annotated_image": os.path.join(output_dir, "yolo_segmentation.jpg")
    },
    "depth_estimation": {
        "map_shape": list(depth_map.shape),
        "minimum_model_output": round(float(np.nanmin(depth_map)), 4),
        "median_model_output": round(float(np.nanmedian(depth_map)), 4),
        "maximum_model_output": round(float(np.nanmax(depth_map)), 4),
        "interpretation": (
            "Estimated scene depth; not physical crack penetration depth."
        )
    },
    "vlm": {
        "model": "SmolVLM",
        "visual_observations": vlm_observations.strip()
    },
    "rag": {
        "knowledge_documents_retrieved": len(rag_docs),
        "documents": rag_docs
    },
    "limitations": [
        "Predictions require visual verification.",
        "Overlapping predictions may be duplicate regions.",
        "Scene depth is not physical crack penetration depth.",
        "Pixel measurements require calibration for physical units.",
        "The report does not establish crack cause or structural safety."
    ]
}

report_path = os.path.join(output_dir, "unified_inspection_report.json")
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

# Back up all current outputs
for name in [
    "unified_inspection_report.json",
    "yolo_segmentation.jpg",
    "scene_depth.npy"
]:
    src = os.path.join(output_dir, name)
    if os.path.isfile(src):
        shutil.copy2(src, os.path.join(backup_dir, name))
        print("Backed up:", name)

# Verify the Drive copy
drive_report = os.path.join(backup_dir, "unified_inspection_report.json")
with open(drive_report, "r", encoding="utf-8") as f:
    saved_report = json.load(f)

print("\nReport image:", saved_report["image"])
print("YOLO detections:", saved_report["yolo"]["detection_count"])
print("Depth shape:", saved_report["depth_estimation"]["map_shape"])
print("VLM:", saved_report["vlm"]["visual_observations"])
print("RAG documents:", saved_report["rag"]["knowledge_documents_retrieved"])

assert saved_report["image"] == os.path.basename(image_path)
print("\nSUCCESS: New-image report saved and verified in Google Drive.")


In [ ]:

# STEP 29: Test severity and recommendation retrieval

question = (
    "A concrete wall has a crack with visible spalling and "
    "possibly exposed reinforcement. What should be documented "
    "and what is the recommended next action?"
)

results = rag_search(question, top_k=4)

for item in results:
    print(f'\n[{item["id"]}] {item["topic"]}')
    print("Similarity:", item["score"])
    print(item["text"])

    if "recommendation" in item:
        print("Recommendation:", item["recommendation"])

    if "source" in item:
        print("Reference:", item["source"])
        print("Links:", item["source_urls"])


In [ ]:

# STEP 30: Check severity documents and FAISS index

print("Total knowledge documents:", len(knowledge_base))
print("FAISS indexed documents:", rag_index.ntotal)

print("\nSeverity documents found:")
for doc in knowledge_base:
    if doc["id"].startswith("SEV"):
        print(doc["id"], "-", doc["topic"])

print("\nCheck expected severity IDs:")
ids = {doc["id"] for doc in knowledge_base}
for expected in ["SEV001", "SEV002", "SEV003", "SEV004", "SEV005"]:
    print(expected, "FOUND" if expected in ids else "MISSING")


In [ ]:

# STEP 31: Add concrete severity triage knowledge to RAG

severity_documents = [
    {
        "id": "SEV001",
        "topic": "Low concern - preliminary visual triage",
        "text": (
            "A small, apparently stable surface crack with no visible "
            "spalling, displacement, or exposed reinforcement may be "
            "recorded for monitoring. A photograph cannot establish "
            "structural safety or prove that a crack is stable. Record "
            "location, orientation, date and calibrated width. Seek "
            "qualified review if the crack is changing, affects a "
            "structural member, or its significance is uncertain. "
            "This is preliminary triage, not a certified severity rating."
        ),
        "recommendation": (
            "Document the crack and monitor changes where appropriate. "
            "Arrange professional review if it progresses or is uncertain."
        ),
        "source": "FEMA 306; ACI PRC-228.4-23",
        "source_urls": [
            "https://www.govinfo.gov/content/pkg/GOVPUB-OTHER-PURL-LPS117718/pdf/GOVPUB-OTHER-PURL-LPS117718.pdf",
            "https://www.concrete.org/publications/concreteresourcesubscriptions/collectiononline/collectionsearch/m/abstract/mcp_docid/4321.aspx"
        ]
    },
    {
        "id": "SEV002",
        "topic": "Moderate concern - further inspection recommended",
        "text": (
            "Multiple or branching cracks, possible widening, localized "
            "surface deterioration, minor apparent spalling, or uncertain "
            "crack extent warrant closer inspection. These visual features "
            "alone do not establish cause or structural severity. Document "
            "location, pattern, calibrated widths and extent. Compare with "
            "previous records. Seek qualified inspection, especially for "
            "beams, columns, slabs and load-bearing walls. Do not apply "
            "a universal crack-width threshold to all concrete structures."
        ),
        "recommendation": (
            "Record crack locations and calibrated widths, compare with "
            "previous inspections, and arrange closer professional review."
        ),
        "source": "FEMA 306; ACI PRC-364.1-19",
        "source_urls": [
            "https://www.govinfo.gov/content/pkg/GOVPUB-OTHER-PURL-LPS117718/pdf/GOVPUB-OTHER-PURL-LPS117718.pdf",
            "https://www.concrete.org/store/productdetail.aspx?Format=DOWNLOAD&ItemID=364119&Language=English&Units=US_AND_METRIC"
        ]
    },
    {
        "id": "SEV003",
        "topic": "High concern - prompt professional assessment",
        "text": (
            "Substantial spalling, crushing, delamination, exposed or "
            "possibly damaged reinforcement, significant displacement, "
            "or pronounced deformation are warning signs requiring prompt "
            "professional assessment. A photograph cannot determine the "
            "remaining structural capacity. If collapse or falling material "
            "appears imminent, keep people away and contact appropriate "
            "emergency or building-safety authorities. This is visual "
            "triage, not a certified structural assessment."
        ),
        "recommendation": (
            "Arrange prompt assessment by a qualified structural "
            "professional. If an immediate hazard is suspected, keep "
            "people clear and contact appropriate safety authorities."
        ),
        "source": "FEMA P-2335",
        "source_urls": [
            "https://www.fema.gov/sites/default/files/documents/fema_rsl_guidelines-for-post-earthquake-repair-and-retrofit-of-buildings-p2355_042025.pdf"
        ]
    },
    {
        "id": "SEV004",
        "topic": "Insufficient evidence - severity undetermined",
        "text": (
            "When the image is blurry, the crack-like feature is uncertain, "
            "calibration is absent, or model outputs conflict, do not force "
            "a low, moderate or high severity label. Request a clearer image, "
            "a scale reference, member type, location and inspection history. "
            "YOLO confidence is not a structural severity score. Monocular "
            "scene-depth estimates do not measure physical crack penetration."
        ),
        "recommendation": (
            "Collect clearer images and calibrated measurements. Leave "
            "severity undetermined until sufficient evidence is available."
        ),
        "source": "ACI PRC-228.4-23; ISO 16311-2:2014",
        "source_urls": [
            "https://www.concrete.org/publications/concreteresourcesubscriptions/collectiononline/collectionsearch/m/abstract/mcp_docid/4321.aspx",
            "https://www.iso.org/obp/ui?_escaped_fragment_=iso%3Astd%3Aiso%3A16311%3A-2%3Aed-1%3Av1%3Aen"
        ]
    },
    {
        "id": "SEV005",
        "topic": "Concrete inspection evidence and documentation",
        "text": (
            "Record the image identifier, date, location, concrete member "
            "type, crack orientation and pattern, representative calibrated "
            "widths, visible spalling, displacement, exposed reinforcement, "
            "and changes from earlier inspections. Record water or corrosion "
            "staining only when actually visible. Do not estimate physical "
            "crack depth, cause, reinforcement condition or structural "
            "capacity from an ordinary photograph alone. Further testing "
            "may be needed by a qualified professional."
        ),
        "recommendation": (
            "Create a structured condition record and obtain appropriate "
            "professional testing when visual evidence is insufficient."
        ),
        "source": "FEMA 306; ACI PRC-364.1-19; ISO 16311-2:2014",
        "source_urls": [
            "https://www.govinfo.gov/content/pkg/GOVPUB-OTHER-PURL-LPS117718/pdf/GOVPUB-OTHER-PURL-LPS117718.pdf",
            "https://www.concrete.org/store/productdetail.aspx?Format=DOWNLOAD&ItemID=364119&Language=English&Units=US_AND_METRIC",
            "https://www.iso.org/obp/ui?_escaped_fragment_=iso%3Astd%3Aiso%3A16311%3A-2%3Aed-1%3Av1%3Aen"
        ]
    }
]

existing_ids = {doc["id"] for doc in knowledge_base}

for doc in severity_documents:
    if doc["id"] not in existing_ids:
        knowledge_base.append(doc)
        existing_ids.add(doc["id"])

# Rebuild the embeddings and FAISS index
knowledge_texts = [
    f'{doc["topic"]}: {doc["text"]}'
    for doc in knowledge_base
]

embeddings = embedding_model.encode(
    knowledge_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

rag_index = faiss.IndexFlatIP(embeddings.shape[1])
rag_index.add(embeddings)

print("\nTotal knowledge documents:", len(knowledge_base))
print("FAISS indexed documents:", rag_index.ntotal)
print("Severity documents added:", sum(
    doc["id"].startswith("SEV") for doc in knowledge_base
))
print("Severity knowledge base ready.")


In [ ]:

# STEP 32: Update RAG retrieval to return complete document metadata

def rag_search(question, top_k=3):
    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = rag_index.search(
        query_embedding,
        min(top_k, rag_index.ntotal)
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue

        doc = knowledge_base[int(idx)]

        result = {
            "id": doc["id"],
            "topic": doc["topic"],
            "score": round(float(score), 3),
            "text": doc["text"]
        }

        for key in ["recommendation", "source", "source_urls"]:
            if key in doc:
                result[key] = doc[key]

        results.append(result)

    return results


print("RAG search updated successfully.")


In [ ]:

question = (
    "A concrete wall has a crack with visible spalling and "
    "possibly exposed reinforcement. What should be documented "
    "and what action is recommended?"
)

results = rag_search(question, top_k=5)

for item in results:
    print(f'\n[{item["id"]}] {item["topic"]}')
    print("Similarity:", item["score"])
    print("Guidance:", item["text"])

    if "recommendation" in item:
        print("Recommendation:", item["recommendation"])

    if "source" in item:
        print("Reference:", item["source"])
        for url in item.get("source_urls", []):
            print("Source link:", url)

print("\nTest complete.")


In [ ]:

# STEP 34: Rule-based preliminary triage for concrete inspection

def classify_concrete_triage(
    spalling=False,
    crushing=False,
    delamination=False,
    exposed_reinforcement=False,
    significant_displacement=False,
    pronounced_deformation=False,
    multiple_or_branching_cracks=False,
    possible_widening=False,
    uncertain_evidence=False,
    immediate_hazard_suspected=False
):
    high_concern = any([
        spalling,
        crushing,
        delamination,
        exposed_reinforcement,
        significant_displacement,
        pronounced_deformation
    ])

    moderate_concern = any([
        multiple_or_branching_cracks,
        possible_widening
    ])

    if immediate_hazard_suspected:
        level = "URGENT SAFETY CONCERN"
        recommendation = (
            "Keep people away from the potentially hazardous area and "
            "contact appropriate emergency or building-safety authorities."
        )
    elif uncertain_evidence:
        level = "INSUFFICIENT EVIDENCE"
        recommendation = (
            "Obtain clearer images, calibrated measurements and relevant "
            "inspection context. Do not infer structural safety."
        )
    elif high_concern:
        level = "HIGH CONCERN"
        recommendation = (
            "Arrange prompt assessment by a qualified structural "
            "professional. If an immediate hazard is suspected, keep "
            "people clear and contact appropriate safety authorities."
        )
    elif moderate_concern:
        level = "MODERATE CONCERN"
        recommendation = (
            "Document crack locations, patterns and calibrated widths; "
            "compare with earlier records and arrange closer professional "
            "inspection."
        )
    else:
        level = "PRELIMINARY / UNDETERMINED"
        recommendation = (
            "Record the visible condition and obtain calibrated "
            "measurements where possible. A photograph alone cannot "
            "establish stability or structural safety."
        )

    return {
        "triage_level": level,
        "recommendation": recommendation,
        "certified_assessment": False,
        "basis": "User-supplied visible findings; not a structural diagnosis."
    }


# Test case: visible spalling and possibly exposed reinforcement
test_result = classify_concrete_triage(
    spalling=True,
    exposed_reinforcement=True
)

print("Triage:", test_result["triage_level"])
print("Recommendation:", test_result["recommendation"])
print("Certified assessment:", test_result["certified_assessment"])
print("Basis:", test_result["basis"])


In [ ]:

# STEP 35: Combine preliminary triage with RAG evidence

def get_inspection_guidance(triage_result, top_k=4):
    level = triage_result["triage_level"]

    question_by_level = {
        "HIGH CONCERN": (
            "Concrete surface has substantial spalling, crushing, "
            "delamination or exposed reinforcement. What professional "
            "assessment and documentation are recommended?"
        ),
        "MODERATE CONCERN": (
            "Concrete has multiple or branching cracks or possible "
            "widening. What should be documented and inspected?"
        ),
        "INSUFFICIENT EVIDENCE": (
            "Concrete crack image is unclear or lacks calibrated "
            "measurements. What evidence is needed?"
        ),
        "URGENT SAFETY CONCERN": (
            "Possible immediate concrete building hazard, falling "
            "material or collapse. What safety action is recommended?"
        )
    }

    question = question_by_level.get(
        level,
        "How should visible concrete cracks be documented, measured "
        "and reviewed without assuming structural safety?"
    )

    retrieved = rag_search(question, top_k=top_k)

    return {
        "triage": triage_result,
        "retrieved_guidance": [
            {
                "id": item["id"],
                "topic": item["topic"],
                "similarity": item["score"],
                "guidance": item["text"],
                "recommendation": item.get("recommendation"),
                "source": item.get("source"),
                "source_urls": item.get("source_urls", [])
            }
            for item in retrieved
        ],
        "note": (
            "Preliminary visual triage only. Retrieved text is guidance, "
            "not proof of defect severity or a certified assessment."
        )
    }


# Test using the result from Step 34
inspection_guidance = get_inspection_guidance(test_result)

print("TRIAGE:", inspection_guidance["triage"]["triage_level"])
print("ACTION:", inspection_guidance["triage"]["recommendation"])
print("\nRETRIEVED GUIDANCE:")

for item in inspection_guidance["retrieved_guidance"]:
    print(f'\n[{item["id"]}] {item["topic"]}')
    print("Similarity:", item["similarity"])
    print("Recommendation:", item["recommendation"])
    print("Source:", item["source"])

print("\n", inspection_guidance["note"])


In [ ]:

# STEP 36: Save triage + RAG guidance in the unified inspection report

import json
import os
from datetime import datetime, timezone

report_path = (
    "/content/drive/MyDrive/CrackDetection_Backup/"
    "unified_inspection_report.json"
)

# Load the existing report without overwriting its other fields
with open(report_path, "r", encoding="utf-8") as f:
    report = json.load(f)

# Add the preliminary triage and retrieved evidence
report["severity_triage"] = {
    **inspection_guidance["triage"],
    "assessment_type": "Preliminary visual triage",
    "timestamp_utc": datetime.now(timezone.utc).isoformat()
}

report["rag_recommendations"] = inspection_guidance["retrieved_guidance"]
report["triage_limitations"] = inspection_guidance["note"]

# Save updated report
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

# Read back to verify
with open(report_path, "r", encoding="utf-8") as f:
    saved_report = json.load(f)

print("Updated report:", report_path)
print("Triage:", saved_report["severity_triage"]["triage_level"])
print("RAG recommendations:", len(saved_report["rag_recommendations"]))
print("Certified assessment:",
      saved_report["severity_triage"]["certified_assessment"])
print("Verification: SUCCESS")


In [ ]:

# STEP 37: Verify all unified inspection report components

import json

report_path = (
    "/content/drive/MyDrive/CrackDetection_Backup/"
    "unified_inspection_report.json"
)

with open(report_path, "r", encoding="utf-8") as f:
    report = json.load(f)

print("=== UNIFIED INSPECTION REPORT ===")
print("Image:", report.get("image", "Check report fields below"))

print("\n=== AVAILABLE REPORT SECTIONS ===")
for key, value in report.items():
    if isinstance(value, dict):
        summary = f"{len(value)} fields"
    elif isinstance(value, list):
        summary = f"{len(value)} items"
    else:
        summary = str(value)[:100]
    print(f"{key}: {summary}")

print("\n=== TRIAGE ===")
print(report["severity_triage"]["triage_level"])
print("Certified:", report["severity_triage"]["certified_assessment"])

print("\n=== RAG ===")
print("Recommendations:", len(report["rag_recommendations"]))

print("\nReport JSON loaded and verified.")


In [ ]:

# STEP 38: Persist RAG files to Google Drive

import os
import json
import numpy as np
import faiss

backup_dir = "/content/drive/MyDrive/CrackDetection_Backup"
os.makedirs(backup_dir, exist_ok=True)

# Save knowledge documents, including recommendations and references
knowledge_path = os.path.join(backup_dir, "concrete_rag_knowledge_base.json")
with open(knowledge_path, "w", encoding="utf-8") as f:
    json.dump(knowledge_base, f, indent=2, ensure_ascii=False)

# Save embedding vectors
embeddings_path = os.path.join(backup_dir, "concrete_rag_embeddings.npy")
np.save(embeddings_path, embeddings)

# Save FAISS index
index_path = os.path.join(backup_dir, "concrete_rag_faiss.index")
faiss.write_index(rag_index, index_path)

# Verify saved files
for path in [knowledge_path, embeddings_path, index_path]:
    if not os.path.isfile(path) or os.path.getsize(path) == 0:
        raise RuntimeError(f"File missing or empty: {path}")
    print(f"Saved: {os.path.basename(path)} ({os.path.getsize(path):,} bytes)")

print("\nKnowledge documents:", len(knowledge_base))
print("Indexed documents:", rag_index.ntotal)
print("RAG backup verified successfully.")


In [ ]:

# STEP 39: Restore saved RAG knowledge base and FAISS index

import os
import json
import numpy as np
import faiss

backup_dir = "/content/drive/MyDrive/CrackDetection_Backup"

knowledge_path = os.path.join(
    backup_dir, "concrete_rag_knowledge_base.json"
)
embeddings_path = os.path.join(
    backup_dir, "concrete_rag_embeddings.npy"
)
index_path = os.path.join(
    backup_dir, "concrete_rag_faiss.index"
)

# Confirm all backup files exist
for path in [knowledge_path, embeddings_path, index_path]:
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"Backup file not found: {path}. "
            "Mount Google Drive and rerun."
        )

# Restore documents, embeddings and index
with open(knowledge_path, "r", encoding="utf-8") as f:
    knowledge_base = json.load(f)

embeddings = np.load(embeddings_path)
rag_index = faiss.read_index(index_path)

# Validate consistency
assert len(knowledge_base) == rag_index.ntotal, (
    "Document count and FAISS index count do not match."
)
assert embeddings.shape[0] == len(knowledge_base), (
    "Embedding count and knowledge document count do not match."
)
assert embeddings.shape[1] == rag_index.d, (
    "Embedding dimensions do not match the FAISS index."
)

# Restore text list used by the RAG workflow
knowledge_texts = [
    f'{doc["topic"]}: {doc["text"]}'
    for doc in knowledge_base
]

print("RAG restoration successful.")
print("Knowledge documents:", len(knowledge_base))
print("FAISS indexed documents:", rag_index.ntotal)
print("Embedding shape:", embeddings.shape)
print("Severity documents:", sum(
    doc["id"].startswith("SEV") for doc in knowledge_base
))


In [ ]:

# STEP 40: Inspect existing YOLO prediction variables

print("YOLO model available:", "model" in globals())
print("YOLO results available:", "results" in globals())
print("Image path:", report.get("image", "Not found"))

if "results" in globals() and len(results) > 0:
    result = results[0]
    print("Boxes available:", result.boxes is not None)
    print("Masks available:", result.masks is not None)

    if result.boxes is not None:
        print("Detection count:", len(result.boxes))
        print("Confidence values:",
              result.boxes.conf.cpu().numpy().round(3).tolist())
else:
    print("YOLO results are not in memory; we can reload the model.")


In [ ]:

# Inspect the structure of the existing YOLO results variable

print("Type of results:", type(results).__name__)

if isinstance(results, dict):
    print("Dictionary keys:", list(results.keys()))

    for key, value in results.items():
        print(f"\nKey: {key}")
        print("Value type:", type(value).__name__)

        if isinstance(value, dict):
            print("Nested keys:", list(value.keys()))
        elif isinstance(value, (list, tuple)):
            print("Number of items:", len(value))
            if value:
                print("First item type:", type(value[0]).__name__)
        elif hasattr(value, "shape"):
            print("Shape:", value.shape)

elif isinstance(results, (list, tuple)):
    print("Number of items:", len(results))
    if results:
        print("First item type:", type(results[0]).__name__)
        print("First item:", str(results[0])[:500])
else:
    print("Value preview:", str(results)[:1000])


In [ ]:

# STEP 41: Recover YOLO predictions without retraining

import os
import cv2

image_path = "/content/Weathered Concrete with Jagged Cracks.png"

if not os.path.isfile(image_path):
    raise FileNotFoundError(f"Image not found: {image_path}")

# Use the existing YOLO model
yolo_results = model.predict(
    source=image_path,
    save=False,
    verbose=False
)

yolo_result = yolo_results[0]

print("YOLO inference successful")
print("Boxes available:", yolo_result.boxes is not None)
print("Masks available:", yolo_result.masks is not None)

if yolo_result.boxes is not None:
    print("Detection count:", len(yolo_result.boxes))
    print(
        "Confidence values:",
        yolo_result.boxes.conf.cpu().numpy().round(3).tolist()
    )

if yolo_result.masks is not None:
    print("Mask array shape:", tuple(yolo_result.masks.data.shape))


In [ ]:

# STEP 41: Load the saved YOLO checkpoint and recover predictions

from ultralytics import YOLO
import os

image_path = "/content/Weathered Concrete with Jagged Cracks.png"
checkpoint_path = (
    "/content/drive/MyDrive/CrackDetection_Backup/best_improved.pt"
)

if not os.path.isfile(image_path):
    raise FileNotFoundError(f"Image not found: {image_path}")

if not os.path.isfile(checkpoint_path):
    raise FileNotFoundError(f"YOLO checkpoint not found: {checkpoint_path}")

# Use a separate variable; do not overwrite `model`
crack_model = YOLO(checkpoint_path)

# Inference only: this does not train the model
yolo_results = crack_model.predict(
    source=image_path,
    save=False,
    verbose=False
)

yolo_result = yolo_results[0]

print("YOLO inference successful")
print("Boxes available:", yolo_result.boxes is not None)
print("Masks available:", yolo_result.masks is not None)

if yolo_result.boxes is not None:
    print("Detection count:", len(yolo_result.boxes))
    print(
        "Confidence values:",
        yolo_result.boxes.conf.cpu().numpy().round(3).tolist()
    )

if yolo_result.masks is not None:
    print("Mask array shape:", tuple(yolo_result.masks.data.shape))


In [ ]:

# STEP 42: Extract YOLO segmentation evidence

import numpy as np

mask_data = yolo_result.masks.data.cpu().numpy()
boxes_xyxy = yolo_result.boxes.xyxy.cpu().numpy()
confidences = yolo_result.boxes.conf.cpu().numpy()

crack_evidence = []

for i, mask in enumerate(mask_data):
    binary_mask = mask > 0.5
    ys, xs = np.where(binary_mask)

    if len(xs) == 0:
        continue

    x1, y1, x2, y2 = boxes_xyxy[i]

    item = {
        "detection_id": i + 1,
        "confidence": round(float(confidences[i]), 3),
        "mask_area_pixels": int(binary_mask.sum()),
        "mask_bbox_width_pixels": int(xs.max() - xs.min() + 1),
        "mask_bbox_height_pixels": int(ys.max() - ys.min() + 1),
        "model_box_xyxy": [
            round(float(v), 2) for v in [x1, y1, x2, y2]
        ],
        "measurement_type": "image-space pixels",
        "review_status": "unverified model prediction"
    }

    crack_evidence.append(item)

print("Extracted detections:", len(crack_evidence))

for item in crack_evidence:
    print("\n", item)


In [ ]:

# STEP 43: Add raw YOLO evidence to the existing inspection report

import json
from datetime import datetime, timezone

report_path = (
    "/content/drive/MyDrive/CrackDetection_Backup/"
    "unified_inspection_report.json"
)

with open(report_path, "r", encoding="utf-8") as f:
    report = json.load(f)

report["yolo"]["segmentation_evidence"] = crack_evidence
report["yolo"]["evidence_summary"] = {
    "raw_detection_count": len(crack_evidence),
    "confidence_range": [
        round(min(x["confidence"] for x in crack_evidence), 3),
        round(max(x["confidence"] for x in crack_evidence), 3)
    ],
    "measurement_units": "pixels",
    "verified_crack_count": None,
    "review_required": True,
    "review_note": (
        "Raw model predictions may overlap or be false positives. "
        "Review masks visually before counting distinct cracks. "
        "Pixel measurements are not physical dimensions."
    )
}

report["updated_at"] = datetime.now(timezone.utc).isoformat()

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

# Verify saved evidence
with open(report_path, "r", encoding="utf-8") as f:
    saved_report = json.load(f)

print("YOLO evidence saved:", len(
    saved_report["yolo"]["segmentation_evidence"]
))
print("Raw detections:", saved_report["yolo"]["evidence_summary"]["raw_detection_count"])
print("Verified crack count:",
      saved_report["yolo"]["evidence_summary"]["verified_crack_count"])
print("Review required:",
      saved_report["yolo"]["evidence_summary"]["review_required"])
print("Report updated and verified:", report_path)


In [ ]:

# STEP 44: Visualize each YOLO segmentation mask

import cv2
import matplotlib.pyplot as plt
import numpy as np

image_path = "/content/Weathered Concrete with Jagged Cracks.png"

image_bgr = cv2.imread(image_path)
if image_bgr is None:
    raise FileNotFoundError(f"Cannot read image: {image_path}")

image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

# Plot the original image with YOLO's predicted masks and boxes
annotated = yolo_result.plot()
annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(16, 9))
plt.imshow(annotated_rgb)
plt.title("YOLO crack predictions — review required")
plt.axis("off")
plt.show()

# Show each mask independently for easier review
mask_data = yolo_result.masks.data.cpu().numpy()

fig, axes = plt.subplots(1, len(mask_data), figsize=(18, 4))
if len(mask_data) == 1:
    axes = [axes]

for i, mask in enumerate(mask_data):
    axes[i].imshow(image_rgb)
    axes[i].imshow(
        cv2.resize(
            mask,
            (image_rgb.shape[1], image_rgb.shape[0]),
            interpolation=cv2.INTER_NEAREST
        ),
        alpha=0.45,
        cmap="gray"
    )
    axes[i].set_title(f"Detection {i + 1}")
    axes[i].axis("off")

plt.tight_layout()
plt.show()


In [ ]:

# STEP 46: Test severity assessment + RAG recommendations

test_cases = [
    {
        "name": "Test 1 - Small crack, no warning signs reported",
        "flags": {},
        "expected": "PRELIMINARY / UNDETERMINED"
    },
    {
        "name": "Test 2 - Multiple or branching cracks",
        "flags": {"multiple_or_branching_cracks": True},
        "expected": "MODERATE CONCERN"
    },
    {
        "name": "Test 3 - Visible spalling",
        "flags": {"spalling": True},
        "expected": "HIGH CONCERN"
    },
    {
        "name": "Test 4 - Exposed reinforcement",
        "flags": {"exposed_reinforcement": True},
        "expected": "HIGH CONCERN"
    },
    {
        "name": "Test 5 - Unclear image",
        "flags": {"uncertain_evidence": True},
        "expected": "INSUFFICIENT EVIDENCE"
    },
    {
        "name": "Test 6 - Possible immediate hazard",
        "flags": {"immediate_hazard_suspected": True},
        "expected": "URGENT SAFETY CONCERN"
    }
]

test_summary = []

for case in test_cases:
    triage = classify_concrete_triage(**case["flags"])
    guidance = get_inspection_guidance(triage, top_k=4)

    passed = triage["triage_level"] == case["expected"]

    print("\n" + "=" * 65)
    print(case["name"])
    print("Expected:", case["expected"])
    print("Actual:", triage["triage_level"])
    print("Rule test:", "PASS" if passed else "FAIL")
    print("Recommendation:", triage["recommendation"])

    print("\nRetrieved RAG guidance:")
    for item in guidance["retrieved_guidance"]:
        print(
            f'- [{item["id"]}] {item["topic"]} '
            f'(similarity={item["similarity"]})'
        )
        if item.get("recommendation"):
            print("  Recommendation:", item["recommendation"])
        if item.get("source"):
            print("  Source:", item["source"])

    test_summary.append({
        "test": case["name"],
        "expected": case["expected"],
        "actual": triage["triage_level"],
        "rule_pass": passed,
        "rag_documents_returned": len(guidance["retrieved_guidance"])
    })

print("\n" + "=" * 65)
print("FINAL TEST SUMMARY")
print("Rule tests passed:", sum(x["rule_pass"] for x in test_summary),
      "/", len(test_summary))
print("All RAG calls returned results:",
      all(x["rag_documents_returned"] > 0 for x in test_summary))


In [ ]:

# FINAL BACKUP: Save RAG, severity tests, report and project artifacts

import os
import json
import shutil
import zipfile
from datetime import datetime, timezone

import numpy as np
import faiss

backup_dir = "/content/drive/MyDrive/CrackDetection_Backup"

if not os.path.isdir(backup_dir):
    raise FileNotFoundError(
        "Backup folder not found. Mount Google Drive first."
    )

# Verify the core RAG files before making the snapshot
required_files = [
    "unified_inspection_report.json",
    "concrete_rag_knowledge_base.json",
    "concrete_rag_embeddings.npy",
    "concrete_rag_faiss.index",
]

for name in required_files:
    path = os.path.join(backup_dir, name)
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Required backup missing: {path}")

# Validate the saved RAG artifacts
with open(os.path.join(backup_dir, required_files[1]), "r",
          encoding="utf-8") as f:
    saved_kb = json.load(f)

saved_embeddings = np.load(
    os.path.join(backup_dir, required_files[2])
)
saved_index = faiss.read_index(
    os.path.join(backup_dir, required_files[3])
)

assert len(saved_kb) == saved_index.ntotal
assert saved_embeddings.shape[0] == len(saved_kb)

# Create a dated snapshot folder
timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
snapshot_name = f"RAG_Severity_Backup_{timestamp}"
snapshot_dir = os.path.join(backup_dir, snapshot_name)
os.makedirs(snapshot_dir, exist_ok=False)

# Copy core artifacts and available existing project outputs
artifact_names = [
    "unified_inspection_report.json",
    "concrete_rag_knowledge_base.json",
    "concrete_rag_embeddings.npy",
    "concrete_rag_faiss.index",
    "yolo_segmentation.jpg",
    "scene_depth.npy",
    "concrete_inspection_rag_report.json",
    "concrete_inspection_summary.txt",
    "best_improved.pt",
    "best_concrete_cracks_1280.pt",
    "best_finecracks_960.pt",
    "best.pt",
    "depth_anything_v2_metric_vkitti_vitb.pth",
    "Fresh_Concrete_Crack_Dataset_v2.zip",
]

saved_names = []

for name in artifact_names:
    source = os.path.join(backup_dir, name)
    if os.path.isfile(source):
        shutil.copy2(source, os.path.join(snapshot_dir, name))
        saved_names.append(name)

# Save the six test results if available in the current runtime
if "test_summary" in globals():
    with open(
        os.path.join(snapshot_dir, "severity_rag_test_results.json"),
        "w", encoding="utf-8"
    ) as f:
        json.dump(test_summary, f, indent=2, ensure_ascii=False)
    saved_names.append("severity_rag_test_results.json")
else:
    print("Note: test_summary not in memory; see notebook output for test results.")

# Save current triage and RAG guidance objects if available
if "inspection_guidance" in globals():
    with open(
        os.path.join(snapshot_dir, "latest_inspection_guidance.json"),
        "w", encoding="utf-8"
    ) as f:
        json.dump(inspection_guidance, f, indent=2, ensure_ascii=False)
    saved_names.append("latest_inspection_guidance.json")

# Save a status note
status = {
    "snapshot_created_utc": datetime.now(timezone.utc).isoformat(),
    "rag_document_count": len(saved_kb),
    "severity_document_count": sum(
        doc["id"].startswith("SEV") for doc in saved_kb
    ),
    "faiss_index_count": int(saved_index.ntotal),
    "embedding_shape": list(saved_embeddings.shape),
    "severity_rule_tests_passed": "6/6 in the reported test run",
    "important_limitations": [
        "Severity is preliminary visual triage, not certified assessment.",
        "YOLO predictions require review; confidence is not severity.",
        "Monocular scene depth is not physical crack penetration depth.",
        "Manual test flags are not automatically verified image findings."
    ],
    "files_in_snapshot": saved_names
}

with open(
    os.path.join(snapshot_dir, "BACKUP_MANIFEST.json"),
    "w", encoding="utf-8"
) as f:
    json.dump(status, f, indent=2, ensure_ascii=False)

# Create a ZIP of the snapshot
zip_path = os.path.join(backup_dir, snapshot_name + ".zip")

with zipfile.ZipFile(
    zip_path, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for root, dirs, files in os.walk(snapshot_dir):
        for filename in files:
            full_path = os.path.join(root, filename)
            relative_path = os.path.relpath(full_path, snapshot_dir)
            archive.write(full_path, arcname=relative_path)

# Verify the archive
with zipfile.ZipFile(zip_path, "r") as archive:
    bad_file = archive.testzip()
    if bad_file is not None:
        raise RuntimeError(f"ZIP verification failed: {bad_file}")

print("BACKUP COMPLETED AND VERIFIED")
print("Snapshot folder:", snapshot_dir)
print("ZIP archive:", zip_path)
print("RAG documents:", len(saved_kb))
print("Severity documents:", status["severity_document_count"])
print("FAISS vectors:", saved_index.ntotal)
print("Artifacts copied:", len(saved_names))
print("ZIP size (MB):", round(os.path.getsize(zip_path) / (1024 * 1024), 2))
print("ZIP integrity: PASS")


In [ ]:

# BACKUP FIX: Save project artifacts without importing FAISS

import os
import json
import shutil
import zipfile
from datetime import datetime, timezone
import numpy as np

backup_dir = "/content/drive/MyDrive/CrackDetection_Backup"

if not os.path.isdir(backup_dir):
    raise FileNotFoundError(
        "Google Drive backup folder not found. Mount Drive first."
    )

required_files = [
    "unified_inspection_report.json",
    "concrete_rag_knowledge_base.json",
    "concrete_rag_embeddings.npy",
    "concrete_rag_faiss.index",
]

for name in required_files:
    path = os.path.join(backup_dir, name)
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Required backup missing: {path}")

# Validate files without loading FAISS
with open(os.path.join(backup_dir, required_files[1]),
          "r", encoding="utf-8") as f:
    saved_kb = json.load(f)

saved_embeddings = np.load(
    os.path.join(backup_dir, "concrete_rag_embeddings.npy")
)

assert len(saved_kb) == saved_embeddings.shape[0]

timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
snapshot_name = f"RAG_Severity_Backup_{timestamp}"
snapshot_dir = os.path.join(backup_dir, snapshot_name)
os.makedirs(snapshot_dir, exist_ok=False)

artifact_names = [
    "unified_inspection_report.json",
    "concrete_rag_knowledge_base.json",
    "concrete_rag_embeddings.npy",
    "concrete_rag_faiss.index",
    "yolo_segmentation.jpg",
    "scene_depth.npy",
    "concrete_inspection_rag_report.json",
    "concrete_inspection_summary.txt",
    "best_improved.pt",
    "best_concrete_cracks_1280.pt",
    "best_finecracks_960.pt",
    "best.pt",
    "depth_anything_v2_metric_vkitti_vitb.pth",
    "Fresh_Concrete_Crack_Dataset_v2.zip",
]

saved_names = []

for name in artifact_names:
    source = os.path.join(backup_dir, name)
    if os.path.isfile(source):
        shutil.copy2(source, os.path.join(snapshot_dir, name))
        saved_names.append(name)

if "test_summary" in globals():
    with open(
        os.path.join(snapshot_dir, "severity_rag_test_results.json"),
        "w", encoding="utf-8"
    ) as f:
        json.dump(test_summary, f, indent=2, ensure_ascii=False)
    saved_names.append("severity_rag_test_results.json")
else:
    print("Note: test_summary is not in memory; the earlier test output remains in your notebook.")

if "inspection_guidance" in globals():
    with open(
        os.path.join(snapshot_dir, "latest_inspection_guidance.json"),
        "w", encoding="utf-8"
    ) as f:
        json.dump(inspection_guidance, f, indent=2, ensure_ascii=False)
    saved_names.append("latest_inspection_guidance.json")

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "knowledge_documents": len(saved_kb),
    "severity_documents": sum(
        doc["id"].startswith("SEV") for doc in saved_kb
    ),
    "embedding_shape": list(saved_embeddings.shape),
    "severity_rule_tests_passed": "6/6 in the reported test run",
    "files": saved_names,
    "limitations": [
        "Preliminary visual triage, not certified structural assessment.",
        "YOLO predictions require review.",
        "Scene depth is not physical crack penetration depth."
    ]
}

with open(
    os.path.join(snapshot_dir, "BACKUP_MANIFEST.json"),
    "w", encoding="utf-8"
) as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

zip_path = os.path.join(backup_dir, snapshot_name + ".zip")

with zipfile.ZipFile(
    zip_path, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for root, _, files in os.walk(snapshot_dir):
        for filename in files:
            full_path = os.path.join(root, filename)
            archive.write(
                full_path,
                arcname=os.path.relpath(full_path, snapshot_dir)
            )

with zipfile.ZipFile(zip_path, "r") as archive:
    assert archive.testzip() is None, "ZIP integrity check failed"

print("BACKUP COMPLETED")
print("Snapshot folder:", snapshot_dir)
print("ZIP archive:", zip_path)
print("Knowledge documents:", len(saved_kb))
print("Severity documents:", manifest["severity_documents"])
print("Embedding shape:", saved_embeddings.shape)
print("Artifacts copied:", len(saved_names))
print("ZIP integrity: PASS")


In [ ]:

from google.colab import drive

drive.mount("/content/drive")


In [ ]:

# STEP 47: Integrate image evidence, severity triage and RAG recommendations

import os
import json
from datetime import datetime, timezone

# Use the existing image and YOLO output
image_path = "/content/Weathered Concrete with Jagged Cracks.png"
report_path = (
    "/content/drive/MyDrive/CrackDetection_Backup/"
    "unified_inspection_report.json"
)

if not os.path.isfile(image_path):
    raise FileNotFoundError(f"Image not found: {image_path}")

# Check that required functions and evidence are available
required_names = [
    "classify_concrete_triage",
    "get_inspection_guidance",
    "crack_evidence",
    "yolo_result"
]
missing = [name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError(
        f"Missing variables/functions: {missing}. "
        "Run the earlier setup and YOLO evidence cells first."
    )

# Do not infer spalling, crushing or exposed reinforcement from
# crack masks alone. The current evidence does not verify these.
triage = classify_concrete_triage(
    uncertain_evidence=True
)

# Retrieve source-backed guidance for the actual evidence state
guidance = get_inspection_guidance(triage, top_k=4)

# Load the existing report if present; preserve existing fields
if os.path.isfile(report_path):
    with open(report_path, "r", encoding="utf-8") as f:
        report = json.load(f)
else:
    report = {}

report["image"] = os.path.basename(image_path)
report["updated_at"] = datetime.now(timezone.utc).isoformat()

# Preserve the raw segmentation evidence
report.setdefault("yolo", {})
report["yolo"]["segmentation_evidence"] = crack_evidence
report["yolo"]["evidence_summary"] = {
    "raw_detection_count": len(crack_evidence),
    "verified_crack_count": None,
    "review_required": True,
    "confidence_is_not_severity": True,
    "measurement_units": "pixels"
}

# Explicitly mark severity as preliminary and evidence-limited
report["severity_triage"] = {
    **triage,
    "assessment_type": "Preliminary visual triage",
    "evidence_basis": (
        "YOLO crack-like segmentation predictions; visible damage "
        "features such as spalling and exposed reinforcement have not "
        "been independently verified."
    )
}

# Save retrieved recommendations and their sources
report["rag_recommendations"] = guidance["retrieved_guidance"]
report["triage_limitations"] = guidance["note"]

# Preserve depth output but clarify what it means
report["depth_interpretation"] = {
    "available": "depth_estimation" in report,
    "meaning": (
        "Monocular scene-depth estimate only; not physical crack "
        "penetration depth or a structural severity measurement."
    ),
    "physical_crack_depth_verified": False
}

# Write and verify the report
os.makedirs(os.path.dirname(report_path), exist_ok=True)

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

with open(report_path, "r", encoding="utf-8") as f:
    verified = json.load(f)

print("INTEGRATED INSPECTION PIPELINE COMPLETE")
print("Image:", verified["image"])
print("YOLO detections:", len(
    verified["yolo"]["segmentation_evidence"]
))
print("Triage:", verified["severity_triage"]["triage_level"])
print("Verified structural severity: No")
print("RAG recommendations:", len(verified["rag_recommendations"]))
print("Physical crack depth verified:",
      verified["depth_interpretation"]["physical_crack_depth_verified"])
print("Report saved:", report_path)
print("JSON verification: PASS")


In [ ]:

import os
from glob import glob

search_roots = [
    "/content",
    "/content/drive/MyDrive/CrackDetection_Backup"
]

matches = []

for root in search_roots:
    if os.path.isdir(root):
        for ext in ("*.png", "*.jpg", "*.jpeg", "*.webp"):
            matches.extend(
                glob(os.path.join(root, "**", ext), recursive=True)
            )

# Prefer the image used in the previous inspection
preferred = [
    p for p in matches
    if "Weathered Concrete with Jagged Cracks" in os.path.basename(p)
]

if preferred:
    image_path = preferred[0]
elif matches:
    print("Exact previous image not found.")
    print("Available image files:")
    for i, path in enumerate(matches[:30], 1):
        print(f"{i}. {path}")
else:
    print("No images found. Upload the inspection image to Colab.")

if preferred:
    print("Found inspection image:", image_path)


In [ ]:

import os
import cv2

image_path = (
    "/content/drive/MyDrive/CrackDetection_Backup/"
    "concrete_inspection_final.png"
)

if not os.path.isfile(image_path):
    raise FileNotFoundError(f"Saved inspection image not found: {image_path}")

image = cv2.imread(image_path)

if image is None:
    raise ValueError("The saved image could not be opened.")

print("Inspection image restored:", image_path)
print("Image dimensions (height, width):", image.shape[:2])
print("Image available for inspection:", True)


In [ ]:

# STEP 49: Assess restored inspection image using YOLO + RAG

import os
import json
from datetime import datetime, timezone
from ultralytics import YOLO

image_path = (
    "/content/drive/MyDrive/CrackDetection_Backup/"
    "concrete_inspection_final.png"
)
backup_dir = "/content/drive/MyDrive/CrackDetection_Backup"
report_path = os.path.join(backup_dir, "unified_inspection_report.json")

# Load model only if it is not already available
if "crack_model" not in globals():
    checkpoint_path = os.path.join(backup_dir, "best_improved.pt")
    if not os.path.isfile(checkpoint_path):
        raise FileNotFoundError(f"YOLO checkpoint missing: {checkpoint_path}")
    crack_model = YOLO(checkpoint_path)

# Run inference on this image only
yolo_results = crack_model.predict(
    source=image_path,
    save=False,
    verbose=False
)
yolo_result = yolo_results[0]

print("YOLO detections:", len(yolo_result.boxes))
print("Segmentation masks available:", yolo_result.masks is not None)

# Extract pixel-space mask evidence
import numpy as np

crack_evidence = []
if yolo_result.masks is not None:
    masks = yolo_result.masks.data.cpu().numpy()
    boxes = yolo_result.boxes.xyxy.cpu().numpy()
    confidences = yolo_result.boxes.conf.cpu().numpy()

    for i, mask in enumerate(masks):
        binary_mask = mask > 0.5
        ys, xs = np.where(binary_mask)
        if len(xs) == 0:
            continue

        crack_evidence.append({
            "detection_id": i + 1,
            "confidence": round(float(confidences[i]), 3),
            "mask_area_pixels": int(binary_mask.sum()),
            "mask_bbox_width_pixels": int(xs.max() - xs.min() + 1),
            "mask_bbox_height_pixels": int(ys.max() - ys.min() + 1),
            "model_box_xyxy": [
                round(float(v), 2) for v in boxes[i]
            ],
            "measurement_type": "image-space pixels",
            "review_status": "unverified model prediction"
        })

# Do not infer spalling or exposed reinforcement from crack masks.
# The image evidence has not independently verified these features.
triage = classify_concrete_triage(uncertain_evidence=True)
guidance = get_inspection_guidance(triage, top_k=4)

# Preserve other sections of the existing report
if os.path.isfile(report_path):
    with open(report_path, "r", encoding="utf-8") as f:
        report = json.load(f)
else:
    report = {}

report["image"] = os.path.basename(image_path)
report["updated_at"] = datetime.now(timezone.utc).isoformat()
report["yolo"] = report.get("yolo", {})
report["yolo"]["segmentation_evidence"] = crack_evidence
report["yolo"]["evidence_summary"] = {
    "raw_detection_count": len(crack_evidence),
    "verified_crack_count": None,
    "review_required": True,
    "measurement_units": "pixels"
}
report["severity_triage"] = {
    **triage,
    "assessment_type": "Preliminary visual triage",
    "evidence_basis": (
        "YOLO crack-like predictions are unverified. No independent "
        "confirmation of spalling, crushing or exposed reinforcement."
    )
}
report["rag_recommendations"] = guidance["retrieved_guidance"]
report["triage_limitations"] = guidance["note"]
report["depth_interpretation"] = {
    "available": "depth_estimation" in report,
    "meaning": (
        "Scene-depth estimate only; not physical crack penetration depth."
    ),
    "physical_crack_depth_verified": False
}

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

# Verify the saved report
with open(report_path, "r", encoding="utf-8") as f:
    verified = json.load(f)

print("\nASSESSMENT SAVED")
print("Image:", verified["image"])
print("Triage:", verified["severity_triage"]["triage_level"])
print("RAG recommendations:", len(verified["rag_recommendations"]))
print("Physical crack depth verified:",
      verified["depth_interpretation"]["physical_crack_depth_verified"])
print("Report:", report_path)
print("JSON verification: PASS")


In [ ]:

%pip install -q ultralytics


In [ ]:

import ultralytics
print("Ultralytics version:", ultralytics.__version__)
print("YOLO import successful")


In [ ]:

# STEP 50: Check model, RAG and triage components

import os

backup_dir = "/content/drive/MyDrive/CrackDetection_Backup"

print("Google Drive mounted:", os.path.isdir(backup_dir))
print("YOLO checkpoint exists:",
      os.path.isfile(os.path.join(backup_dir, "best_improved.pt")))

for name in [
    "crack_model",
    "classify_concrete_triage",
    "get_inspection_guidance",
    "rag_search",
    "embedding_model",
    "knowledge_base",
    "rag_index"
]:
    print(f"{name} available:", name in globals())


In [ ]:

# STEP 51: Restore the inspection pipeline after runtime reset

import os
import json
import numpy as np
import faiss
from ultralytics import YOLO
from sentence_transformers import SentenceTransformer

backup_dir = "/content/drive/MyDrive/CrackDetection_Backup"

# 1. Restore the saved knowledge base
with open(
    os.path.join(backup_dir, "concrete_rag_knowledge_base.json"),
    "r", encoding="utf-8"
) as f:
    knowledge_base = json.load(f)

# 2. Restore embeddings and FAISS index
embeddings = np.load(
    os.path.join(backup_dir, "concrete_rag_embeddings.npy")
)
rag_index = faiss.read_index(
    os.path.join(backup_dir, "concrete_rag_faiss.index")
)

assert len(knowledge_base) == rag_index.ntotal
assert embeddings.shape[0] == len(knowledge_base)

knowledge_texts = [
    f'{doc["topic"]}: {doc["text"]}'
    for doc in knowledge_base
]

# 3. Load embedding model for future RAG queries
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# 4. Restore YOLO checkpoint
crack_model = YOLO(
    os.path.join(backup_dir, "best_improved.pt")
)

# 5. Restore RAG retrieval with source metadata
def rag_search(question, top_k=3):
    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = rag_index.search(
        query_embedding, min(top_k, rag_index.ntotal)
    )

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue

        doc = knowledge_base[int(idx)]
        item = {
            "id": doc["id"],
            "topic": doc["topic"],
            "score": round(float(score), 3),
            "text": doc["text"]
        }

        for key in ["recommendation", "source", "source_urls"]:
            if key in doc:
                item[key] = doc[key]

        results.append(item)

    return results

# 6. Restore preliminary severity triage
def classify_concrete_triage(
    spalling=False,
    crushing=False,
    delamination=False,
    exposed_reinforcement=False,
    significant_displacement=False,
    pronounced_deformation=False,
    multiple_or_branching_cracks=False,
    possible_widening=False,
    uncertain_evidence=False,
    immediate_hazard_suspected=False
):
    high_concern = any([
        spalling, crushing, delamination, exposed_reinforcement,
        significant_displacement, pronounced_deformation
    ])

    moderate_concern = any([
        multiple_or_branching_cracks, possible_widening
    ])

    if immediate_hazard_suspected:
        level = "URGENT SAFETY CONCERN"
        action = (
            "Keep people away from the potentially hazardous area and "
            "contact appropriate emergency or building-safety authorities."
        )
    elif uncertain_evidence:
        level = "INSUFFICIENT EVIDENCE"
        action = (
            "Obtain clearer images, calibrated measurements and inspection "
            "context. Do not infer structural safety."
        )
    elif high_concern:
        level = "HIGH CONCERN"
        action = (
            "Arrange prompt assessment by a qualified structural "
            "professional."
        )
    elif moderate_concern:
        level = "MODERATE CONCERN"
        action = (
            "Document crack patterns and calibrated widths, compare with "
            "earlier records, and arrange closer professional inspection."
        )
    else:
        level = "PRELIMINARY / UNDETERMINED"
        action = (
            "Document the visible condition. A photograph alone cannot "
            "establish stability or structural safety."
        )

    return {
        "triage_level": level,
        "recommendation": action,
        "certified_assessment": False,
        "basis": "Preliminary visual triage, not a structural diagnosis."
    }

# 7. Restore severity-aware RAG guidance
def get_inspection_guidance(triage_result, top_k=4):
    level = triage_result["triage_level"]

    query_by_level = {
        "HIGH CONCERN": (
            "Substantial concrete spalling, crushing or exposed "
            "reinforcement; prompt professional assessment."
        ),
        "MODERATE CONCERN": (
            "Multiple or branching concrete cracks and possible widening; "
            "documentation and closer inspection."
        ),
        "INSUFFICIENT EVIDENCE": (
            "Unclear concrete image, uncertain findings and missing "
            "calibrated measurements."
        ),
        "URGENT SAFETY CONCERN": (
            "Immediate suspected building hazard, falling material "
            "or possible collapse; urgent safety actions."
        ),
        "PRELIMINARY / UNDETERMINED": (
            "Concrete crack documentation, calibrated measurements "
            "and preliminary visual assessment."
        )
    }

    retrieved = rag_search(
        query_by_level.get(level, query_by_level["INSUFFICIENT EVIDENCE"]),
        top_k=top_k
    )

    # Prioritize the matching severity document in the displayed results.
    preferred_id = {
        "HIGH CONCERN": "SEV003",
        "MODERATE CONCERN": "SEV002",
        "INSUFFICIENT EVIDENCE": "SEV004",
        "URGENT SAFETY CONCERN": "SEV003",
        "PRELIMINARY / UNDETERMINED": "SEV005"
    }.get(level)

    retrieved.sort(
        key=lambda item: (
            item["id"] != preferred_id,
            -item["score"]
        )
    )

    return {
        "triage": triage_result,
        "retrieved_guidance": retrieved,
        "note": (
            "Preliminary visual triage only. Retrieved guidance does not "
            "prove structural severity or certify building safety."
        )
    }

print("PIPELINE RESTORED")
print("Knowledge documents:", len(knowledge_base))
print("FAISS indexed documents:", rag_index.ntotal)
print("Severity documents:", sum(
    doc["id"].startswith("SEV") for doc in knowledge_base
))
print("YOLO model loaded:", crack_model is not None)
print("RAG search available:", callable(rag_search))
print("Severity triage available:", callable(classify_concrete_triage))
print("RAG guidance available:", callable(get_inspection_guidance))


In [ ]:

%pip install -q faiss-cpu sentence-transformers ultralytics


In [ ]:

import faiss
import sentence_transformers
import ultralytics

print("FAISS:", faiss.__version__ if hasattr(faiss, "__version__") else "Imported")
print("Sentence Transformers:", sentence_transformers.__version__)
print("Ultralytics:", ultralytics.__version__)
print("All required packages imported successfully.")


In [ ]:
import os
import json
import numpy as np
import faiss
from ultralytics import YOLO
from sentence_transformers import SentenceTransformer

# Google Drive backup folder
BACKUP_DIR = "/content/drive/MyDrive/CrackDetection_Backup"

# Check required saved files
required_files = {
    "YOLO checkpoint": "best_improved.pt",
    "RAG knowledge base": "concrete_rag_knowledge_base.json",
    "RAG embeddings": "concrete_rag_embeddings.npy",
    "FAISS index": "concrete_rag_faiss.index",
    "Inspection image": "concrete_inspection_final.png"
}

for label, filename in required_files.items():
    path = os.path.join(BACKUP_DIR, filename)
    print(f"{label}: {'FOUND' if os.path.exists(path) else 'MISSING'}")

# Load RAG knowledge base
with open(
    os.path.join(BACKUP_DIR, "concrete_rag_knowledge_base.json"),
    "r",
    encoding="utf-8"
) as f:
    knowledge_base = json.load(f)

# Load embeddings and FAISS index
rag_embeddings = np.load(
    os.path.join(BACKUP_DIR, "concrete_rag_embeddings.npy")
)
rag_index = faiss.read_index(
    os.path.join(BACKUP_DIR, "concrete_rag_faiss.index")
)

# Load embedding model
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# Load trained YOLO model
crack_model = YOLO(
    os.path.join(BACKUP_DIR, "best_improved.pt")
)

# Set image path
IMAGE_PATH = os.path.join(
    BACKUP_DIR, "concrete_inspection_final.png"
)

print("\n--- RESTORATION SUMMARY ---")
print("Knowledge-base documents:", len(knowledge_base))
print("Embedding shape:", rag_embeddings.shape)
print("FAISS vectors:", rag_index.ntotal)
print("YOLO model loaded:", crack_model is not None)
print("Inspection image exists:", os.path.exists(IMAGE_PATH))

In [ ]:
import numpy as np

def rag_search(query, top_k=4):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = rag_index.search(
        query_embedding, min(top_k, rag_index.ntotal)
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx < 0 or idx >= len(knowledge_base):
            continue

        doc = knowledge_base[idx]

        # Support either dictionary-based or list-based knowledge bases
        if isinstance(knowledge_base, dict):
            doc = doc.get(str(idx), doc)

        results.append({
            "id": doc.get("id", f"DOC_{idx}"),
            "topic": doc.get("topic", doc.get("title", "Untitled")),
            "score": float(score),
            "text": doc.get("text", doc.get("content", "")),
            "recommendation": doc.get("recommendation", ""),
            "source": doc.get("source", ""),
            "source_urls": doc.get("source_urls", [])
        })

    return results


# Test the restored RAG pipeline
test_results = rag_search(
    "What should be done when concrete cracks are visible?",
    top_k=4
)

print("RAG search results:", len(test_results))

for i, result in enumerate(test_results, 1):
    print(f"\n{i}. {result['id']} — {result['topic']}")
    print("Similarity score:", round(result["score"], 4))
    print("Guidance:", result["text"][:500])

In [ ]:
def classify_concrete_triage(
    immediate_hazard_suspected=False,
    uncertain_evidence=False,
    spalling=False,
    crushing=False,
    delamination=False,
    exposed_reinforcement=False,
    significant_displacement=False,
    pronounced_deformation=False,
    multiple_or_branching_cracks=False,
    possible_widening=False
):
    high_concern_flags = {
        "spalling": spalling,
        "crushing": crushing,
        "delamination": delamination,
        "exposed_reinforcement": exposed_reinforcement,
        "significant_displacement": significant_displacement,
        "pronounced_deformation": pronounced_deformation
    }

    if immediate_hazard_suspected:
        level = "URGENT SAFETY CONCERN"
        recommendation = (
            "Keep people away from the potentially unsafe area and "
            "contact the appropriate building-safety authority or "
            "qualified structural professional promptly."
        )

    elif uncertain_evidence:
        level = "INSUFFICIENT EVIDENCE"
        recommendation = (
            "Obtain clearer images and additional inspection evidence. "
            "Do not infer structural safety from the photograph."
        )

    elif any(high_concern_flags.values()):
        level = "HIGH CONCERN"
        recommendation = (
            "Arrange prompt assessment by a qualified structural "
            "professional. Escalate immediately if there are signs "
            "of imminent danger."
        )

    elif multiple_or_branching_cracks or possible_widening:
        level = "MODERATE CONCERN"
        recommendation = (
            "Document the crack pattern and extent, compare with "
            "previous records if available, and arrange further "
            "inspection as appropriate."
        )

    else:
        level = "PRELIMINARY / UNDETERMINED"
        recommendation = (
            "Record the visible condition and obtain calibrated "
            "measurements where possible. A photograph alone cannot "
            "establish stability or structural safety."
        )

    return {
        "triage_level": level,
        "recommendation": recommendation,
        "certified_assessment": False,
        "basis": "Rule-based preliminary triage; supplied flags are not automatically detected facts.",
        "flags": {
            "immediate_hazard_suspected": immediate_hazard_suspected,
            "uncertain_evidence": uncertain_evidence,
            **high_concern_flags,
            "multiple_or_branching_cracks": multiple_or_branching_cracks,
            "possible_widening": possible_widening
        }
    }


print("Severity triage function created successfully.")

In [ ]:
test_case = classify_concrete_triage(
    multiple_or_branching_cracks=True
)

print("Triage level:", test_case["triage_level"])
print("Recommendation:", test_case["recommendation"])
print("Certified assessment:", test_case["certified_assessment"])

In [ ]:
import os
import json
import numpy as np
from datetime import datetime

# 1. Run YOLO inference on the saved inspection image
results = crack_model.predict(
    source=IMAGE_PATH,
    conf=0.25,
    verbose=False
)

result = results[0]
num_detections = len(result.boxes) if result.boxes is not None else 0

detection_evidence = []

if result.boxes is not None:
    for i, box in enumerate(result.boxes):
        detection_evidence.append({
            "detection_id": i + 1,
            "confidence": round(float(box.conf[0]), 4),
            "box_xyxy_pixels": [
                round(float(v), 2)
                for v in box.xyxy[0].cpu().tolist()
            ]
        })

# 2. Use conservative triage.
# Do not automatically infer spalling, exposed reinforcement,
# or structural danger from crack detections alone.
triage = classify_concrete_triage(
    uncertain_evidence=(num_detections == 0)
)

# 3. Retrieve guidance based on the triage result
rag_query = (
    "Concrete inspection guidance for " +
    triage["triage_level"] +
    ". " +
    triage["recommendation"]
)

guidance = rag_search(rag_query, top_k=4)

# 4. Prepare the report
report = {
    "image": IMAGE_PATH,
    "generated_at": datetime.now().astimezone().isoformat(),
    "yolo": {
        "checkpoint": os.path.join(BACKUP_DIR, "best_improved.pt"),
        "confidence_threshold": 0.25,
        "detection_count": num_detections,
        "detections": detection_evidence,
        "interpretation_limit": (
            "Detection count and confidence are model outputs, "
            "not crack severity or structural safety measurements."
        )
    },
    "severity_triage": triage,
    "rag_recommendations": guidance,
    "limitations": [
        "This is preliminary visual triage, not a certified structural assessment.",
        "YOLO detections may contain false positives or overlapping regions.",
        "Crack depth and physical width cannot be established from these outputs alone.",
        "Do not infer spalling or exposed reinforcement from crack segmentation alone.",
        "RAG similarity scores indicate textual relevance, not engineering certainty."
    ]
}

# 5. Save to Google Drive
report_path = os.path.join(
    BACKUP_DIR,
    "concrete_inspection_rag_report.json"
)

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("Inspection report saved:", report_path)
print("\nImage:", os.path.basename(IMAGE_PATH))
print("YOLO detections:", num_detections)
print("Preliminary triage:", triage["triage_level"])
print("\nRAG recommendations:")

for item in guidance:
    print(f"\n- {item['id']}: {item['topic']}")
    print(item["text"][:350])

print("\nCertified assessment:", triage["certified_assessment"])

In [ ]:
import os
from IPython.display import display, Image

# Generate a visualization using the existing YOLO result
annotated_image = result.plot()

preview_path = os.path.join(
    BACKUP_DIR,
    "concrete_inspection_yolo_preview.jpg"
)

import cv2
cv2.imwrite(preview_path, annotated_image)

print("Preview saved:", preview_path)
display(Image(filename=preview_path))

In [ ]:
import numpy as np

boxes = result.boxes
masks = result.masks

print("Total detections:", len(boxes) if boxes is not None else 0)

# 1. Bounding-box overlap (Intersection over Union)
if boxes is not None and len(boxes) > 1:
    xyxy = boxes.xyxy.cpu().numpy()
    confidences = boxes.conf.cpu().numpy()

    print("\nPairwise overlapping detections (IoU >= 0.30):")

    overlap_count = 0

    for i in range(len(xyxy)):
        for j in range(i + 1, len(xyxy)):
            x1 = max(xyxy[i, 0], xyxy[j, 0])
            y1 = max(xyxy[i, 1], xyxy[j, 1])
            x2 = min(xyxy[i, 2], xyxy[j, 2])
            y2 = min(xyxy[i, 3], xyxy[j, 3])

            intersection = max(0, x2 - x1) * max(0, y2 - y1)

            area_i = (
                (xyxy[i, 2] - xyxy[i, 0]) *
                (xyxy[i, 3] - xyxy[i, 1])
            )
            area_j = (
                (xyxy[j, 2] - xyxy[j, 0]) *
                (xyxy[j, 3] - xyxy[j, 1])
            )

            union = area_i + area_j - intersection
            iou = intersection / union if union > 0 else 0

            if iou >= 0.30:
                overlap_count += 1
                print(
                    f"Detection {i+1} and {j+1}: "
                    f"IoU={iou:.3f}, "
                    f"confidences={confidences[i]:.2f}, "
                    f"{confidences[j]:.2f}"
                )

    if overlap_count == 0:
        print("No box pairs exceeded the overlap threshold.")

# 2. Segmentation-mask evidence
if masks is not None:
    mask_array = masks.data.cpu().numpy()

    print("\nSegmentation masks:", len(mask_array))
    for i, mask in enumerate(mask_array):
        pixels = int((mask > 0.5).sum())
        print(f"Mask {i+1}: {pixels} foreground pixels")
else:
    print("\nNo segmentation masks were returned.")

In [ ]:
import os
import json
from datetime import datetime

report_path = os.path.join(
    BACKUP_DIR,
    "concrete_inspection_rag_report.json"
)

with open(report_path, "r", encoding="utf-8") as f:
    report = json.load(f)

mask_pixel_counts = [
    int((mask > 0.5).sum())
    for mask in masks.data.cpu().numpy()
] if masks is not None else []

report["yolo"]["segmentation_evidence"] = {
    "mask_count": len(mask_pixel_counts),
    "mask_foreground_pixels_at_model_resolution": mask_pixel_counts,
    "total_mask_foreground_pixels": sum(mask_pixel_counts),
    "box_pairs_iou_at_least_0_30": 0,
    "overlap_threshold_iou": 0.30,
    "interpretation": [
        "No bounding-box pair reached the selected IoU threshold.",
        "Low box IoU does not prove detections correspond to distinct cracks.",
        "Mask pixel counts are not calibrated physical measurements.",
        "Detection count and confidence do not establish structural severity."
    ]
}

report["updated_at"] = datetime.now().astimezone().isoformat()

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

# Verify the saved report
with open(report_path, "r", encoding="utf-8") as f:
    saved_report = json.load(f)

print("Updated report saved successfully.")
print("Path:", report_path)
print("Masks recorded:", saved_report["yolo"]["segmentation_evidence"]["mask_count"])
print("Total foreground pixels:", saved_report["yolo"]["segmentation_evidence"]["total_mask_foreground_pixels"])
print("Triage:", saved_report["severity_triage"]["triage_level"])

In [ ]:
import os
import json
import numpy as np
from datetime import datetime

BACKUP_DIR = "/content/drive/MyDrive/CrackDetection_Backup"
REPORT_PATH = os.path.join(
    BACKUP_DIR, "concrete_inspection_rag_report.json"
)
DEPTH_PATH = os.path.join(BACKUP_DIR, "scene_depth.npy")

checks = {}

# --------------------------------------------------
# 1. RAG: validate saved index and retrieve guidance
# --------------------------------------------------
checks["RAG knowledge base"] = len(knowledge_base) == 12
checks["RAG index"] = rag_index.ntotal == 12
checks["RAG embeddings"] = rag_embeddings.shape == (12, 384)

rag_results = rag_search(
    "Concrete crack inspection documentation, severity assessment, "
    "and recommended further inspection",
    top_k=4
)
checks["RAG retrieval"] = len(rag_results) > 0

print("=== RAG CHECK ===")
print("Knowledge-base documents:", len(knowledge_base))
print("FAISS vectors:", rag_index.ntotal)
print("Embeddings shape:", rag_embeddings.shape)
print("Retrieved documents:", len(rag_results))

for doc in rag_results:
    print(f"- {doc['id']}: {doc['topic']}")

# --------------------------------------------------
# 2. Severity rules: test expected outcomes
# --------------------------------------------------
severity_tests = [
    (
        "Default / uncertain severity",
        {},
        "PRELIMINARY / UNDETERMINED"
    ),
    (
        "Multiple or branching cracks",
        {"multiple_or_branching_cracks": True},
        "MODERATE CONCERN"
    ),
    (
        "Visible spalling flag",
        {"spalling": True},
        "HIGH CONCERN"
    ),
    (
        "Insufficient evidence",
        {"uncertain_evidence": True},
        "INSUFFICIENT EVIDENCE"
    ),
    (
        "Potential immediate hazard",
        {"immediate_hazard_suspected": True},
        "URGENT SAFETY CONCERN"
    )
]

print("\n=== SEVERITY RULE TESTS ===")
severity_results = []

for name, flags, expected in severity_tests:
    actual = classify_concrete_triage(**flags)["triage_level"]
    passed = actual == expected
    severity_results.append({
        "test": name,
        "expected": expected,
        "actual": actual,
        "passed": passed
    })
    print(f"{'PASS' if passed else 'FAIL'}: {name} -> {actual}")

checks["Severity rule tests"] = all(
    item["passed"] for item in severity_results
)

# Keep the existing image triage unchanged.
with open(REPORT_PATH, "r", encoding="utf-8") as f:
    report = json.load(f)

# --------------------------------------------------
# 3. Depth map: check numerical validity
# --------------------------------------------------
print("\n=== DEPTH ESTIMATION CHECK ===")

if os.path.exists(DEPTH_PATH):
    depth_map = np.load(DEPTH_PATH)
    finite = np.isfinite(depth_map)
    finite_values = depth_map[finite]

    depth_stats = {
        "file": DEPTH_PATH,
        "shape": list(depth_map.shape),
        "dtype": str(depth_map.dtype),
        "finite_values": int(finite.sum()),
        "total_values": int(depth_map.size),
        "finite_fraction": (
            float(finite.sum() / depth_map.size)
            if depth_map.size else 0.0
        ),
        "minimum": (
            float(finite_values.min()) if finite_values.size else None
        ),
        "maximum": (
            float(finite_values.max()) if finite_values.size else None
        ),
        "median": (
            float(np.median(finite_values))
            if finite_values.size else None
        ),
        "numerically_valid": bool(
            finite_values.size > 0
            and np.all(np.isfinite(finite_values))
        ),
        "interpretation": (
            "Saved scene-depth map. Units and metric accuracy must be "
            "verified from the model configuration and preprocessing. "
            "This is not crack penetration depth."
        )
    }

    print("Depth-map shape:", depth_stats["shape"])
    print("Finite values:", depth_stats["finite_values"],
          "/", depth_stats["total_values"])
    print("Minimum:", depth_stats["minimum"])
    print("Maximum:", depth_stats["maximum"])
    print("Median:", depth_stats["median"])

    checks["Depth map exists"] = True
    checks["Depth map has finite values"] = bool(
        finite_values.size > 0
    )
else:
    depth_stats = {
        "file": DEPTH_PATH,
        "available": False,
        "interpretation": "Saved depth map not found; rerun depth inference."
    }
    print("Saved depth map not found.")
    checks["Depth map exists"] = False
    checks["Depth map has finite values"] = False

# --------------------------------------------------
# 4. Save final validation into the existing report
# --------------------------------------------------
report["rag_recommendations"] = rag_results
report["depth_estimation"] = {
    **depth_stats,
    "crack_penetration_depth_mm": None,
    "physical_crack_depth_verified": False,
    "calibration_and_ground_truth_required": True
}
report["severity_validation"] = {
    "tests": severity_results,
    "all_rule_tests_passed": checks["Severity rule tests"],
    "image_triage": report.get("severity_triage", {}),
    "note": (
        "Synthetic rule tests validate software logic only, not "
        "real-world structural assessment accuracy."
    )
}
report["final_validation"] = {
    "checked_at": datetime.now().astimezone().isoformat(),
    "checks": checks,
    "all_checks_passed": all(checks.values())
}

report.setdefault("limitations", [])
new_limits = [
    "Scene-depth estimation does not directly measure crack penetration depth.",
    "Physical depth in millimetres requires suitable calibration and independent ground truth.",
    "Rule-based severity tests do not establish structural safety."
]
for limit in new_limits:
    if limit not in report["limitations"]:
        report["limitations"].append(limit)

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

# Verify that the updated JSON can be read back.
with open(REPORT_PATH, "r", encoding="utf-8") as f:
    verified_report = json.load(f)

print("\n=== FINAL STATUS ===")
for name, passed in checks.items():
    print(f"{'PASS' if passed else 'CHECK NEEDED'}: {name}")

print("\nReport saved and reopened successfully:", REPORT_PATH)
print("Current image triage:",
      verified_report.get("severity_triage", {}).get("triage_level"))
print("Verified physical crack depth:",
      verified_report["depth_estimation"]["physical_crack_depth_verified"])

In [ ]:
# Restore RAG files and models after a Colab runtime reset

import os
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

BACKUP_DIR = "/content/drive/MyDrive/CrackDetection_Backup"

# Mount Google Drive if needed
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

# Check required files
required_files = [
    "concrete_rag_knowledge_base.json",
    "concrete_rag_embeddings.npy",
    "concrete_rag_faiss.index",
]

for filename in required_files:
    path = os.path.join(BACKUP_DIR, filename)
    print(filename, "FOUND" if os.path.exists(path) else "MISSING")

# Load saved RAG data
with open(os.path.join(BACKUP_DIR, "concrete_rag_knowledge_base.json"), "r") as f:
    knowledge_base = json.load(f)

rag_embeddings = np.load(
    os.path.join(BACKUP_DIR, "concrete_rag_embeddings.npy")
)

rag_index = faiss.read_index(
    os.path.join(BACKUP_DIR, "concrete_rag_faiss.index")
)

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("\nRAG restoration complete")
print("Knowledge base documents:", len(knowledge_base))
print("Embedding shape:", rag_embeddings.shape)
print("FAISS vectors:", rag_index.ntotal)

In [ ]:
!pip install -q faiss-cpu sentence-transformers ultralytics

In [ ]:
import faiss
import sentence_transformers
import ultralytics

print("FAISS:", faiss.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("Ultralytics:", ultralytics.__version__)
print("All packages imported successfully!")

In [ ]:
# ============================================================
# FINAL VALIDATION: RAG + SEVERITY + DEPTH ESTIMATION
# ============================================================

import os
import json
import numpy as np

BACKUP_DIR = "/content/drive/MyDrive/CrackDetection_Backup"
REPORT_PATH = os.path.join(
    BACKUP_DIR, "concrete_inspection_rag_report.json"
)

checks = {}

# ------------------------------------------------------------
# 1. RAG: validate saved index and retrieve guidance
# ------------------------------------------------------------
checks["RAG knowledge base"] = len(knowledge_base) == 12
checks["RAG index"] = rag_index.ntotal == 12
checks["RAG embeddings"] = rag_embeddings.shape == (12, 384)

query = "How should concrete cracks be inspected for severity and depth?"
rag_results = rag_search(query, top_k=4)

checks["RAG retrieval"] = (
    rag_results is not None and len(rag_results) > 0
)

print("\n========== RAG VALIDATION ==========")
print("Knowledge base documents:", len(knowledge_base))
print("FAISS vectors:", rag_index.ntotal)
print("Embedding shape:", rag_embeddings.shape)
print("Retrieved recommendations:", len(rag_results))

for i, item in enumerate(rag_results, start=1):
    print(f"\n{i}.", item.get("topic", item.get("id", "Unknown")))
    print("Guidance:", item.get("text", item.get("recommendation", "")))


# ------------------------------------------------------------
# 2. SEVERITY: validate triage rule behaviour
# ------------------------------------------------------------
severity_tests = {
    "default_case": {},
    "multiple_or_branching_cracks": {
        "multiple_or_branching_cracks": True
    },
    "spalling": {
        "spalling": True
    },
    "uncertain_evidence": {
        "uncertain_evidence": True
    },
    "immediate_hazard_suspected": {
        "immediate_hazard_suspected": True
    },
}

expected_levels = {
    "default_case": "PRELIMINARY / UNDETERMINED",
    "multiple_or_branching_cracks": "MODERATE CONCERN",
    "spalling": "HIGH CONCERN",
    "uncertain_evidence": "INSUFFICIENT EVIDENCE",
    "immediate_hazard_suspected": "URGENT SAFETY CONCERN",
}

severity_results = {}

print("\n========== SEVERITY VALIDATION ==========")

for test_name, flags in severity_tests.items():
    result = classify_concrete_triage(**flags)
    level = result.get("triage_level")

    passed = level == expected_levels[test_name]
    checks[f"Severity test: {test_name}"] = passed

    severity_results[test_name] = {
        "actual": level,
        "expected": expected_levels[test_name],
        "passed": passed,
        "result": result,
    }

    print(
        f"{test_name}: {level} | "
        f"Expected: {expected_levels[test_name]} | "
        f"{'PASS' if passed else 'FAIL'}"
    )


# ------------------------------------------------------------
# 3. DEPTH: inspect saved scene-depth map
# ------------------------------------------------------------
depth_path = os.path.join(BACKUP_DIR, "scene_depth.npy")
depth_stats = {
    "file_found": os.path.exists(depth_path),
    "shape": None,
    "finite_pixel_count": 0,
    "min": None,
    "max": None,
    "median": None,
    "crack_penetration_depth_mm": None,
    "physical_crack_depth_verified": False,
}

print("\n========== DEPTH VALIDATION ==========")

if os.path.exists(depth_path):
    depth_map = np.load(depth_path)
    finite = depth_map[np.isfinite(depth_map)]

    depth_stats["shape"] = list(depth_map.shape)
    depth_stats["finite_pixel_count"] = int(finite.size)

    if finite.size > 0:
        depth_stats["min"] = float(np.min(finite))
        depth_stats["max"] = float(np.max(finite))
        depth_stats["median"] = float(np.median(finite))

    checks["Depth map exists"] = True
    checks["Depth map has finite values"] = finite.size > 0

    print("Depth map shape:", depth_stats["shape"])
    print("Finite pixels:", depth_stats["finite_pixel_count"])
    print("Minimum:", depth_stats["min"])
    print("Maximum:", depth_stats["max"])
    print("Median:", depth_stats["median"])
    print("Verified crack penetration depth (mm): NOT AVAILABLE")
else:
    checks["Depth map exists"] = False
    checks["Depth map has finite values"] = False
    print("WARNING: scene_depth.npy was not found.")


# ------------------------------------------------------------
# 4. SAVE FINAL VALIDATION TO EXISTING REPORT
# ------------------------------------------------------------
if os.path.exists(REPORT_PATH):
    with open(REPORT_PATH, "r") as f:
        report = json.load(f)
else:
    report = {}

report["rag_recommendations_validation"] = {
    "query": query,
    "retrieved_results": rag_results,
    "knowledge_base_count": len(knowledge_base),
    "faiss_vector_count": int(rag_index.ntotal),
}

report["severity_validation"] = severity_results
report["depth_estimation_validation"] = depth_stats

report["final_validation"] = {
    "checks": checks,
    "passed": sum(bool(v) for v in checks.values()),
    "total": len(checks),
    "all_passed": all(checks.values()),
    "limitations": [
        "Severity tests validate rule behaviour, not structural safety.",
        "RAG retrieval does not independently verify engineering conclusions.",
        "Scene-depth estimates are not crack penetration depth.",
        "Physical crack depth in millimetres requires suitable calibration "
        "and independent ground-truth measurements."
    ],
}

with open(REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2, default=str)

print("\n========== FINAL SUMMARY ==========")
for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")

print(
    f"\nChecks passed: {sum(bool(v) for v in checks.values())}"
    f"/{len(checks)}"
)
print("Report saved to:", REPORT_PATH)

In [ ]:
import os
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

BACKUP_DIR = "/content/drive/MyDrive/CrackDetection_Backup"

# Load saved RAG data
with open(os.path.join(BACKUP_DIR, "concrete_rag_knowledge_base.json"), "r") as f:
    knowledge_base = json.load(f)

rag_embeddings = np.load(
    os.path.join(BACKUP_DIR, "concrete_rag_embeddings.npy")
)

rag_index = faiss.read_index(
    os.path.join(BACKUP_DIR, "concrete_rag_faiss.index")
)

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Knowledge base:", len(knowledge_base))
print("Embeddings:", rag_embeddings.shape)
print("FAISS vectors:", rag_index.ntotal)
print("RAG variables restored!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
print(os.path.exists(
    "/content/drive/MyDrive/CrackDetection_Backup/concrete_rag_knowledge_base.json"
))

In [ ]:
import os
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

BACKUP_DIR = "/content/drive/MyDrive/CrackDetection_Backup"

with open(os.path.join(BACKUP_DIR, "concrete_rag_knowledge_base.json"), "r") as f:
    knowledge_base = json.load(f)

rag_embeddings = np.load(
    os.path.join(BACKUP_DIR, "concrete_rag_embeddings.npy")
)

rag_index = faiss.read_index(
    os.path.join(BACKUP_DIR, "concrete_rag_faiss.index")
)

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Knowledge base documents:", len(knowledge_base))
print("Embedding shape:", rag_embeddings.shape)
print("FAISS vectors:", rag_index.ntotal)
print("RAG data restored successfully!")

In [ ]:
def rag_search(query, top_k=4):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = rag_index.search(
        query_embedding,
        min(top_k, len(knowledge_base))
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if 0 <= idx < len(knowledge_base):
            doc = knowledge_base[idx]

            results.append({
                "id": doc.get("id", ""),
                "topic": doc.get("topic", ""),
                "text": doc.get("text", ""),
                "recommendation": doc.get("recommendation", ""),
                "source": doc.get("source", ""),
                "source_urls": doc.get("source_urls", []),
                "score": float(score)
            })

    return results


# Test retrieval
query = "How should concrete cracks be assessed for severity and depth?"
rag_results = rag_search(query, top_k=4)

print("RAG results:", len(rag_results))

for i, item in enumerate(rag_results, start=1):
    print(f"\n{i}. {item['topic'] or item['id']}")
    print("Similarity score:", round(item["score"], 4))
    print("Guidance:", item["text"] or item["recommendation"])

In [ ]:
def classify_concrete_triage(
    multiple_or_branching_cracks=False,
    possible_widening=False,
    spalling=False,
    crushing=False,
    delamination=False,
    exposed_reinforcement=False,
    significant_displacement=False,
    pronounced_deformation=False,
    uncertain_evidence=False,
    immediate_hazard_suspected=False
):
    high_concern = (
        spalling or crushing or delamination
        or exposed_reinforcement
        or significant_displacement
        or pronounced_deformation
    )

    if immediate_hazard_suspected:
        level = "URGENT SAFETY CONCERN"
        recommendation = (
            "Keep people away from the potentially unsafe area and "
            "contact qualified building or structural safety personnel."
        )
    elif uncertain_evidence:
        level = "INSUFFICIENT EVIDENCE"
        recommendation = (
            "Obtain clearer images and additional inspection evidence "
            "before assigning a severity category."
        )
    elif high_concern:
        level = "HIGH CONCERN"
        recommendation = (
            "Arrange prompt assessment by a qualified structural professional."
        )
    elif multiple_or_branching_cracks or possible_widening:
        level = "MODERATE CONCERN"
        recommendation = (
            "Document the cracks and arrange closer inspection by a "
            "qualified professional."
        )
    else:
        level = "PRELIMINARY / UNDETERMINED"
        recommendation = (
            "Record the visible evidence. This image-based triage does "
            "not establish structural safety or severity."
        )

    return {
        "triage_level": level,
        "recommendation": recommendation,
        "certified_assessment": False,
        "basis": "Rule-based preliminary visual triage",
        "flags": {
            "multiple_or_branching_cracks": multiple_or_branching_cracks,
            "possible_widening": possible_widening,
            "spalling": spalling,
            "crushing": crushing,
            "delamination": delamination,
            "exposed_reinforcement": exposed_reinforcement,
            "significant_displacement": significant_displacement,
            "pronounced_deformation": pronounced_deformation,
            "uncertain_evidence": uncertain_evidence,
            "immediate_hazard_suspected": immediate_hazard_suspected
        }
    }


# Test expected rule behaviour
tests = [
    ({}, "PRELIMINARY / UNDETERMINED"),
    ({"multiple_or_branching_cracks": True}, "MODERATE CONCERN"),
    ({"spalling": True}, "HIGH CONCERN"),
    ({"uncertain_evidence": True}, "INSUFFICIENT EVIDENCE"),
    ({"immediate_hazard_suspected": True}, "URGENT SAFETY CONCERN"),
]

all_passed = True

for flags, expected in tests:
    result = classify_concrete_triage(**flags)
    passed = result["triage_level"] == expected
    all_passed = all_passed and passed

    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"Expected: {expected} | "
        f"Actual: {result['triage_level']}"
    )

print("\nAll severity rule tests passed:", all_passed)

In [ ]:

# ============================================================
# UPLOAD IMAGE -> CRACK DETECTION -> DEPTH -> RAG REPORT
# ============================================================

import os
import json
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files
from IPython.display import display

BACKUP_DIR = "/content/drive/MyDrive/CrackDetection_Backup"

# 1. Upload a new concrete image
uploaded = files.upload()

if not uploaded:
    raise ValueError("No image uploaded. Please select a concrete image.")

image_name = next(iter(uploaded))
image_path = os.path.abspath(image_name)

image = Image.open(image_path).convert("RGB")
image_np = np.array(image)

print("Uploaded image:", image_name)
print("Image dimensions:", image.size)

# 2. Crack detection using your trained YOLO model
results = crack_model.predict(
    source=image_path,
    conf=0.25,
    verbose=False
)

result = results[0]
num_detections = 0 if result.boxes is None else len(result.boxes)
num_masks = 0 if result.masks is None else len(result.masks.data)

annotated = result.plot()
annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

print("\n========== CRACK DETECTION ==========")
print("Detected objects:", num_detections)
print("Segmentation masks:", num_masks)

plt.figure(figsize=(12, 7))
plt.imshow(annotated)
plt.title("YOLO Crack Detection and Segmentation")
plt.axis("off")
plt.show()

# 3. Depth estimation, if your depth model is loaded
depth_map = None

if "depth_model" in globals() and hasattr(depth_model, "infer_image"):
    depth_map = depth_model.infer_image(image_np)

elif "depth_anything_model" in globals() and hasattr(
    depth_anything_model, "infer_image"
):
    depth_map = depth_anything_model.infer_image(image_np)

if depth_map is not None:
    depth_map = np.asarray(depth_map).squeeze()
    finite = depth_map[np.isfinite(depth_map)]

    print("\n========== SCENE DEPTH ==========")
    print("Depth map shape:", depth_map.shape)

    if finite.size:
        print("Minimum:", float(finite.min()))
        print("Maximum:", float(finite.max()))
        print("Median:", float(np.median(finite)))

    plt.figure(figsize=(10, 6))
    plt.imshow(depth_map, cmap="viridis")
    plt.colorbar(label="Model-estimated scene depth")
    plt.title("Estimated Scene Depth")
    plt.axis("off")
    plt.show()

    print(
        "Note: Scene depth is NOT physical crack penetration depth in mm."
    )
else:
    print("\nDepth model is not loaded under the expected variable names.")
    print("The depth checkpoint alone is not sufficient to run inference.")
    print("Crack detection and RAG will continue; load the depth model next.")

# 4. Retrieve recommendations from your concrete RAG knowledge base
query = (
    "Concrete crack inspection: detected cracks, documenting crack "
    "patterns and width, severity triage, depth estimation limitations, "
    "recommended further inspection and safety precautions."
)

recommendations = rag_search(query, top_k=4)

print("\n========== RAG RECOMMENDATIONS ==========")

for i, item in enumerate(recommendations, start=1):
    print(f"\n{i}. {item.get('topic') or item.get('id', 'Guidance')}")
    print(item.get("text") or item.get("recommendation", "No text available."))

# 5. Conservative preliminary triage
# Detection count alone cannot establish structural severity.
triage = classify_concrete_triage(
    uncertain_evidence=(num_detections == 0)
)

print("\n========== PRELIMINARY TRIAGE ==========")
print("Triage:", triage["triage_level"])
print("Recommendation:", triage["recommendation"])
print("Certified assessment:", triage["certified_assessment"])

# 6. Save results for this uploaded image
report = {
    "image_name": image_name,
    "image_dimensions": list(image.size),
    "crack_detection": {
        "detected_objects": int(num_detections),
        "segmentation_masks": int(num_masks)
    },
    "scene_depth": {
        "available": depth_map is not None,
        "shape": list(depth_map.shape) if depth_map is not None else None,
        "crack_penetration_depth_mm": None,
        "physical_crack_depth_verified": False
    },
    "severity_triage": triage,
    "rag_recommendations": recommendations,
    "limitations": [
        "YOLO detection count is not a structural severity score.",
        "Scene depth does not directly measure crack penetration.",
        "Triage is preliminary and is not a structural safety certification."
    ]
}

report_path = os.path.join(BACKUP_DIR, "uploaded_image_inspection_report.json")

with open(report_path, "w") as f:
    json.dump(report, f, indent=2, default=str)

print("\nReport saved to:", report_path)
print("Inspection pipeline completed.")


In [ ]:
import os
from ultralytics import YOLO

BACKUP_DIR = "/content/drive/MyDrive/CrackDetection_Backup"

model_path = os.path.join(BACKUP_DIR, "best_improved.pt")

if not os.path.exists(model_path):
    raise FileNotFoundError(
        f"Model not found: {model_path}\n"
        "Check that best_improved.pt exists in your backup folder."
    )

crack_model = YOLO(model_path)

print("YOLO model loaded successfully!")
print("Model:", model_path)

In [ ]:
import os
import glob
import matplotlib.pyplot as plt
from IPython.display import display

# Locate your uploaded image
image_path = "/content/Weathered Concrete with Jagged Cracks.png"

if not os.path.exists(image_path):
    matches = glob.glob("/content/*Weathered*Concrete*Cracks.png")
    if not matches:
        raise FileNotFoundError("Uploaded image not found. Upload it again.")
    image_path = matches[0]

# Run YOLO crack detection
results = crack_model.predict(
    source=image_path,
    conf=0.25,
    verbose=False
)

result = results[0]

num_detections = 0 if result.boxes is None else len(result.boxes)
num_masks = 0 if result.masks is None else len(result.masks.data)

print("Image:", os.path.basename(image_path))
print("Detected objects:", num_detections)
print("Segmentation masks:", num_masks)

# Display annotated image
annotated = result.plot()[:, :, ::-1]

plt.figure(figsize=(14, 8))
plt.imshow(annotated)
plt.title("YOLO Concrete Crack Detection")
plt.axis("off")
plt.show()